# Phase 5 — Concurrency, Async and Performance

**Outcome:** choose threads, processes, subinterpreters or asyncio for a workload
and justify it with a benchmark; explain the asyncio event loop as precisely as
Node's; profile and optimize CPU and memory.

| Unit | Topic |
| --- | --- |
| 5.1 | Concurrency fundamentals, threads and the GIL |
| 5.2 | Multiprocessing and subinterpreters |
| 5.3 | asyncio internals: the event loop |
| 5.4 | asyncio in production |
| 5.5 | Performance and memory optimization |
| Project | 10× faster sales report |

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- **Timings are measurements, not facts.** Every benchmark prints the numbers it
  measured on your machine and the ratio between them; no cell asserts a speedup.
  Your numbers will differ with CPU, core count, load and Python build.
- **One knob for workload sizes.** Every unit setup defines
  `DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))`. Benchmarks
  multiply their sizes by it, and child processes inherit the variable. The
  default keeps every cell under about 20 seconds; start Jupyter with
  `PHASE5_DEMO_SCALE=4` for steadier numbers.
- **Workers live in files.** Process pools, `ProcessPoolExecutor` and
  subinterpreter pools need worker functions that a fresh interpreter can import
  (Unit 5.2 explains why). Cells starting with `%%writefile` save such modules
  under `notebooks/_scratch/phase5/` (git-ignored); demos that need a clean
  process (start methods, the main guard, daemon threads at exit) run as scripts
  in a child interpreter.
- **Optional builds and tools are detected, never assumed.** The free-threaded
  build (`python3.15t`), subinterpreters, the JIT, `uvloop` and the profilers are
  checked first; when one is missing its cell prints a skip message.
- **No public network.** HTTP demos talk to a local server on `127.0.0.1`
  (the shop API from Unit 4.6) or to `httpx.MockTransport`.
- *Predict the output* questions hide the answer in a collapsible block: decide
  first, then run the proof cell.
- Labs: replace the `raise NotImplementedError` in each stub, then run the check
  cell. Until then the checks print FAIL; that is expected.
- Features newer than Python 3.12 are labelled (*New in 3.13*, *Changed in 3.14*, …).

## What you need from Phases 1–4

Phase 5 is about doing several things at once. These earlier ideas come back
constantly; re-read the unit if one feels shaky.

| Earlier idea | Unit | What Phase 5 adds |
| --- | --- | --- |
| Generators, `yield`, iterator protocol | 1.13 | Coroutines are generator-like objects driven by an event loop (5.3) |
| Closures, decorators, `functools` | 1.11–1.12 | Passing work to threads and pools; `Timer` and `timer` measure it |
| Context managers, `ExitStack` | 1.14 | `with lock:`, `with executor:`, cleanup on shutdown |
| Reference counting, the GIL's place in CPython, the free-threaded build | 3.1 | What the GIL protects, when it is released, and what changes without it |
| Bytecode, `dis`, the eval breaker and the switch interval | 3.2 | Why `counter += 1` is not atomic; where thread switches can happen |
| Modules, `sys.path`, `__main__` | 3.3 | Why process workers must live in importable modules |
| `pickle` | 4.5 | Every argument sent to a process is pickled: constraints and cost |
| `httpx`, the local shop API, NumPy | 4.6 | Concurrent downloads; NumPy arrays in shared memory |

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 5.1 — Concurrency Fundamentals, Threads and the GIL

**Objectives**

- Tell concurrency from parallelism and CPU-bound from I/O-bound work, and pick
  processes, threads or coroutines from a comparison table and real latency
  numbers.
- Create, name, join and stop threads; know what a daemon thread is.
- Explain what the GIL protects, why it exists and when it is released, and
  prove it with CPU-bound and I/O-bound benchmarks.
- Find race conditions (in `dis` output and in check-then-act code) and fix them
  with the right primitive: `Lock`, `RLock`, `Semaphore`, `Event`, `Condition`,
  `Barrier`, `Timer` or a queue.
- Use `ThreadPoolExecutor` and `Future` well: submit, map, as_completed, wait,
  sizing, shutdown and error handling.
- Run the free-threaded build (`python3.15t`) and measure what it changes.

**JS bridge**

| JavaScript / Node | Python | The difference that bites |
| --- | --- | --- |
| One thread runs all your JavaScript; the event loop interleaves callbacks | Many OS threads can run Python code; in the default build the GIL lets one of them execute bytecode at a time | Python threads are **preemptive**: a switch can happen between two bytecodes of the same line, so shared state needs locks |
| No data races on ordinary objects (only one thread touches them) | Every thread sees the same heap | `counter += 1` from two threads can lose updates |
| libuv thread pool (4 threads by default) runs `fs`, `dns.lookup`, `crypto.pbkdf2`, `zlib` for you | `ThreadPoolExecutor`: you create it, size it and submit work, including your own Python functions | Nothing is offloaded implicitly |
| `worker_threads`: separate V8 isolate, separate heap, messages via structured clone | `threading.Thread` shares one heap; subinterpreters (5.2.6) are the isolate analogue | A Python thread is *not* a Worker |
| `Atomics`, `Atomics.wait` / `notify` on a `SharedArrayBuffer` | `Lock`, `Condition`, `Event`, `Semaphore` | Python's primitives work on any object, not just a shared buffer |
| `Promise` | `concurrent.futures.Future` | `future.result()` **blocks** the calling thread; there is no `await` on it outside asyncio |

**How this unit runs.** Benchmarks print what they measured and the ratio; they
never assert a speedup. I/O demos call the local shop API from Unit 4.6 (its
`/slow?seconds=` route sleeps on the server, like a slow upstream service).
Demos that need a fresh process (daemon threads at exit, the default
`threading.excepthook` output, the free-threaded build) run as scripts in a
child interpreter via `run_child(source, python=...)`.

**The local shop API (copied unchanged from Unit 4.6).** A
`BaseHTTPRequestHandler` with a few routes, served by `ThreadingHTTPServer` in a
daemon thread on `127.0.0.1:18046` (any free port if that one is busy).

In [2]:
import http.server
import json
import random
import threading
import time
import urllib.parse

SHOP_API_PORT = 18046  # fixed so outputs (URLs in messages) are stable; falls back to any free port


def demo_orders() -> list[dict[str, object]]:
    """40 seeded orders as an API would return them, with 3 invalid records planted."""
    generator = random.Random(46)
    cities = ["Pune", "Mumbai", "Bengaluru", "Hubli & Dharwad"]
    prices = {"TEA-500": "240.00", "MUG-02": "199.00", "JAR-750": "350.50", "KETTLE-9": "899.00"}
    orders: list[dict[str, object]] = []
    for order_id in range(1, 41):
        sku = generator.choice(sorted(prices))
        orders.append(
            {
                "order_id": order_id,
                "customer": f"customer-{order_id:03d}",
                "city": generator.choice(cities),
                "sku": sku,
                "quantity": generator.randint(1, 5),
                "unit_price": prices[sku],
                "placed_at": f"2026-11-{generator.randint(1, 7):02d}"
                f"T{generator.randint(9, 20):02d}:00:00+05:30",
            }
        )
    orders[6]["quantity"] = -2  # invalid: not positive
    del orders[17]["city"]  # invalid: missing field
    orders[29]["unit_price"] = "two hundred"  # invalid: not a number
    return orders


PRODUCTS_HTML = """<!doctype html>
<html><head><title>Chai Corner - Products</title></head>
<body>
  <h1>Products</h1>
  <ul id="catalogue">
    <li class="product" data-sku="TEA-500">
      <a href="/products/tea-500">Masala tea 500 g</a> <span class="price">₹240.00</span>
    </li>
    <li class="product" data-sku="MUG-02">
      <a href="/products/mug-02">Steel mug</a> <span class="price">₹199.00</span>
    </li>
    <li class="product sold-out" data-sku="JAR-750">
      <a href="/products/jar-750">Pickle jar 750 ml</a> <span class="price">₹350.50</span>
    </li>
  </ul>
  <p class="note">Prices include GST &amp; free delivery above ₹500.</p>
</body></html>
"""
ROBOTS_TXT = "User-agent: *\nDisallow: /admin/\nCrawl-delay: 1\n\nUser-agent: BadBot\nDisallow: /\n"


class ShopAPIHandler(http.server.BaseHTTPRequestHandler):
    """A tiny local HTTP API used by every HTTP demo (no external network)."""

    protocol_version = "HTTP/1.1"  # keep-alive, so clients can reuse connections
    hits: dict[str, int] = {}
    connections = 0

    def setup(self) -> None:
        """Count each new TCP connection."""
        type(self).connections += 1  # one per TCP connection, not per request
        super().setup()

    def log_message(self, format: str, *args: object) -> None:  # keep the notebook quiet
        """Silence the default request log."""
        pass

    def _send(self, status: int, body: object, content_type: str = "application/json") -> None:
        """Send status and a JSON or text body."""
        data = body.encode() if isinstance(body, str) else json.dumps(body).encode()
        try:
            self.send_response(status)
            self.send_header("Content-Type", f"{content_type}; charset=utf-8")
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        except BrokenPipeError, ConnectionResetError:  # the client gave up (a timeout demo)
            self.close_connection = True

    def _count(self, key: str) -> int:
        """Increment and return the hit counter for key."""
        self.hits[key] = self.hits.get(key, 0) + 1
        return self.hits[key]

    def do_GET(self) -> None:
        """Route GET requests to the demo endpoints."""
        url = urllib.parse.urlsplit(self.path)
        query = dict(urllib.parse.parse_qsl(url.query))
        parts = url.path.strip("/").split("/")
        fail_first = int(query.get("fail", 0))
        if url.path == "/rates":
            base = query.get("base", "INR")
            self._send(
                200,
                {
                    "base": base,
                    "date": "2026-11-07",
                    "rates": {"USD": 0.012, "EUR": 0.011, "GBP": 0.0094},
                },
            )
        elif url.path == "/orders":
            if "key" in query and self._count(query["key"]) <= fail_first:
                self._send(503, {"error": "service busy, try again"})
            else:
                self._send(200, demo_orders())
        elif parts[0] == "flaky" and len(parts) == 2:
            attempt = self._count(parts[1])
            if attempt <= fail_first:
                self._send(503, {"error": "busy", "attempt": attempt})
            else:
                self._send(200, {"ok": True, "attempt": attempt})
        elif url.path == "/hits":
            self._send(
                200, {"key": query.get("key"), "hits": self.hits.get(query.get("key", ""), 0)}
            )
        elif url.path == "/connections":
            self._send(200, {"connections": type(self).connections})
        elif url.path == "/reset":
            self.hits.clear()
            self._send(200, {"reset": True})
        elif url.path == "/slow":
            time.sleep(float(query.get("seconds", "2")))
            self._send(200, {"slow": True})
        elif parts[0] == "status" and len(parts) == 2:
            self._send(int(parts[1]), {"status": int(parts[1])})
        elif url.path == "/headers":
            self._send(
                200,
                {
                    name.lower(): value
                    for name, value in self.headers.items()
                    if name.lower() in ("user-agent", "x-request-id", "authorization")
                },
            )
        elif url.path == "/products.html":
            self._send(200, PRODUCTS_HTML, "text/html")
        elif url.path == "/robots.txt":
            self._send(200, ROBOTS_TXT, "text/plain")
        else:
            self._send(404, {"error": f"no route {url.path}"})

    def do_POST(self) -> None:
        """Echo the posted JSON back."""
        length = int(self.headers.get("Content-Length", 0))
        payload = json.loads(self.rfile.read(length) or b"null")
        self._send(201 if self.path == "/orders" else 200, {"received": payload})


def start_shop_api() -> str:
    """Start the local API once (in a daemon thread) and return its base URL."""
    server = globals().get("_SHOP_API_SERVER")
    if server is None:
        try:
            server = http.server.ThreadingHTTPServer(("127.0.0.1", SHOP_API_PORT), ShopAPIHandler)
        except OSError:
            server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), ShopAPIHandler)
        server.daemon_threads = True
        threading.Thread(target=server.serve_forever, daemon=True).start()
        globals()["_SHOP_API_SERVER"] = server
    return f"http://127.0.0.1:{server.server_address[1]}"

Unit setup, part 1: imports, `DEMO_SCALE`, the scratch folder, the API's base
URL (`SHOP_API`) and the error and interpreter helpers shared since Phase 3.

In [3]:
import asyncio
import concurrent.futures
import dis
import functools
import hashlib
import inspect
import itertools
import json
import os
import platform
import queue
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import sysconfig
import textwrap
import threading
import time
import timeit
import traceback
import urllib.request
from collections.abc import Callable, Iterator
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path

DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))  # the one knob for workload sizes
REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
PHASE5_DIR = scratch_dir / "phase5"
PHASE5_DIR.mkdir(parents=True, exist_ok=True)
CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0", "PYTHONUNBUFFERED": "1"}
SHOP_API = start_shop_api()


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


def interpreter_label() -> str:
    """Implementation, version and build flavour: print it next to every benchmark."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"

Unit setup, part 2: `Timer` (the timing context manager from Unit 1.14),
`find_free_threaded_python()` (from Unit 3.1) and `run_child()`, which runs a
script in a child interpreter and hides machine-specific paths.

In [4]:
class Timer:
    """Timing context manager (shared since Unit 1.14): reports on exit, keeps elapsed_seconds."""

    def __init__(self, label: str, report: Callable[[str], None] | None = print) -> None:
        self.label = label
        self.report = report
        self.elapsed_seconds = 0.0

    def __enter__(self) -> Timer:
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc_info: object) -> None:
        self.elapsed_seconds = time.perf_counter() - self._start
        if self.report is not None:
            self.report(f"{self.label} took {self.elapsed_seconds * 1000:.2f} ms")


def find_free_threaded_python() -> str | None:
    """Path of an installed free-threaded 3.15 interpreter, or None (shared since Unit 3.1)."""
    candidates: list[str] = []
    if uv_path := shutil.which("uv"):
        found = subprocess.run(
            [uv_path, "python", "find", "3.15t"], capture_output=True, text=True, timeout=60
        )
        if found.returncode == 0:
            candidates.append(found.stdout.strip())
    if on_path := shutil.which("python3.15t"):
        candidates.append(on_path)
    probe = "import sysconfig; print(sysconfig.get_config_var('Py_GIL_DISABLED'))"
    for candidate in candidates:
        answer = subprocess.run([candidate, "-c", probe], capture_output=True, text=True)
        if answer.stdout.strip() == "1":
            return candidate
    return None


def run_child(
    source: str, python: str = sys.executable, extra_env: dict[str, str] | None = None
) -> str:
    """Run dedented source as a script in a child interpreter; return its stdout + stderr.

    Paths of the scratch folder and interpreters are hidden so outputs stay machine-neutral.
    """
    script = PHASE5_DIR / "child_script.py"
    script.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    completed = subprocess.run(
        [python, str(script)], capture_output=True, text=True, timeout=120,
        env={**CHILD_ENV, **(extra_env or {})}, cwd=PHASE5_DIR,
    )  # fmt: skip
    output = (completed.stdout + completed.stderr).rstrip()
    for path, label in ((str(PHASE5_DIR), "<scratch>"), (str(Path(python).parent), "<python>")):
        output = output.replace(path, label)
    return re.sub(r"/[^\s\"']*/lib/python3\.\d+t?/", "<stdlib>/", output)


FREE_THREADED_PYTHON = find_free_threaded_python()
print(interpreter_label())
print(f"{DEMO_SCALE=}  {os.process_cpu_count()=}  {SHOP_API=}")
print(f"free-threaded interpreter: {'found' if FREE_THREADED_PYTHON else 'not installed'}")

CPython 3.15.1, default (GIL) build, x86_64
DEMO_SCALE=1  os.process_cpu_count()=16  SHOP_API='http://127.0.0.1:18046'
free-threaded interpreter: found


### 5.1.1 Core concepts

- **Concurrency** is about *structure*: several tasks are in progress at the same
  time and their steps interleave. One core is enough.
- **Parallelism** is about *execution*: several tasks run at the same instant on
  different cores.

Node gives you concurrency (one thread, many callbacks in flight) and gets
parallelism only inside libuv's thread pool or from `worker_threads`. CPython's
default build gives threads concurrency but, for Python bytecode, not
parallelism.

```mermaid
gantt
    dateFormat x
    axisFormat %L ms
    section Concurrency (1 core)
    task A :a1, 0, 5
    task B :b1, 5, 10
    task A :a2, 10, 15
    task B :b2, 15, 20
    section Parallelism (2 cores)
    task A (core 1) :0, 20
    task B (core 2) :0, 20
```

<details><summary>ASCII fallback</summary>

```text
concurrency, 1 core:   AAAAABBBBBAAAAABBBBB   (interleaved: both in progress)
parallelism, 2 cores:  AAAAAAAAAAAAAAAAAAAA   core 1
                       BBBBBBBBBBBBBBBBBBBB   core 2 (both running at once)
```

</details>

The next cell makes this visible. Two CPU-bound threads each record a timestamp
on every loop iteration for 60 ms. Afterwards the timeline is cut into 1 ms
slots: `A` or `B` means only that thread ran in the slot, `*` means both ran
(parallelism). Under the GIL you should see runs of `A` and `B` (about 5 ms
each, the switch interval) and almost no `*`.

In [5]:
def record_activity(stop_at: float, samples: list[float]) -> None:
    """Spin until stop_at, appending a timestamp on every iteration."""
    while (now := time.perf_counter()) < stop_at:
        samples.append(now)


def activity_timeline(duration: float = 0.06, slot: float = 0.001) -> str:
    """Run two spinning threads and draw which one ran in each time slot."""
    start = time.perf_counter()
    samples: dict[str, list[float]] = {"A": [], "B": []}
    threads = [
        threading.Thread(target=record_activity, args=(start + duration, samples[name]))
        for name in samples
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    slot_count = int(duration / slot)
    busy = {name: {int((t - start) / slot) for t in times} for name, times in samples.items()}
    cells = []
    for index in range(slot_count):
        ran = [name for name in samples if index in busy[name]]
        cells.append("*" if len(ran) == 2 else (ran[0] if ran else "."))
    return "".join(cells)


print(f"[{interpreter_label()}]")
print(activity_timeline())

[CPython 3.15.1, default (GIL) build, x86_64]
AAAAAAAAAA*BBBB*AAAA*BBBBB*AAAA*BBBB*AAAA*BBBB*AAAA*BBBBB*AA


**What to notice:** long runs of one letter, then the other. Both threads were
*in progress* the whole time (concurrency), but at any moment only one executed
Python code. A few `*` slots can appear where one thread hands over in the
middle of a millisecond. The free-threaded build (5.1.10) prints mostly `*`.

#### CPU-bound vs I/O-bound

| | CPU-bound | I/O-bound |
| --- | --- | --- |
| Time is spent | executing instructions | waiting for a disk, network or another process |
| Examples | parsing, hashing, image resizing, pure-Python loops | HTTP calls, database queries, reading files, `sleep` |
| More threads help? | only if the work runs outside the GIL (C code that releases it) or on a free-threaded build | yes: waiting threads release the GIL |
| Usual tool | processes, subinterpreters, NumPy, free-threaded threads | threads or asyncio |

You can **measure** which kind a function is: compare the CPU time its thread
used (`time.thread_time()`) with the wall-clock time (`time.perf_counter()`).
A ratio near 1 means CPU-bound; near 0 means the thread mostly waited.

In [6]:
def busy_sum(count: int) -> int:
    """Pure-Python CPU work. `% 1000` keeps every value a small int, so each step costs the same."""
    return sum(number % 1000 for number in range(count))


def fetch_slow_page(seconds: float = 0.2) -> bytes:
    """I/O work: the local API sleeps `seconds` before answering."""
    with urllib.request.urlopen(f"{SHOP_API}/slow?seconds={seconds}", timeout=10) as response:
        return response.read()


def cpu_share(work: Callable[[], object]) -> float:
    """CPU time used by this thread divided by wall-clock time, for one call of work."""
    wall_start, cpu_start = time.perf_counter(), time.thread_time()
    work()
    return (time.thread_time() - cpu_start) / (time.perf_counter() - wall_start)


print(f"busy_sum:        CPU/wall = {cpu_share(lambda: busy_sum(2_000_000 * DEMO_SCALE)):.2f}")
print(f"fetch_slow_page: CPU/wall = {cpu_share(fetch_slow_page):.2f}")

busy_sum:        CPU/wall = 0.99


fetch_slow_page: CPU/wall = 0.01


#### Latency numbers

Choosing a concurrency model starts with knowing what you wait for. The classic
table ("latency numbers every programmer should know", rounded; real hardware
varies by a factor of a few):

| Operation | Approximate time | Scaled: if 1 ns were 1 second |
| --- | --- | --- |
| L1 cache reference | 1 ns | 1 s |
| Branch mispredict | 3 ns | 3 s |
| L2 cache reference | 4 ns | 4 s |
| Uncontended mutex lock/unlock (C) | 20 ns | 20 s |
| Main memory reference | 100 ns | under 2 minutes |
| Read 1 MB sequentially from memory | 10 µs | 3 hours |
| Random read from an NVMe SSD | 20–100 µs | 6–28 hours |
| Round trip inside one data centre | 500 µs | 6 days |
| Read 1 MB sequentially from an SSD | 50–200 µs | 14–56 hours |
| Disk seek (spinning disk) | 5 ms | 2 months |
| Packet round trip, Europe ↔ US west coast | 150 ms | 5 years |

Python adds its own costs on top. The next cell measures a few of them **on this
machine** with `timeit` and one-off timings: a function call, an uncontended
`threading.Lock`, starting a thread, creating and running a coroutine, a local
HTTP round trip, reading 1 MB from the page cache and starting a child
interpreter. Notice the six orders of magnitude between the first and last rows.

In [7]:
def per_call_seconds(statement: Callable[[], object], number: int) -> float:
    """Best-of-3 average seconds per call of statement."""
    return min(timeit.repeat(statement, number=number, repeat=3)) / number


def human_time(seconds: float) -> str:
    """Format seconds as ns, µs or ms."""
    for unit, factor in (("ns", 1e9), ("µs", 1e6), ("ms", 1e3)):
        if seconds * factor < 1000:
            return f"{seconds * factor:,.1f} {unit}"
    return f"{seconds:.2f} s"


def start_and_join_thread() -> None:
    """Create, start and join a thread that does nothing."""
    worker = threading.Thread(target=lambda: None)
    worker.start()
    worker.join()


lock_for_timing = threading.Lock()
megabyte_file = PHASE5_DIR / "one_megabyte.bin"
megabyte_file.write_bytes(os.urandom(1024 * 1024))


def lock_round_trip() -> None:
    """Acquire and release an uncontended lock."""
    with lock_for_timing:
        pass


def coroutine_round_trip() -> None:
    """Create a coroutine and run it to completion without an event loop."""
    coroutine = asyncio.sleep(0)
    try:
        coroutine.send(None)
    except StopIteration:
        pass

The table itself. The child-interpreter row runs `python -c pass` five times, so
this cell takes a fraction of a second.

In [8]:
def start_child_interpreter() -> None:
    """Start a fresh interpreter that does nothing."""
    subprocess.run([sys.executable, "-c", "pass"], check=True)


def local_http_round_trip() -> None:
    """GET a tiny JSON document from the local API (new TCP connection each time)."""
    with urllib.request.urlopen(f"{SHOP_API}/rates", timeout=5) as response:
        response.read()


measurements = [
    ("Python function call", per_call_seconds(lambda: None, 200_000)),
    ("threading.Lock acquire + release", per_call_seconds(lock_round_trip, 200_000)),
    ("coroutine create + run (no loop)", per_call_seconds(coroutine_round_trip, 50_000)),
    ("read 1 MB file (page cache)", per_call_seconds(megabyte_file.read_bytes, 200)),
    ("thread start + join", per_call_seconds(start_and_join_thread, 500)),
    ("local HTTP round trip", per_call_seconds(local_http_round_trip, 50)),
    ("child interpreter start", per_call_seconds(start_child_interpreter, 5)),
]
print(f"[{interpreter_label()}] (varies by machine)")
for label, seconds in measurements:
    print(f"{label:<36}{human_time(seconds):>14}")

[CPython 3.15.1, default (GIL) build, x86_64] (varies by machine)
Python function call                       12.3 ns
threading.Lock acquire + release           40.7 ns
coroutine create + run (no loop)          274.8 ns
read 1 MB file (page cache)                43.5 µs
thread start + join                        33.4 µs
local HTTP round trip                     427.9 µs
child interpreter start                     9.5 ms


#### Processes vs threads vs coroutines

| | Process | Thread | Coroutine (asyncio) |
| --- | --- | --- | --- |
| Memory | separate address space (copy or fresh interpreter) | shared heap, own stack | shared heap; a suspended coroutine is a small heap object |
| Start cost | milliseconds | tens of microseconds | about a microsecond |
| How many | a few per core | hundreds to a few thousand | hundreds of thousands |
| Runs Python bytecode in parallel | yes (one GIL each) | no in the default build; yes on a free-threaded build | no (one thread) |
| Who decides when to switch | the OS scheduler | the OS, plus the GIL handoff every switch interval (preemptive) | your code, at each `await` (cooperative) |
| Shared state | none by default; pass messages or use shared memory | everything; needs locks | everything, but switches only at `await`, so fewer races |
| A crash affects | only that process | the whole process | the whole event loop |
| Best for | CPU-bound pure Python | blocking I/O, C code that releases the GIL, legacy blocking libraries | many concurrent network connections |
| Node equivalent | `child_process.fork`, `cluster` | libuv pool (C++ only), `worker_threads` (but isolated) | `async`/`await` on the event loop |

The cell below checks the "start cost" and "how many" rows: it starts 200 threads
that each sleep 10 ms, then runs 200 coroutines that each `await asyncio.sleep`
for 10 ms (top-level `await` works in Jupyter because the kernel already runs an
event loop). It also prints the size of one suspended coroutine object.

In [9]:
def sleep_briefly() -> None:
    """A thread body that waits 10 ms."""
    time.sleep(0.01)


with Timer("200 threads (start, sleep 10 ms, join)"):
    many_threads = [threading.Thread(target=sleep_briefly) for _ in range(200)]
    for thread in many_threads:
        thread.start()
    for thread in many_threads:
        thread.join()

with Timer("200 coroutines (sleep 10 ms, gather)"):
    await asyncio.gather(*(asyncio.sleep(0.01) for _ in range(200)))

suspended = asyncio.sleep(1)
print(f"one coroutine object: {sys.getsizeof(suspended)} bytes [{interpreter_label()}]")
suspended.close()  # never awaited: close it so Python does not warn
print(
    f"thread stack size setting: {threading.stack_size()} "
    "(0 = the OS default, 8 MiB virtual on Linux)"
)

200 threads (start, sleep 10 ms, join) took 22.33 ms
200 coroutines (sleep 10 ms, gather) took 12.24 ms
one coroutine object: 256 bytes [CPython 3.15.1, default (GIL) build, x86_64]
thread stack size setting: 0 (0 = the OS default, 8 MiB virtual on Linux)


Both finish in roughly 10–30 ms, but the threads pay for 200 OS threads (each
with its own stack reserved in virtual memory) while the coroutines are a few
hundred bytes each on the heap. That is why a server holding 10 000 open
connections uses asyncio (Units 5.3–5.4), not 10 000 threads.

### 5.1.2 `threading.Thread`

| API | What it does |
| --- | --- |
| `Thread(target=f, args=(...), kwargs={...}, name="...", daemon=...)` | describe a thread; nothing runs yet |
| `thread.start()` | ask the OS to start it; `run()` then calls `target(*args, **kwargs)` |
| `thread.join(timeout=None)` | wait until it finishes (or the timeout passes; check `is_alive()` afterwards) |
| `thread.is_alive()` | `True` between `start()` and the end of `run()` |
| `thread.name`, `thread.ident`, `thread.native_id` | label you choose; Python's id; the OS thread id (what `top -H` shows) |
| `thread.daemon` | daemon threads are killed when the interpreter exits |
| `threading.current_thread()`, `main_thread()`, `get_ident()`, `get_native_id()` | who am I? |
| `threading.enumerate()`, `active_count()` | all live `Thread` objects |

**Creation, `start` and `join`.** A thread function cannot return a value to
the caller: `join()` returns `None`. The usual workaround is to pass a container
the thread writes into (or use a `ThreadPoolExecutor`, 5.1.8).

In [10]:
def greet_customer(customer_name: str, greetings: list[str]) -> None:
    """Thread body: append a greeting that names the thread running it."""
    me = threading.current_thread()
    greetings.append(f"{me.name} greeted {customer_name}")


greetings: list[str] = []
greeter = threading.Thread(
    target=greet_customer, args=("Asha",), kwargs={"greetings": greetings}, name="greeter-1"
)
print(f"before start: {greeter.is_alive()=}, {greeter.ident=}")
greeter.start()
join_result = greeter.join()
print(f"after join:   {greeter.is_alive()=}, {join_result=}")
print(f"{greetings=}")

before start: greeter.is_alive()=False, greeter.ident=None
after join:   greeter.is_alive()=False, join_result=None
greetings=['greeter-1 greeted Asha']


**Names and IDs.** `ident` is Python's identifier (reused after a thread ends);
`native_id` is the kernel's thread id. The main thread's `native_id` equals the
process id on Linux. Exact numbers change on every run, so the cell prints
comparisons and leaves the raw values for you to look at.

In [11]:
ids_seen: dict[str, tuple[int | None, int | None]] = {}


def remember_ids() -> None:
    """Record this thread's Python ident and OS native id."""
    me = threading.current_thread()
    ids_seen[me.name] = (threading.get_ident(), threading.get_native_id())


workers = [threading.Thread(target=remember_ids, name=f"id-probe-{n}") for n in range(2)]
for worker in workers:
    worker.start()
for worker in workers:
    worker.join()
main = threading.main_thread()
print(f"{main.name=}, main native_id == os.getpid(): {main.native_id == os.getpid()}")
print(f"probe threads have distinct native ids: {len({ids[1] for ids in ids_seen.values()}) == 2}")
print(f"raw ids (vary per run): {ids_seen}")
print(f"default name of an unnamed thread: {threading.Thread(target=print).name!r}")

main.name='MainThread', main native_id == os.getpid(): True
probe threads have distinct native ids: True
raw ids (vary per run): {'id-probe-0': (139961986385600, 374105), 'id-probe-1': (139961994778304, 374106)}
default name of an unnamed thread: 'Thread-1859 (print)'


`threading.enumerate()` lists every live thread in this process. In a notebook
that includes the kernel's own threads (heartbeat, I/O publishing, control) and
the shop API's server thread; your list depends on the ipykernel version.

In [12]:
print(f"{threading.active_count()=}")
for live_thread in threading.enumerate():
    print(f"  {live_thread.name:<28} daemon={live_thread.daemon}")

threading.active_count()=10
  MainThread                   daemon=False
  IOPub                        daemon=True
  Heartbeat                    daemon=True
  Thread-2 (_watch_pipe_fd)    daemon=True
  Thread-3 (_watch_pipe_fd)    daemon=True
  Control                      daemon=True
  Shell channel                daemon=True
  IPythonHistorySavingThread   daemon=True
  Thread-1                     daemon=True
  Thread-5 (serve_forever)     daemon=True


**Lifecycle errors.** A `Thread` object runs at most once: a second `start()`
raises `RuntimeError`. `join(timeout)` returns when the timeout passes even if
the thread is still running, so always check `is_alive()` afterwards.

In [13]:
sleeper = threading.Thread(target=time.sleep, args=(0.3,), name="sleeper")
sleeper.start()
sleeper.join(timeout=0.05)
print(f"after join(timeout=0.05): {sleeper.is_alive()=}")
print(raised_message(sleeper.start))
sleeper.join()
print(f"after a full join:        {sleeper.is_alive()=}")

after join(timeout=0.05): sleeper.is_alive()=True
RuntimeError: threads can only be started once


after a full join:        sleeper.is_alive()=False


**Daemon threads.** The interpreter waits for every non-daemon thread before it
exits; daemon threads are simply stopped (no `finally`, no `with` cleanup). A
new thread inherits `daemon` from the thread that creates it (the main thread is
not a daemon). This only shows at interpreter exit, so the demo runs as a script
in a child process: the main thread returns immediately in both cases.

In [14]:
DAEMON_DEMO = """
import threading, time

def background_job():
    try:
        time.sleep(0.3)
        print("background job finished")
    finally:
        print("background job cleanup ran")

daemon = {daemon}
threading.Thread(target=background_job, daemon=daemon).start()
print(f"main thread done (daemon={{daemon}})")
"""
print(f"{threading.Thread(target=print).daemon=} (inherited from the main thread)")
for daemon_flag in (False, True):
    print(f"--- child run with daemon={daemon_flag}")
    print(run_child(DAEMON_DEMO.format(daemon=daemon_flag)))

threading.Thread(target=print).daemon=False (inherited from the main thread)
--- child run with daemon=False


main thread done (daemon=False)
background job finished
background job cleanup ran
--- child run with daemon=True
main thread done (daemon=True)


With `daemon=False` the process waits 0.3 s and the job finishes; with
`daemon=True` the process exits at once and **neither print runs**, not even the
`finally`. Use daemon threads only for work that is safe to lose (a heartbeat, a
cache warmer), and give important workers a stop flag instead (5.1.9).

**Subclassing vs `target`.** You can subclass `Thread` and override `run()`;
this is handy when the thread carries state (here, its result). Passing a
`target` keeps the work a plain function you can also call and test without
threads.

In [15]:
class PriceFetcher(threading.Thread):
    """A thread subclass that fetches exchange rates and keeps the result."""

    def __init__(self, base_currency: str) -> None:
        super().__init__(name=f"rates-{base_currency}")
        self.base_currency = base_currency
        self.rates: dict[str, float] = {}

    def run(self) -> None:
        """Called in the new thread by start()."""
        url = f"{SHOP_API}/rates?base={self.base_currency}"
        with urllib.request.urlopen(url, timeout=5) as reply:
            self.rates = json.load(reply)["rates"]


fetcher = PriceFetcher("INR")
fetcher.start()
fetcher.join()
print(f"subclass: {fetcher.name} -> {fetcher.rates}")


def fetch_rates(base_currency: str, results: dict[str, dict[str, float]]) -> None:
    """The same work as a plain function, usable with or without threads."""
    with urllib.request.urlopen(f"{SHOP_API}/rates?base={base_currency}", timeout=5) as reply:
        results[base_currency] = json.load(reply)["rates"]


rate_results: dict[str, dict[str, float]] = {}
plain = threading.Thread(target=fetch_rates, args=("INR", rate_results))
plain.start()
plain.join()
print(f"target:   {plain.name} -> {rate_results}")

subclass: rates-INR -> {'USD': 0.012, 'EUR': 0.011, 'GBP': 0.0094}
target:   Thread-1862 (fetch_rates) -> {'INR': {'USD': 0.012, 'EUR': 0.011, 'GBP': 0.0094}}


| Style | Use it when |
| --- | --- |
| `Thread(target=func, args=...)` | almost always: the work stays a testable function |
| `class Worker(Thread): def run(self)` | the thread is a long-lived object with its own state and methods (a poller, a server loop) |
| `ThreadPoolExecutor().submit(func, ...)` | you want return values and exceptions back (5.1.8): the default for new code |

> **Note:** override `run()`, never `start()`. And call `super().__init__()` in a
> subclass's `__init__`, or `start()` fails with
> `RuntimeError: thread.__init__() not called`.

### 5.1.3 The Global Interpreter Lock

**What it protects.** The GIL is a lock inside the interpreter. In the default
build a thread must hold it to execute bytecode or touch Python objects. It
protects CPython's *own* state: reference counts (incremented and decremented
with plain, non-atomic instructions), the internals of `dict`, `list` and other
built-ins, the memory allocator, the import system. It does **not** protect
your program's invariants across several bytecodes (5.1.4).

**Why it exists.** Reference counting (Unit 3.1) touches a counter on almost
every operation; making every one of those atomic or locked would slow
single-threaded code. One big lock made CPython simple and fast for one thread,
and made C extensions easy to write (they could assume no other Python code runs
concurrently). PEP 703's free-threaded build removes it at a cost (5.1.10).

**When it is released.**

| Situation | What happens |
| --- | --- |
| Blocking I/O: `socket.recv`, `file.read`, `select`, `subprocess` waits | the C code releases the GIL around the system call |
| `time.sleep(seconds)` | released for the whole sleep |
| C extensions doing long work: `hashlib` (data over 2 KiB), `zlib`, `bz2`, `lzma`, NumPy, `sqlite3` queries | released if the extension says so (`Py_BEGIN_ALLOW_THREADS`) |
| A pure-Python loop | **not released voluntarily**; after the switch interval another thread asks for it and the eval loop hands it over at the next check |

```mermaid
sequenceDiagram
    participant A as Thread A (CPU loop)
    participant G as GIL
    participant B as Thread B (waiting)
    A->>G: holds the GIL, runs bytecode
    B->>G: waits up to the switch interval (5 ms), then sets "drop request"
    G-->>A: eval breaker: A drops the GIL at the next check
    G->>B: B takes the GIL and runs
    A->>G: A waits for its turn
```

<details><summary>ASCII fallback</summary>

```text
Thread A: [run bytecode .......5 ms.......] drop -> wait ............ [run ...
Thread B: wait ......... set drop request -> [run bytecode ...5 ms...] drop
            a waiting thread asks after `sys.getswitchinterval()` seconds
```

</details>

**The switch interval.** `sys.getswitchinterval()` is 0.005 s by default. The
next cell reuses `count_thread_switches` from Unit 3.2.10: two CPU-bound threads
count how often the running thread changes in 0.2 s, at two intervals.

In [16]:
def count_thread_switches(switch_interval: float, duration: float = 0.2) -> int:
    """Two CPU-bound threads; count how often the running thread changes."""
    state = {"owner": "", "switches": 0}
    deadline = time.perf_counter() + duration

    def spin(name: str) -> None:
        while time.perf_counter() < deadline:
            if state["owner"] != name:
                state["owner"] = name
                state["switches"] += 1

    original_interval = sys.getswitchinterval()
    sys.setswitchinterval(switch_interval)
    try:
        threads = [threading.Thread(target=spin, args=(name,)) for name in ("A", "B")]
        for thread in threads:
            thread.start()
        for thread in threads:
            thread.join()
    finally:
        sys.setswitchinterval(original_interval)
    return state["switches"]


print(f"{sys.getswitchinterval()=}  [{interpreter_label()}]")
for interval in (0.005, 0.0005):
    print(
        f"switch interval {interval * 1000:>4} ms -> "
        f"{count_thread_switches(interval):>5} switches in 0.2 s"
    )

sys.getswitchinterval()=0.005  [CPython 3.15.1, default (GIL) build, x86_64]


switch interval  5.0 ms ->    37 switches in 0.2 s
switch interval  0.5 ms ->   338 switches in 0.2 s


About 40 switches in 0.2 s at 5 ms (one every 5 ms), about ten times more at
0.5 ms. A shorter interval makes threads more responsive but adds handoff
overhead; leave it alone in real code.

#### Benchmark: CPU-bound work in threads

The same total work (`busy_sum` over 8 million numbers × `DEMO_SCALE`) split
across 1, 2 and 4 threads. With the GIL only one thread executes bytecode at a
time, so expect the wall time to stay about the same (or get slightly worse
from handoffs).

In [17]:
def run_in_threads(work: Callable[[int], object], items: list[int]) -> float:
    """Run work(item) for every item, one thread per item; return wall seconds."""
    threads = [threading.Thread(target=work, args=(item,)) for item in items]
    start = time.perf_counter()
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    return time.perf_counter() - start


total_numbers = 8_000_000 * DEMO_SCALE
print(f"[{interpreter_label()}] busy_sum over {total_numbers:,} numbers in total")
cpu_baseline = 0.0
for thread_count in (1, 2, 4):
    seconds = run_in_threads(busy_sum, [total_numbers // thread_count] * thread_count)
    cpu_baseline = cpu_baseline or seconds
    print(
        f"{thread_count} thread(s): {seconds:.3f} s   "
        f"speedup vs 1 thread: {cpu_baseline / seconds:.2f}x"
    )

[CPython 3.15.1, default (GIL) build, x86_64] busy_sum over 8,000,000 numbers in total
1 thread(s): 0.186 s   speedup vs 1 thread: 1.00x


2 thread(s): 0.173 s   speedup vs 1 thread: 1.08x
4 thread(s): 0.190 s   speedup vs 1 thread: 0.98x


#### Benchmark: I/O-bound work in threads

Now 8 requests to the local API, each of which waits 0.1 s on the server. While
a thread waits on its socket it has released the GIL, so the waits overlap.

In [18]:
request_count = 8
print(f"{request_count} requests, each waiting 0.1 s on the server")
io_baseline = 0.0
for thread_count in (1, 2, 4, 8):
    per_thread = request_count // thread_count
    seconds = run_in_threads(
        lambda repeat: [fetch_slow_page(0.1) for _ in range(repeat)], [per_thread] * thread_count
    )
    io_baseline = io_baseline or seconds
    print(
        f"{thread_count} thread(s): {seconds:.3f} s   "
        f"speedup vs 1 thread: {io_baseline / seconds:.2f}x"
    )

8 requests, each waiting 0.1 s on the server


1 thread(s): 0.817 s   speedup vs 1 thread: 1.00x


2 thread(s): 0.412 s   speedup vs 1 thread: 1.98x


4 thread(s): 0.214 s   speedup vs 1 thread: 3.82x
8 thread(s): 0.112 s   speedup vs 1 thread: 7.30x


Close to linear: 8 threads take about as long as one request. Same threads, same
GIL, opposite result; the difference is *where the time goes*.

**`sleep` releases the GIL.** While the main thread sleeps for 50 ms, a
background thread keeps counting. If `sleep` held the GIL the count would stay
at zero.

In [19]:
progress = {"count": 0}
stop_counting = threading.Event()


def count_until_stopped() -> None:
    """CPU loop that counts until the event is set."""
    while not stop_counting.is_set():
        progress["count"] += 1


counter_thread = threading.Thread(target=count_until_stopped)
counter_thread.start()
before_sleep = progress["count"]
time.sleep(0.05)  # the main thread releases the GIL here
during_sleep = progress["count"] - before_sleep
stop_counting.set()
counter_thread.join()
print(f"the other thread counted {during_sleep:,} times while the main thread slept 50 ms")

the other thread counted 1,506,813 times while the main thread slept 50 ms


**C extensions release it too.** `hashlib` releases the GIL while it hashes
buffers larger than 2 KiB, so hashing four 32 MiB buffers in four threads runs
on four cores. Hashing 100 000 tiny 64-byte strings spends its time in Python
(calls, loop) with the GIL held, so threads do not help there.

In [20]:
big_buffers = [os.urandom(32 * 1024 * 1024) for _ in range(4)]
tiny_messages = [os.urandom(64) for _ in range(100_000)]


def hash_big(index: int) -> None:
    """Hash one large buffer DEMO_SCALE times (the GIL is released inside)."""
    for _ in range(DEMO_SCALE):
        hashlib.sha256(big_buffers[index]).digest()


def hash_tiny(index: int) -> None:
    """Hash a quarter of the tiny messages (mostly Python overhead)."""
    for _ in range(DEMO_SCALE):
        for message in tiny_messages[index::4]:
            hashlib.sha256(message).digest()


def all_four(work: Callable[[int], None]) -> None:
    """Do the four pieces of work one after another (the 1-thread baseline)."""
    for index in range(4):
        work(index)


for label, work in (("4 x 32 MiB buffers", hash_big), ("100 000 x 64 B messages", hash_tiny)):
    sequential = run_in_threads(all_four, [work])
    threaded = run_in_threads(work, [0, 1, 2, 3])
    print(
        f"{label:<24} 1 thread {sequential:.3f} s | 4 threads "
        f"{threaded:.3f} s | {sequential / threaded:.2f}x"
    )
big_buffers.clear()  # free 128 MiB

4 x 32 MiB buffers       1 thread 0.060 s | 4 threads 0.017 s | 3.57x
100 000 x 64 B messages  1 thread 0.025 s | 4 threads 0.030 s | 0.82x


> **Note:** "Is my workload CPU-bound in Python, or CPU-bound in C code that
> releases the GIL?" decides whether threads help. NumPy, `hashlib`, `zlib`,
> image codecs and database drivers often release it; your own `for` loops never
> do (on the default build).

**Under the hood.** Whether this interpreter has a GIL is a build-time property
(`Py_GIL_DISABLED`) plus a runtime switch on free-threaded builds
(`sys._is_gil_enabled()`, *new in 3.13*).

In [21]:
print(f"{sysconfig.get_config_var('Py_GIL_DISABLED')=}")
print(f"{sys._is_gil_enabled()=}")
print(f"{sys.getswitchinterval()=}")

sysconfig.get_config_var('Py_GIL_DISABLED')=0
sys._is_gil_enabled()=True
sys.getswitchinterval()=0.005


### 5.1.4 Race conditions

A **race condition** is a bug whose outcome depends on how threads happen to
interleave. It needs shared mutable state and at least one writer, and it
usually hides in a *read–modify–write* or *check-then-act* sequence.

**`counter += 1` is not atomic.** One line of Python, several bytecodes: load
the value, add one, store the result. A thread switch between the load and the
store means another thread's increment is overwritten.

In [22]:
stock_counter = 0


def add_one() -> None:
    """Increment a module-level counter."""
    global stock_counter
    stock_counter += 1


print(f"# dis on {interpreter_label()} (instruction names change between versions)")
dis.dis(add_one)

# dis on CPython 3.15.1, default (GIL) build, x86_64 (instruction names change between versions)
  4           RESUME                   0

  7           LOAD_GLOBAL              0 (stock_counter)
              LOAD_SMALL_INT           1
              BINARY_OP               13 (+=)
              STORE_GLOBAL             0 (stock_counter)
              LOAD_COMMON_CONSTANT     7 (None)
              RETURN_VALUE


Read the listing as three steps: `LOAD_GLOBAL stock_counter` (read), `BINARY_OP
... (+=)` (modify), `STORE_GLOBAL stock_counter` (write). Two threads can both
read 41, both compute 42, and both store 42: one increment is lost.

**Demo 1: plain `+=` on the default build.** Four threads add 100 000 each.

In [23]:
class PlainCounter:
    """A counter with no locking at all."""

    def __init__(self) -> None:
        self.value = 0


def add_many(counter: PlainCounter, times: int) -> None:
    """Increment counter.value `times` times with +=."""
    for _ in range(times):
        counter.value += 1


plain_counter = PlainCounter()
run_in_threads(lambda _: add_many(plain_counter, 100_000), [0, 1, 2, 3])
print(f"[{interpreter_label()}] expected 400,000, got {plain_counter.value:,}")

[CPython 3.15.1, default (GIL) build, x86_64] expected 400,000, got 400,000


On CPython 3.15's default build this usually prints the right total, and that
is the trap. The eval loop only hands the GIL over at certain instructions (loop
back-edges and calls), which here fall *between* increments, never inside one.
That is an implementation detail, not a guarantee: other Python versions,
a function call inside the loop, or the free-threaded build (5.1.10 runs this
exact code there) lose updates.

**Demo 2: make the window visible.** `time.sleep(0)` between the read and the
write gives up the GIL, the way any I/O or call into slow code would. Now
updates are lost on every build.

In [24]:
class YieldingCounter:
    """Read, give up the GIL, then write: the classic lost-update race."""

    def __init__(self) -> None:
        self.value = 0

    def increment(self) -> None:
        current = self.value  # read
        time.sleep(0)  # another thread can run here
        self.value = current + 1  # write (may overwrite someone else's update)


yielding_counter = YieldingCounter()
run_in_threads(lambda _: [yielding_counter.increment() for _ in range(500)], [0, 1, 2, 3])
print(f"expected 2,000, got {yielding_counter.value:,}")

expected 2,000, got 500


**Check-then-act.** The check is correct when it runs, but the world changes
before the action. Five customers try to buy the last three phone cases; each
checks stock, then (after a pause that stands for a database round trip)
decrements it.

In [25]:
stock = {"phone-case": 3}
sales: list[str] = []


def buy(customer: str) -> None:
    """Check stock, pause, then sell: racy."""
    if stock["phone-case"] > 0:  # check
        time.sleep(0.01)  # e.g. charge the card
        stock["phone-case"] -= 1  # act
        sales.append(customer)


run_in_threads(buy, ["Asha", "Ravi", "Meena", "John", "Fatima"])
print(f"sold {len(sales)} of 3 cases; stock is now {stock['phone-case']}")

sold 5 of 3 cases; stock is now -2


All five checks saw `3 > 0` before anyone decremented, so the shop oversold and
stock went negative. The same pattern hides in lazy initialization: `if key not
in cache: cache[key] = load(key)` can call `load` once per thread.

In [26]:
cache: dict[str, str] = {}
load_calls: list[str] = []


def load_profile(user_id: str) -> str:
    """Expensive load (counts how often it runs)."""
    load_calls.append(threading.current_thread().name)
    time.sleep(0.01)
    return f"profile of {user_id}"


def get_profile(user_id: str) -> str:
    """Racy lazy cache."""
    if user_id not in cache:
        cache[user_id] = load_profile(user_id)
    return cache[user_id]


run_in_threads(get_profile, ["u-42"] * 4)
print(f"load_profile ran {len(load_calls)} times for one key")

load_profile ran 4 times for one key


#### What is atomic?

"Atomic" here means: no other thread can observe a half-done state. In CPython,
single operations on built-in containers are atomic. The default build gets that
from the GIL; the free-threaded build keeps the same guarantees with per-object
locks. This is documented in the
[FAQ](https://docs.python.org/3/faq/library.html#what-kinds-of-global-value-mutation-are-thread-safe),
but it is a CPython property, not a language promise.

| Atomic (one operation on a built-in) | Not atomic (several steps) |
| --- | --- |
| `items.append(x)`, `items.pop()`, `items[i] = x` | `counter += 1`, `items[i] += 1` |
| `mapping[key] = value`, `mapping.get(key)`, `mapping.pop(key, None)` | `if key not in mapping: mapping[key] = value` |
| `mapping.setdefault(key, value)` | `mapping[key] = mapping.get(key, 0) + 1` |
| `deque.append(x)`, `deque.popleft()` | iterating a list or dict while another thread changes it |
| `name = other_object` (rebinding) | keeping two objects consistent with each other |
| `next(it)` on built-in iterators such as `itertools.count` | `next(generator)` from two threads (raises `ValueError`) |

Rule of thumb: if correctness depends on two steps happening together, use a
lock. The next cell shows the atomic side: 4 threads × 50 000 `append`s always
give exactly 200 000 items, and `setdefault` picks exactly one winner.

In [27]:
shared_items: list[int] = []
run_in_threads(lambda _: [shared_items.append(1) for _ in range(50_000)], [0, 1, 2, 3])
print(f"{len(shared_items)=:,} (expected 200,000)")

winners: dict[str, str] = {}
claims: list[str] = []
run_in_threads(
    lambda name: claims.append(winners.setdefault("seat-1A", name)), ["Asha", "Ravi", "Meena"]
)
print(f"setdefault winner seen by every thread: {set(claims) == {winners['seat-1A']}}")

len(shared_items)=200,000 (expected 200,000)
setdefault winner seen by every thread: True


**Generators are not thread-safe.** Two threads calling `next()` on the same
generator at the same moment get `ValueError: generator already executing`.
*New in 3.15:* `threading.serialize_iterator(iterator)` wraps any iterator (or
generator) with a lock around `__next__`, and `threading.synchronized_iterator`
does the same as a decorator for generator functions. The cell is guarded so it
prints a skip message on older kernels.

In [28]:
def order_numbers(count: int) -> Iterator[int]:
    """A generator with a pause inside, so two threads overlap in next()."""
    for number in range(count):
        time.sleep(0.0005)
        yield number


def drain(source: Iterator[int], taken: list[int], errors: list[str]) -> None:
    """Consume the shared iterator, recording values and errors."""
    try:
        for number in source:
            taken.append(number)
    except ValueError as error:
        errors.append(str(error))


def share_between_two_threads(shared_source: Iterator[int]) -> str:
    """Let two threads drain one iterator; summarize what they got."""
    taken: list[int] = []
    errors: list[str] = []
    run_in_threads(lambda _: drain(shared_source, taken, errors), [0, 1])
    return (
        f"items taken={len(taken)}, no duplicates={len(set(taken)) == len(taken)}, errors={errors}"
    )


print(f"bare generator:     {share_between_two_threads(order_numbers(200))}")
if hasattr(threading, "serialize_iterator"):
    print(
        "serialize_iterator: "
        f"{share_between_two_threads(threading.serialize_iterator(order_numbers(200)))}"
    )
else:
    print("serialize_iterator: SKIP (new in 3.15)")

bare generator:     items taken=200, no duplicates=True, errors=['generator already executing']


serialize_iterator: items taken=200, no duplicates=True, errors=[]


### 5.1.5 Synchronization primitives

| Primitive | What it does | Typical use | JS analogue |
| --- | --- | --- | --- |
| `Lock` | one holder at a time; not reentrant | protect a read–modify–write | `Atomics` on a `SharedArrayBuffer`, or a promise-based mutex library |
| `RLock` | like `Lock`, but the holding thread may re-acquire it | recursive or layered code that locks the same object | — |
| `Semaphore(n)` | at most `n` holders | cap concurrent calls to a service | `p-limit` |
| `BoundedSemaphore(n)` | a semaphore that raises if released more than acquired | the same, with a bug check: prefer it | — |
| `Event` | a flag threads can wait for | "start now", "stop now" | a promise that resolves once |
| `Condition` | a lock plus wait/notify on a predicate | "wait until stock > 0" | `Atomics.wait` / `notify` |
| `Barrier(n)` | blocks until `n` threads arrive | phases that must start together | `Promise.all` used as a rendezvous |
| `Timer(delay, f)` | a thread that calls `f` after `delay` unless cancelled | timeouts, debouncing | `setTimeout` / `clearTimeout` |

All of them are context managers where it makes sense (`with lock:`), which
guarantees the release even when the body raises.

#### `Lock`

The lost-update counter from 5.1.4, fixed: the read and the write happen while
the lock is held, so no other thread can interleave.

In [29]:
class LockedYieldingCounter:
    """The yielding counter from 5.1.4 with its read-modify-write under a lock."""

    def __init__(self) -> None:
        self.value = 0
        self._lock = threading.Lock()

    def increment(self) -> None:
        with self._lock:
            current = self.value
            time.sleep(0)  # other threads may run, but cannot enter this block
            self.value = current + 1


locked_counter = LockedYieldingCounter()
run_in_threads(lambda _: [locked_counter.increment() for _ in range(500)], [0, 1, 2, 3])
print(f"expected 2,000, got {locked_counter.value:,}")

expected 2,000, got 2,000


`acquire()` can also be non-blocking or time-limited, and `locked()` reports the
state. A plain `Lock` is **not reentrant**: the thread holding it blocks forever
if it tries again. Here the second attempt uses `blocking=False` so the cell
does not hang.

In [30]:
inventory_lock = threading.Lock()
print(f"{inventory_lock.acquire()=}  {inventory_lock.locked()=}")
print(f"second acquire, same thread, non-blocking: {inventory_lock.acquire(blocking=False)}")
from_other_thread: list[bool] = []
run_in_threads(lambda _: from_other_thread.append(inventory_lock.acquire(timeout=0.05)), [0])
print(f"acquire(timeout=0.05) from another thread: {from_other_thread[0]}")
inventory_lock.release()
print(f"after release: {inventory_lock.locked()=}")

inventory_lock.acquire()=True  inventory_lock.locked()=True
second acquire, same thread, non-blocking: False
acquire(timeout=0.05) from another thread: False
after release: inventory_lock.locked()=False


#### `RLock`

A reentrant lock counts how many times its owner acquired it and frees it only
after the same number of releases. It fits recursive code: totalling a category
tree locks the tree at every level. With a plain `Lock` the recursive call would
deadlock; the `timeout` in the plain-lock version turns that into a `False`.

In [31]:
category_tree = {"Tea": {"Green": {}, "Black": {"Assam": {}, "Darjeeling": {}}}}


def count_categories(tree: dict[str, dict], lock: threading.Lock | threading.RLock) -> int:
    """Count nodes, taking the lock at every level of recursion."""
    if not lock.acquire(timeout=0.1):
        raise TimeoutError("could not re-acquire the lock (it is not reentrant)")
    try:
        return sum(1 + count_categories(children, lock) for children in tree.values())
    finally:
        lock.release()


print(f"RLock: {count_categories(category_tree, threading.RLock())} categories")
print(f"Lock:  {raised_message(lambda: count_categories(category_tree, threading.Lock()))}")

RLock: 5 categories


Lock:  TimeoutError: could not re-acquire the lock (it is not reentrant)


#### `Semaphore`

A semaphore holds a counter: `acquire()` decrements it (blocking at zero),
`release()` increments it. Ten threads call a payment gateway that allows three
concurrent requests; the semaphore keeps the peak at three.

In [32]:
gateway_slots = threading.Semaphore(3)
in_flight = {"now": 0, "peak": 0}
in_flight_lock = threading.Lock()


def charge_card(order_id: int) -> None:
    """Call the 'gateway' while holding one of the three slots."""
    with gateway_slots:
        with in_flight_lock:
            in_flight["now"] += 1
            in_flight["peak"] = max(in_flight["peak"], in_flight["now"])
        time.sleep(0.02)  # the network call
        with in_flight_lock:
            in_flight["now"] -= 1


seconds = run_in_threads(charge_card, list(range(10)))
print(f"peak concurrent charges: {in_flight['peak']} (limit 3); 10 charges took {seconds:.2f} s")

peak concurrent charges: 3 (limit 3); 10 charges took 0.08 s


#### `BoundedSemaphore`

A plain `Semaphore` accepts extra `release()` calls silently, so a bug that
releases twice raises the limit for good. `BoundedSemaphore` raises
`ValueError` instead. Prefer it whenever releases should match acquires.

In [33]:
def acquirable(semaphore: threading.Semaphore) -> int:
    """How many non-blocking acquires succeed (then release them all again)."""
    count = 0
    while semaphore.acquire(blocking=False):
        count += 1
    for _ in range(count):
        semaphore.release()
    return count


loose = threading.Semaphore(2)
loose.release()  # a bug: one release too many
print(f"Semaphore(2) after an extra release allows {acquirable(loose)} holders")

strict = threading.BoundedSemaphore(2)
print(f"BoundedSemaphore(2) extra release: {raised_message(strict.release)}")

Semaphore(2) after an extra release allows 3 holders
BoundedSemaphore(2) extra release: ValueError: Semaphore released too many times


#### `Event`

An event is a boolean flag with `set()`, `clear()`, `is_set()` and
`wait(timeout)`. Workers block on `wait()` until the main thread sets it: all
three start at (almost) the same moment. `wait(timeout)` returns `False` if the
timeout passes first.

In [34]:
shop_open = threading.Event()
start_times: dict[str, float] = {}


def cashier(name: str) -> None:
    """Wait for the shop to open, then record the start time."""
    shop_open.wait()
    start_times[name] = time.perf_counter()


cashiers = [
    threading.Thread(target=cashier, args=(name,)) for name in ("till-1", "till-2", "till-3")
]
for till in cashiers:
    till.start()
time.sleep(0.05)
print(f"before set: {shop_open.is_set()=}, started={len(start_times)}")
shop_open.set()
for till in cashiers:
    till.join()
spread_ms = (max(start_times.values()) - min(start_times.values())) * 1000
print(f"after set: started={len(start_times)}, spread {spread_ms:.2f} ms")
shop_open.clear()
print(f"after clear, wait(timeout=0.02) returns {shop_open.wait(timeout=0.02)}")

before set: shop_open.is_set()=False, started=0
after set: started=3, spread 0.15 ms


after clear, wait(timeout=0.02) returns False


#### `Condition`

A condition variable combines a lock with "wait until something is true".
`wait_for(predicate)` releases the lock while waiting and re-checks the
predicate after every `notify`. Here two packers wait for stock; a delivery
thread adds items and calls `notify_all()`.

In [35]:
stock_arrived = threading.Condition()
warehouse: list[str] = []
packed: list[str] = []


def packer(name: str, wanted: int) -> None:
    """Wait until enough items exist, then take them."""
    with stock_arrived:
        stock_arrived.wait_for(lambda: len(warehouse) >= wanted, timeout=2)
        taken = [warehouse.pop() for _ in range(min(wanted, len(warehouse)))]
        packed.append(f"{name} packed {len(taken)}")


def deliver(items: list[str]) -> None:
    """Add items one by one, waking the waiters each time."""
    for item in items:
        time.sleep(0.01)
        with stock_arrived:
            warehouse.append(item)
            stock_arrived.notify_all()


run_in_threads(
    lambda job: job(),
    [lambda: packer("packer-A", 2), lambda: packer("packer-B", 3), lambda: deliver(list("abcde"))],
)
print(f"{sorted(packed)}, left in warehouse: {warehouse}")

['packer-A packed 2', 'packer-B packed 3'], left in warehouse: []


> **Note:** always wait in a loop (or with `wait_for`): a thread can wake up
> without its condition being true (another thread took the stock first, or a
> spurious wake-up). `notify()` wakes one waiter, `notify_all()` wakes all.

#### `Barrier`

A barrier for `n` parties blocks every caller of `wait()` until `n` have
arrived, then releases them together; `wait()` returns a distinct index
`0 … n-1` and an optional `action` runs once per round. If a party never
arrives, `wait(timeout)` breaks the barrier and every waiter gets
`BrokenBarrierError`.

In [36]:
report_ready = threading.Barrier(3, action=lambda: print("action: all 3 parts loaded, merging"))
arrival_indexes: list[int] = []


def load_part(seconds: float) -> None:
    """Load a part of a report, then wait for the others."""
    time.sleep(seconds)
    arrival_indexes.append(report_ready.wait())


run_in_threads(load_part, [0.01, 0.03, 0.02])
print(f"indexes returned by wait(): {sorted(arrival_indexes)}")

lonely = threading.Barrier(2)
print(f"only 1 of 2 parties arrives: {raised_error_name(lambda: lonely.wait(timeout=0.05))}")
print(f"{lonely.broken=}")

action: all 3 parts loaded, merging
indexes returned by wait(): [0, 1, 2]
only 1 of 2 parties arrives: BrokenBarrierError
lonely.broken=True


#### `Timer`

`threading.Timer(delay, function, args)` is a `Thread` subclass that waits
`delay` seconds, then calls the function, unless `cancel()` was called first.

In [37]:
reminders: list[str] = []
fires = threading.Timer(0.05, reminders.append, args=("payment reminder sent",))
cancelled = threading.Timer(0.05, reminders.append, args=("this never happens",))
fires.start()
cancelled.start()
cancelled.cancel()  # before the delay is over
fires.join()
cancelled.join()
print(f"{reminders=}  {isinstance(fires, threading.Thread)=}")

reminders=['payment reminder sent']  isinstance(fires, threading.Thread)=True


#### Deadlocks and prevention

A **deadlock** is a set of threads each waiting for a lock another one holds.
It needs all four (Coffman) conditions: mutual exclusion, hold-and-wait, no
preemption, and a **circular wait**. Remove any one and deadlock is impossible.

| Prevention | How | Breaks |
| --- | --- | --- |
| Lock ordering | every thread takes locks in one global order (for example by id) | circular wait |
| Try-lock with back-off | `acquire(timeout=...)`; on failure release everything, wait a random moment, retry | hold-and-wait |
| One coarser lock | protect both resources with a single lock | hold-and-wait (at the cost of concurrency) |
| Don't hold a lock while calling unknown code | callbacks, I/O and other objects' methods may take locks of their own | hidden cycles |
| Share by message passing | hand work to one owner thread through a `Queue` (5.1.6) | mutual exclusion on shared data |

The demo: one thread updates the *inventory* then the *ledger*, another the
ledger then the inventory. A `Barrier` makes both take their first lock before
either asks for its second, so the circular wait happens every time; the
`acquire(timeout=0.2)` turns the hang into a reported failure.

In [38]:
def lock_pair_naive(
    first: threading.Lock, second: threading.Lock, both_hold_one: threading.Barrier
) -> str:
    """Take first, wait until the other thread holds its first lock, then try second."""
    with first:
        both_hold_one.wait()
        if not second.acquire(timeout=0.2):
            return "gave up: deadlock"
        second.release()
        return "done"


inventory_lock, ledger_lock = threading.Lock(), threading.Lock()
rendezvous = threading.Barrier(2)
outcomes: list[str] = []
run_in_threads(
    lambda locks: outcomes.append(lock_pair_naive(*locks, rendezvous)),
    [(inventory_lock, ledger_lock), (ledger_lock, inventory_lock)],
)
print(f"{outcomes=}")

outcomes=['gave up: deadlock', 'done']


Both threads held one lock and waited for the other: a circular wait. The first
thread to time out gave up and released its lock, and only then could the other
finish. Without the timeout both would wait forever (and the kernel would hang).

**Fix with try-lock and back-off.** If the second lock is not free soon, release
the first, sleep a short *random* time so the two threads stop colliding, and
try again. (Lab 5.1 fixes a deadlock the other way, with lock ordering.)

In [39]:
def lock_pair_backoff(
    first: threading.Lock, second: threading.Lock, both_hold_one: threading.Barrier, seed: int
) -> str:
    """Acquire both locks or neither; retry with random back-off."""
    backoff = random.Random(seed)
    for attempt in range(1, 20):
        with first:
            if attempt == 1:
                both_hold_one.wait()  # force the collision on the first try
            if second.acquire(timeout=0.05):
                second.release()
                return f"done on attempt {attempt}"
        time.sleep(backoff.uniform(0.0, 0.05))  # first is released here
    return "gave up"


rendezvous = threading.Barrier(2)
outcomes = []
run_in_threads(
    lambda job: outcomes.append(lock_pair_backoff(job[0], job[1], rendezvous, seed=job[2])),
    [(inventory_lock, ledger_lock, 1), (ledger_lock, inventory_lock, 2)],
)
print(f"{sorted(outcomes)=}")

sorted(outcomes)=['done on attempt 1', 'done on attempt 2']


### 5.1.6 Thread-safe queues

The `queue` module gives thread-safe queues with the locking built in, which is
usually the simplest correct design: threads share a queue instead of sharing
data structures.

| Class / method | Behaviour |
| --- | --- |
| `queue.Queue(maxsize=0)` | FIFO; `maxsize > 0` makes `put` block when full (back-pressure) |
| `queue.LifoQueue` | LIFO (a stack) |
| `queue.PriorityQueue` | smallest item first (a heap); put `(priority, tie_breaker, item)` tuples |
| `put(item, block=True, timeout=None)`, `put_nowait` | `queue.Full` when it cannot add |
| `get(block=True, timeout=None)`, `get_nowait` | `queue.Empty` when nothing arrives |
| `task_done()`, `join()` | consumers mark items finished; `join()` waits until every `put` has a `task_done` |
| `qsize()`, `empty()`, `full()` | snapshots only: another thread may change the answer immediately |
| `shutdown(immediate=False)` | *new in 3.13*: later `put`s raise `queue.ShutDown`; `get` raises it once the queue is drained |

**FIFO, LIFO and priority order** for the same three items:

In [40]:
for queue_class in (queue.Queue, queue.LifoQueue):
    tickets = queue_class()
    for ticket in ("T-1 login bug", "T-2 typo", "T-3 outage"):
        tickets.put(ticket)
    print(f"{queue_class.__name__:<10} -> {[tickets.get() for _ in range(3)]}")

by_priority: queue.PriorityQueue[tuple[int, int, str]] = queue.PriorityQueue()
arrival = itertools.count()  # tie-breaker: equal priorities keep arrival order
for priority, ticket in (
    (3, "T-1 login bug"),
    (5, "T-2 typo"),
    (1, "T-3 outage"),
    (3, "T-4 slow page"),
):
    by_priority.put((priority, next(arrival), ticket))
print(f"PriorityQueue -> {[by_priority.get()[2] for _ in range(4)]}")

Queue      -> ['T-1 login bug', 'T-2 typo', 'T-3 outage']
LifoQueue  -> ['T-3 outage', 'T-2 typo', 'T-1 login bug']
PriorityQueue -> ['T-3 outage', 'T-1 login bug', 'T-4 slow page', 'T-2 typo']


The tie-breaker matters: without it, two equal priorities would compare the
next tuple element, and two items that cannot be compared (dicts, custom objects)
raise `TypeError` inside the queue.

**Bounded queues and timeouts.** `maxsize` makes a fast producer wait for a slow
consumer instead of filling memory. The non-blocking and timeout forms raise
`queue.Full` and `queue.Empty`.

In [41]:
small_buffer: queue.Queue[str] = queue.Queue(maxsize=2)
small_buffer.put("line 1")
small_buffer.put("line 2")
print(
    f"{small_buffer.full()=}  put_nowait: "
    f"{raised_error_name(lambda: small_buffer.put_nowait('line 3'))}"
)
print(f"put with timeout: {raised_error_name(lambda: small_buffer.put('line 3', timeout=0.02))}")
print(f"{small_buffer.get()=}  {small_buffer.get()=}")
print(
    f"get with timeout on empty queue: {raised_error_name(lambda: small_buffer.get(timeout=0.02))}"
)

small_buffer.full()=True  put_nowait: Full
put with timeout: Full
small_buffer.get()='line 1'  small_buffer.get()='line 2'
get with timeout on empty queue: Empty


**`task_done` and `join`.** Every `put` increments an "unfinished tasks"
counter; every `task_done()` decrements it; `queue.join()` blocks until it is
zero. That lets the producer wait until the work is *processed*, not just taken
off the queue.

In [42]:
resize_jobs: queue.Queue[str] = queue.Queue()
resized: list[str] = []


def resize_worker() -> None:
    """Process jobs forever (a daemon thread); mark each one done."""
    while True:
        image_name = resize_jobs.get()
        time.sleep(0.01)
        resized.append(image_name)
        resize_jobs.task_done()


threading.Thread(target=resize_worker, daemon=True, name="resizer").start()
for image_name in ("a.png", "b.png", "c.png"):
    resize_jobs.put(image_name)
print(f"right after put: {resize_jobs.unfinished_tasks=}")
resize_jobs.join()
print(f"after join:      {resize_jobs.unfinished_tasks=}, {resized=}")

right after put: resize_jobs.unfinished_tasks=3
after join:      resize_jobs.unfinished_tasks=0, resized=['a.png', 'b.png', 'c.png']


That worker never ends: it is a daemon, so it would simply be killed at exit.

**Sentinel shutdown.** To stop consumers cleanly, put one *sentinel* value per
consumer after the real work. A unique `object()` is safer than `None`, which
could be real data. Each consumer exits when it takes a sentinel, so `join()` on
the threads returns.

In [43]:
STOP = object()  # sentinel: compared by identity
log_lines: queue.Queue[object] = queue.Queue()
error_counts: dict[str, int] = {}
counts_lock = threading.Lock()


def count_errors() -> None:
    """Consume log lines until the sentinel arrives."""
    while (line := log_lines.get()) is not STOP:
        if " ERROR " in line:
            reader = threading.current_thread().name
            with counts_lock:
                error_counts[reader] = error_counts.get(reader, 0) + 1
        log_lines.task_done()
    log_lines.task_done()  # the sentinel counts as a task too


consumers = [threading.Thread(target=count_errors, name=f"log-reader-{n}") for n in (1, 2)]
for consumer in consumers:
    consumer.start()
for number in range(1, 101):
    log_lines.put(
        f"2026-11-16 10:00:{number % 60:02d} "
        f"{'ERROR' if number % 10 == 0 else 'INFO'} request {number}"
    )
for _ in consumers:
    log_lines.put(STOP)
for consumer in consumers:
    consumer.join()
print(
    f"total errors: {sum(error_counts.values())}; "
    f"threads alive: {[c.is_alive() for c in consumers]}"
)

total errors: 10; threads alive: [False, False]


**`Queue.shutdown()`** (*new in 3.13*) replaces the sentinel dance: after
`shutdown()`, `put` raises `queue.ShutDown`, and `get` keeps returning the
remaining items, then raises `queue.ShutDown` once the queue is empty.
`shutdown(immediate=True)` drops the remaining items instead.

In [44]:
if sys.version_info >= (3, 13):
    closing_queue: queue.Queue[str] = queue.Queue()
    for order_id in ("A-1", "A-2"):
        closing_queue.put(order_id)
    closing_queue.shutdown()
    print(f"put after shutdown: {raised_error_name(lambda: closing_queue.put('A-3'))}")
    print(
        f"drain: {closing_queue.get()}, {closing_queue.get()}, "
        f"then {raised_error_name(closing_queue.get)}"
    )
else:
    print("SKIP: Queue.shutdown is new in 3.13")

put after shutdown: ShutDown
drain: A-1, A-2, then ShutDown


### 5.1.7 Thread-local state

`threading.local()` creates an object whose attributes are **separate for every
thread**: each thread sees only what it set itself. Web frameworks used this for
"the current request" (Flask's `request`); in asyncio code the equivalent is
`contextvars` (Unit 5.3), and in Node it is `AsyncLocalStorage`.

Three request-handling threads store their user on a shared `local` object; a
deeply nested function reads it without the value being passed around.

In [45]:
request_context = threading.local()
audit_log: list[str] = []


def write_audit(action: str) -> None:
    """Deep in the call stack: read the user of *this* thread's request."""
    audit_log.append(f"{request_context.user} {action}")


def handle_request(user: str) -> None:
    """Set the per-thread context, then do work that needs it."""
    request_context.user = user
    time.sleep(0.01)  # other threads set their own user meanwhile
    write_audit("viewed orders")


run_in_threads(handle_request, ["asha", "ravi", "meena"])
print(sorted(audit_log))
print(f"main thread: {raised_message(lambda: request_context.user)}")

['asha viewed orders', 'meena viewed orders', 'ravi viewed orders']
main thread: AttributeError: '_thread._local' object has no attribute 'user'


The main thread never set `user`, so it gets `AttributeError`. Subclass `local`
to give every thread defaults: `__init__` runs once **per thread**, the first
time that thread touches the object.

A real use: a `sqlite3` connection may only be used by the thread that created
it (`check_same_thread=True`), so each worker thread keeps its own connection.

In [46]:
class PerThreadDatabase(threading.local):
    """One in-memory SQLite connection per thread, created on first use."""

    def __init__(self) -> None:
        self.connection = sqlite3.connect(":memory:")
        self.owner = threading.current_thread().name


shared_connection = sqlite3.connect(":memory:")
print("shared connection used from another thread:")
run_in_threads(
    lambda _: print(" ", raised_error_name(lambda: shared_connection.execute("select 1"))), [0]
)

per_thread_db = PerThreadDatabase()
connections_seen: list[sqlite3.Connection] = []


def query_in_thread(_: object) -> None:
    """Use this thread's own connection (created on first access)."""
    connections_seen.append(per_thread_db.connection)
    per_thread_db.connection.execute("select 1")


run_in_threads(query_in_thread, [0, 1])
print(f"each worker got its own connection: {connections_seen[0] is not connections_seen[1]}")
print(f"the main thread's is a third one: {per_thread_db.connection not in connections_seen}")
print(f"main thread owner: {per_thread_db.owner}")

shared connection used from another thread:
  ProgrammingError
each worker got its own connection: True
the main thread's is a third one: True
main thread owner: MainThread


> **Warning:** thread pools reuse threads, so thread-local values **leak from
> one task to the next** on the same worker. Reset them at the start of each
> task (or use `contextvars`, which executors and asyncio copy per task).

In [47]:
task_local = threading.local()


def tag_task(task_name: str) -> str:
    """Report what the previous task on this thread left behind, then tag it."""
    leftover = getattr(task_local, "task", None)
    task_local.task = task_name
    return f"{task_name} saw leftover {leftover!r}"


with concurrent.futures.ThreadPoolExecutor(max_workers=1) as single_worker:
    print(list(single_worker.map(tag_task, ["task-1", "task-2", "task-3"])))

['task-1 saw leftover None', "task-2 saw leftover 'task-1'", "task-3 saw leftover 'task-2'"]


### 5.1.8 `concurrent.futures.ThreadPoolExecutor`

An executor owns a pool of worker threads and a work queue. You hand it
callables; it hands back `Future` objects, which hold the eventual result **or
exception**. This is the high-level API to reach for first.

| API | What it does |
| --- | --- |
| `ThreadPoolExecutor(max_workers=None, thread_name_prefix="")` | pool; threads start lazily as work arrives |
| `executor.submit(fn, *args, **kwargs)` | schedule one call; returns a `Future` immediately |
| `executor.map(fn, *iterables, timeout=None, buffersize=None)` | results **in input order**; raises a task's exception when you reach it |
| `as_completed(futures, timeout=None)` | yields futures **in completion order** |
| `wait(futures, timeout=None, return_when=ALL_COMPLETED)` | returns `(done, not_done)` sets; also `FIRST_COMPLETED`, `FIRST_EXCEPTION` |
| `future.result(timeout)`, `exception(timeout)` | block for the outcome; `result()` re-raises the task's exception |
| `future.done()`, `running()`, `cancelled()`, `cancel()` | state; only a *pending* future can be cancelled |
| `future.add_done_callback(fn)` | call `fn(future)` when it finishes |
| `executor.shutdown(wait=True, cancel_futures=False)` | stop accepting work; `with` calls it for you |

**`submit` and `Future`.** `slow_echo` waits on the local API for the given
time, then returns its first argument. (In JS terms: `submit` is like calling an
async function; the `Future` is the promise. But `result()` *blocks* the
calling thread instead of being awaited.)

In [48]:
def slow_echo(value: str, seconds: float) -> str:
    """Wait `seconds` on the local API, then return value."""
    fetch_slow_page(seconds)
    return value


with concurrent.futures.ThreadPoolExecutor(max_workers=2, thread_name_prefix="shop") as executor:
    invoice_future = executor.submit(slow_echo, "invoice-17", 0.2)
    time.sleep(0.02)
    print(f"just after submit: done={invoice_future.done()}, running={invoice_future.running()}")
    print(f"result(timeout=5) -> {invoice_future.result(timeout=5)!r}")
    print(
        f"afterwards:        done={invoice_future.done()}, exception={invoice_future.exception()}"
    )
    print(
        f"worker thread name: {executor.submit(lambda: threading.current_thread().name).result()}"
    )

just after submit: done=False, running=True


result(timeout=5) -> 'invoice-17'
afterwards:        done=True, exception=None
worker thread name: shop_0


**Cancelling and callbacks.** With one worker, the first task runs and the
others wait in the queue. `cancel()` succeeds only for a task that has not
started. A done-callback runs in the worker thread that finished the task (or
immediately in the caller if the future is already done).

In [49]:
finished: list[str] = []
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    running_job = executor.submit(slow_echo, "job-1", 0.1)
    queued_job = executor.submit(slow_echo, "job-2", 0.1)
    doomed_job = executor.submit(slow_echo, "job-3", 0.1)
    for job in (running_job, queued_job, doomed_job):
        job.add_done_callback(
            lambda done: finished.append("cancelled" if done.cancelled() else done.result())
        )
    time.sleep(0.02)
    print(f"cancel running job-1: {running_job.cancel()}")
    print(f"cancel queued job-3:  {doomed_job.cancel()}")
print(f"callbacks saw: {finished}")

cancel running job-1: False
cancel queued job-3:  True


callbacks saw: ['cancelled', 'job-1', 'job-2']


**`map`: input order, not finish order.** The three tasks finish in the order
0.1 s, 0.2 s, 0.3 s, but `map` yields results in the order you passed them. The
total time is the slowest task, not the sum. `timeout` applies to the whole
iteration and raises `TimeoutError` (the built-in, since 3.11) when exceeded.

In [50]:
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    with Timer("map over 0.3 s, 0.1 s, 0.2 s tasks"):
        print(list(executor.map(slow_echo, ["slow", "fast", "medium"], [0.3, 0.1, 0.2])))
    too_slow = executor.map(slow_echo, ["late"], [0.3], timeout=0.05)
    print(f"with timeout=0.05: {raised_error_name(lambda: list(too_slow))}")

['slow', 'fast', 'medium']
map over 0.3 s, 0.1 s, 0.2 s tasks took 304.46 ms
with timeout=0.05: TimeoutError


**`map` with `buffersize`** (*new in 3.14*). Plain `map` submits a task for
every input item straight away, so it reads the whole input iterable up front
(an infinite generator would hang). `buffersize=n` keeps at most `n` tasks in
flight and reads more input only as you consume results.

In [51]:
def numbered_rows(limit: int, consumed: list[int]) -> Iterator[int]:
    """Yield 0..limit-1, recording how many rows were read."""
    for row_number in range(limit):
        consumed.append(row_number)
        yield row_number


if sys.version_info >= (3, 14):
    for buffer_size in (None, 4):
        consumed_rows: list[int] = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
            squares = executor.map(
                lambda n: n * n, numbered_rows(10_000, consumed_rows), buffersize=buffer_size
            )
            first_square = next(squares)
            print(
                f"buffersize={buffer_size}: first result {first_square}, "
                f"rows read so far: {len(consumed_rows):,}"
            )
else:
    print("SKIP: Executor.map(buffersize=) is new in 3.14")

buffersize=None: first result 0, rows read so far: 10,000
buffersize=4: first result 0, rows read so far: 5


**`as_completed`: finish order.** Use it when you want to handle each result as
soon as it is ready (show progress, start follow-up work). Keep a dict from
future to input so you know which task finished.

In [52]:
delays = {"report-A": 0.3, "report-B": 0.1, "report-C": 0.2}
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    future_to_name = {
        executor.submit(slow_echo, name, seconds): name for name, seconds in delays.items()
    }
    print(
        "completion order:",
        [future.result() for future in concurrent.futures.as_completed(future_to_name)],
    )
    stragglers = [executor.submit(slow_echo, "late", 0.3)]
    waiting = concurrent.futures.as_completed(stragglers, timeout=0.05)
    print(f"as_completed(timeout=0.05): {raised_error_name(lambda: list(waiting))}")

completion order: ['report-B', 'report-C', 'report-A']
as_completed(timeout=0.05): TimeoutError


**`wait`.** Returns two sets, `done` and `not_done`, when the condition is met.
`FIRST_COMPLETED` is handy for "take the fastest mirror"; `FIRST_EXCEPTION`
stops waiting as soon as one task fails.

In [53]:
def fail_after(seconds: float) -> str:
    """Raise after a short wait."""
    time.sleep(seconds)
    raise ConnectionError("mirror unreachable")


with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    mirrors = [
        executor.submit(slow_echo, f"mirror-{n}", seconds)
        for n, seconds in enumerate((0.3, 0.1, 0.2))
    ]
    done, not_done = concurrent.futures.wait(
        mirrors, return_when=concurrent.futures.FIRST_COMPLETED
    )
    print(f"FIRST_COMPLETED: done={[f.result() for f in done]}, still running={len(not_done)}")

    mixed = [executor.submit(fail_after, 0.05), executor.submit(slow_echo, "slow", 0.3)]
    done, not_done = concurrent.futures.wait(mixed, return_when=concurrent.futures.FIRST_EXCEPTION)
    print(
        f"FIRST_EXCEPTION: done={len(done)} "
        f"({next(iter(done)).exception()!r}), not done={len(not_done)}"
    )

    done, not_done = concurrent.futures.wait(mirrors + mixed)  # ALL_COMPLETED
    print(f"ALL_COMPLETED: done={len(done)}, not done={len(not_done)}")

FIRST_COMPLETED: done=['mirror-1'], still running=2
FIRST_EXCEPTION: done=1 (ConnectionError('mirror unreachable')), not done=1


ALL_COMPLETED: done=5, not done=0


**Sizing.** The default `max_workers` is `min(32, (os.process_cpu_count() or 1)
+ 4)` (*changed in 3.13*: it used `os.cpu_count()` before). The right number
depends on the work:

| Work | Size the pool | Why |
| --- | --- | --- |
| I/O-bound (HTTP, database) | about the concurrency you want: throughput × latency (Little's law), capped by what the server allows | waiting threads cost little |
| C code that releases the GIL (hashing, compression, NumPy) | about the number of cores | more threads than cores just queue up |
| Pure-Python CPU work (default build) | threads do not help | use processes (5.2) or a free-threaded build |

The cell prints the default (via the private `_max_workers` attribute, for
demonstration only) and times 16 requests of 0.05 s each with 1, 4 and 16
workers.

In [54]:
print(
    f"{os.process_cpu_count()=}, default "
    f"max_workers={concurrent.futures.ThreadPoolExecutor()._max_workers}"
)
for worker_count in (1, 4, 16):
    with concurrent.futures.ThreadPoolExecutor(max_workers=worker_count) as executor:
        with Timer(f"16 requests x 0.05 s, {worker_count:>2} workers", report=None) as timing:
            list(executor.map(fetch_slow_page, [0.05] * 16))
    print(
        f"{timing.label}: {timing.elapsed_seconds:.2f} s "
        f"(ideal {0.05 * -(-16 // worker_count):.2f} s)"
    )

os.process_cpu_count()=16, default max_workers=20


16 requests x 0.05 s,  1 workers: 0.85 s (ideal 0.80 s)


16 requests x 0.05 s,  4 workers: 0.22 s (ideal 0.20 s)
16 requests x 0.05 s, 16 workers: 0.07 s (ideal 0.05 s)


**Shutdown.** Leaving the `with` block calls `shutdown(wait=True)`: it waits for
every submitted task. `shutdown(wait=False, cancel_futures=True)` cancels
everything still queued and returns at once (running tasks still finish).
After shutdown, `submit` raises `RuntimeError`.

In [55]:
executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
backlog = [executor.submit(slow_echo, f"email-{n}", 0.05) for n in range(10)]
time.sleep(0.02)
executor.shutdown(wait=False, cancel_futures=True)
print(
    f"cancelled: {sum(f.cancelled() for f in backlog)}, ran "
    f"or running: {sum(not f.cancelled() for f in backlog)}"
)
print(f"submit after shutdown: {raised_message(lambda: executor.submit(print))}")
started = [future.result() for future in backlog if not future.cancelled()]  # let them finish

cancelled: 9, ran or running: 1
submit after shutdown: RuntimeError: cannot schedule new futures after shutdown


**Exceptions.** A task's exception is stored in its future. `exception()`
returns it; `result()` re-raises the *same* object, with the worker's traceback
attached. The danger is the opposite case: a fire-and-forget `submit` whose
future nobody looks at swallows the error silently.

In [56]:
def parse_quantity(text: str) -> int:
    """Parse a quantity; fails on bad input."""
    return int(text)


with concurrent.futures.ThreadPoolExecutor() as executor:
    bad = executor.submit(parse_quantity, "three")
    stored = bad.exception()
    print(f"exception(): {stored!r}")
    try:
        bad.result()
    except ValueError as error:
        last_frame = traceback.extract_tb(error.__traceback__)[-1].name
        print(
            "result() re-raised the same object: "
            f"{error is stored}; traceback ends in {last_frame!r}"
        )
    executor.submit(parse_quantity, "seven")  # fire and forget: the ValueError vanishes
print("no error was printed for 'seven': always check futures (or use map / as_completed)")

exception(): ValueError("invalid literal for int() with base 10: 'three'")
result() re-raised the same object: True; traceback ends in 'parse_quantity'
no error was printed for 'seven': always check futures (or use map / as_completed)


### 5.1.9 Errors and shutdown

An exception that escapes a thread's `run()` does **not** reach the thread that
called `join()`. The thread dies, `threading.excepthook` is called, and the rest
of the program carries on. The default hook prints `Exception in thread NAME:`
and the traceback to stderr. Seen from a script in a child process:

In [57]:
print(
    run_child("""
    import threading

    def import_prices():
        raise KeyError("sku-404")

    worker = threading.Thread(target=import_prices, name="price-importer")
    worker.start()
    worker.join()
    print("main thread: join() returned normally, worker.is_alive() =", worker.is_alive())
""")
)

main thread: join() returned normally, worker.is_alive() = False
Exception in thread price-importer:
Traceback (most recent call last):
  File "<stdlib>/threading.py", line 1218, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "<stdlib>/threading.py", line 1160, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<scratch>/child_script.py", line 4, in import_prices
    raise KeyError("sku-404")
KeyError: 'sku-404'


**A custom `threading.excepthook`** receives one `ExceptHookArgs` argument with
`exc_type`, `exc_value`, `exc_traceback` and `thread`. Install one to log thread
failures properly (or to collect them, as here). Restore the previous hook
afterwards; `threading.__excepthook__` always holds the original.

In [58]:
thread_failures: list[str] = []


def collect_failure(hook_args: threading.ExceptHookArgs) -> None:
    """Record which thread failed and why, instead of printing a traceback."""
    thread_name = hook_args.thread.name if hook_args.thread else "?"
    thread_failures.append(f"{thread_name}: {hook_args.exc_type.__name__}: {hook_args.exc_value}")


def import_prices() -> None:
    """A thread body that fails."""
    raise KeyError("sku-404")


previous_hook = threading.excepthook
threading.excepthook = collect_failure
try:
    failing = threading.Thread(target=import_prices, name="price-importer")
    failing.start()
    failing.join()
finally:
    threading.excepthook = previous_hook
print(f"{thread_failures=}")
print(f"{threading.__excepthook__ is not collect_failure=}")

thread_failures=["price-importer: KeyError: 'sku-404'"]
threading.__excepthook__ is not collect_failure=True


**Daemon threads at shutdown, again.** A daemon thread is frozen mid-step when
the interpreter exits: buffered data is lost and `finally` blocks never run. The
child script below starts two writers that append a line every 10 ms and write
a footer in `finally`. One is a daemon; the other is stopped by a flag the main
thread sets before exiting. Then the parent reads both files.

In [59]:
WRITERS_DEMO = """
    import threading, time

    stop = threading.Event()

    def write_lines(path, stop_flag):
        with open(path, "w") as output:
            try:
                for number in range(1000):
                    if stop_flag is not None and stop_flag.is_set():
                        break
                    output.write(f"line {number}\\n")
                    time.sleep(0.01)
            finally:
                output.write("FOOTER\\n")

    threading.Thread(target=write_lines, args=("daemon.txt", None), daemon=True).start()
    flagged = threading.Thread(target=write_lines, args=("flagged.txt", stop))
    flagged.start()
    time.sleep(0.05)
    stop.set()  # ask the non-daemon writer to finish
    print("main thread exiting")
"""
print(run_child(WRITERS_DEMO))
for name in ("daemon.txt", "flagged.txt"):
    lines = (PHASE5_DIR / name).read_text().splitlines()
    print(f"{name:<12} {len(lines)} lines, last line: {lines[-1] if lines else '(empty file)'!r}")

main thread exiting
daemon.txt   0 lines, last line: '(empty file)'
flagged.txt  6 lines, last line: 'FOOTER'


The daemon's file has no footer (and may even be empty: its buffered lines were
never flushed). The flagged writer stopped at a clean point and closed its file.

**Stop flags.** Python has no `thread.kill()`: killing a thread could leave
locks held and data half-written. Stopping is cooperative: the thread checks a
flag. Use an `Event` and wait on it with a timeout instead of `time.sleep`, so
the thread wakes up the moment the flag is set.

In [60]:
def poll_with_sleep(stop: threading.Event) -> None:
    """Responds to stop only after the current sleep ends."""
    while not stop.is_set():
        time.sleep(0.3)


def poll_with_wait(stop: threading.Event) -> None:
    """Event.wait returns as soon as stop is set."""
    while not stop.wait(timeout=0.3):
        pass


for poller in (poll_with_sleep, poll_with_wait):
    stop_flag = threading.Event()
    poller_thread = threading.Thread(target=poller, args=(stop_flag,))
    poller_thread.start()
    time.sleep(0.05)
    with Timer(f"{poller.__name__}: time from set() to exit", report=None) as stop_timing:
        stop_flag.set()
        poller_thread.join()
    print(f"{stop_timing.label}: {stop_timing.elapsed_seconds * 1000:.0f} ms")

poll_with_sleep: time from set() to exit: 250 ms
poll_with_wait: time from set() to exit: 0 ms


> **Note:** `KeyboardInterrupt` (Ctrl+C) and every signal handler run only in
> the **main** thread. A worker blocked in `queue.get()` or `lock.acquire()`
> never sees Ctrl+C, so the main thread must set the stop flag (or put
> sentinels) when it catches `KeyboardInterrupt`, then join the workers.

### 5.1.10 Free-threaded Python

| | |
| --- | --- |
| [PEP 703](https://peps.python.org/pep-0703/) | Making the GIL optional: a separate *free-threaded* build of CPython. Experimental in 3.13. |
| [PEP 779](https://peps.python.org/pep-0779/) | Criteria for "supported" status. Since 3.14 the free-threaded build is officially supported (phase II), still a separate build, not the default. |
| Executable | `python3.15t` (the `t` suffix); install with `uv python install 3.15t` |
| Detect the build | `sysconfig.get_config_var("Py_GIL_DISABLED") == 1`, `sys.abiflags` contains `t` |
| Detect the GIL at run time | `sys._is_gil_enabled()` (*new in 3.13*); can be `True` on a free-threaded build |
| Force it | `PYTHON_GIL=0` / `PYTHON_GIL=1`, or `-X gil=0` / `-X gil=1` |

**How it works without the GIL** (Unit 3.1 showed the memory side): biased
reference counting (the owning thread updates its count cheaply; other threads
use atomic operations), immortal objects, per-object locks inside `dict`, `list`
and friends (so the "atomic" table in 5.1.4 still holds), and the `mimalloc`
allocator. The price is some single-threaded speed and memory.

**Detect and probe.** The next cell runs the same probe in this kernel's
interpreter and in `python3.15t` (and once more with `PYTHON_GIL=1`), or prints
a skip message when no free-threaded interpreter is installed.

In [61]:
BUILD_PROBE = """
    import sys, sysconfig
    print("Py_GIL_DISABLED:", sysconfig.get_config_var("Py_GIL_DISABLED"))
    print("sys.abiflags:", repr(sys.abiflags))
    print("extension suffix:", sysconfig.get_config_var("EXT_SUFFIX"))
    print("sys._is_gil_enabled():", sys._is_gil_enabled())
    print("threads inherit contextvars:", bool(sys.flags.thread_inherit_context))
"""
print(f"--- this kernel [{interpreter_label()}]")
print(run_child(BUILD_PROBE))
if FREE_THREADED_PYTHON is None:
    print("SKIP: no free-threaded interpreter found (install one with: uv python install 3.15t)")
else:
    print("--- python3.15t")
    print(run_child(BUILD_PROBE, python=FREE_THREADED_PYTHON))
    print("--- python3.15t with PYTHON_GIL=1")
    print(run_child(BUILD_PROBE, python=FREE_THREADED_PYTHON, extra_env={"PYTHON_GIL": "1"}))

--- this kernel [CPython 3.15.1, default (GIL) build, x86_64]
Py_GIL_DISABLED: 0
sys.abiflags: ''
extension suffix: .cpython-315-x86_64-linux-gnu.so
sys._is_gil_enabled(): True
threads inherit contextvars: False
--- python3.15t
Py_GIL_DISABLED: 1
sys.abiflags: 't'
extension suffix: .cpython-315t-x86_64-linux-gnu.so
sys._is_gil_enabled(): False
threads inherit contextvars: True
--- python3.15t with PYTHON_GIL=1
Py_GIL_DISABLED: 1
sys.abiflags: 't'
extension suffix: .cpython-315t-x86_64-linux-gnu.so
sys._is_gil_enabled(): True
threads inherit contextvars: True


**Compatibility.**

- **Pure-Python packages** run unchanged.
- **C extensions** need a separate wheel (ABI tag `cp315t`; note the different
  extension suffix above) **and** must declare that they are safe without the
  GIL (the `Py_mod_gil` slot). Importing an extension that does not declare it
  **re-enables the GIL** for the whole process and emits a `RuntimeWarning`.
  Check `sys._is_gil_enabled()` after your imports in production.
- **Your own code**: races that the GIL happened to hide become real. The plain
  `counter.value += 1` from 5.1.4, unchanged, in both builds:

In [62]:
PLAIN_INCREMENT_RACE = """
    import sys, threading

    class PlainCounter:
        value = 0

    counter = PlainCounter()

    def add_many(times):
        for _ in range(times):
            counter.value += 1

    threads = [threading.Thread(target=add_many, args=(100_000,)) for _ in range(4)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    print(f"GIL enabled: {sys._is_gil_enabled()!s:<5}  expected 400,000, got {counter.value:,}")
"""
print(run_child(PLAIN_INCREMENT_RACE))
if FREE_THREADED_PYTHON is None:
    print("SKIP: no free-threaded interpreter found")
else:
    print(run_child(PLAIN_INCREMENT_RACE, python=FREE_THREADED_PYTHON))

GIL enabled: True   expected 400,000, got 400,000
GIL enabled: False  expected 400,000, got 160,483


**Benchmark.** Each thread runs the same pure-Python loop (1 000 000 iterations
× `DEMO_SCALE`); 1, 2, 4 and 8 threads run at once. *Throughput speedup* =
threads × (time for one thread) ÷ (time for all threads): 1.0 means no gain,
`n` means perfect scaling. The last line compares single-thread speed between
the builds. Expect about 1.0× everywhere on the default build, and scaling up
to your core count on 3.15t.

In [63]:
SCALING_BENCHMARK = """
    import json, os, threading, time

    LOOPS = 1_000_000 * int(os.environ.get("PHASE5_DEMO_SCALE", "1"))

    def spin():
        total = 0
        for number in range(LOOPS):
            total += number * number

    def wall_seconds(thread_count):
        threads = [threading.Thread(target=spin) for _ in range(thread_count)]
        start = time.perf_counter()
        for thread in threads:
            thread.start()
        for thread in threads:
            thread.join()
        return time.perf_counter() - start

    print(json.dumps({count: wall_seconds(count) for count in (1, 2, 4, 8)}))
"""


def scaling_table(python: str) -> dict[str, float]:
    """Run the benchmark in a child interpreter; return {thread count: seconds}."""
    return json.loads(run_child(SCALING_BENCHMARK, python=python).splitlines()[-1])


builds = {"default build": scaling_table(sys.executable)}
if FREE_THREADED_PYTHON is None:
    print("SKIP: no free-threaded interpreter found; showing the default build only")
else:
    builds["free-threaded"] = scaling_table(FREE_THREADED_PYTHON)
print(f"CPU cores available: {os.process_cpu_count()}")
for build_name, seconds in builds.items():
    cells = [
        f"{count}T {seconds[count]:.2f}s ({int(count) * seconds['1'] / seconds[count]:.1f}x)"
        for count in seconds
    ]
    print(f"{build_name:<14} " + " | ".join(cells))
if "free-threaded" in builds:
    print(
        "single-thread time, free-threaded / default: "
        f"{builds['free-threaded']['1'] / builds['default build']['1']:.2f}"
    )

CPU cores available: 16
default build  1T 0.03s (1.0x) | 2T 0.07s (1.0x) | 4T 0.14s (1.0x) | 8T 0.32s (0.9x)
free-threaded  1T 0.04s (1.0x) | 2T 0.04s (1.9x) | 4T 0.04s (3.5x) | 8T 0.07s (4.2x)
single-thread time, free-threaded / default: 1.05


On a machine with 4 or more cores the free-threaded row should approach 4× at
four threads, while the default row stays near 1×. The single-thread ratio shows
the cost of running without the GIL: usually a little above 1.0.

| Choose the free-threaded build when | Stay on the default build when |
| --- | --- |
| CPU-bound pure-Python work must run in parallel **and** share large in-memory data | the work is I/O-bound (threads or asyncio already work) |
| every C extension you need ships `cp315t` wheels | a key dependency has no free-threaded wheel (the GIL would come back anyway) |
| you can test for races you never saw before | single-thread speed matters most |

### 5.1.11 JS bridge

**libuv's thread pool.** Node runs your JavaScript on one thread. Network sockets
never use a thread: libuv watches them with `epoll` / `kqueue` / IOCP. Work the
OS cannot do asynchronously goes to libuv's **thread pool**: 4 threads by
default, set with the `UV_THREADPOOL_SIZE` environment variable before startup
(maximum 1024). It runs `fs.*` calls, `dns.lookup`, `crypto.pbkdf2`,
`crypto.scrypt`, `crypto.randomBytes` (async form), async `zlib`, and C++ addon
work. Your JavaScript itself never runs in that pool.

**`worker_threads`.** Each `Worker` is a new V8 isolate with its **own heap,
own event loop and own globals**. Workers communicate by `postMessage`
(structured clone copies), transfer `ArrayBuffer`s, or share memory explicitly
through a `SharedArrayBuffer` with `Atomics`.

| Node | Python | Precise difference |
| --- | --- | --- |
| libuv pool runs C++ work for built-in modules | `ThreadPoolExecutor` runs **any** callable, including Python code | Python code in the pool still takes turns on the GIL (default build); C code that releases it runs in parallel, like libuv's pool |
| `UV_THREADPOOL_SIZE=4` (process-wide, set before start) | `max_workers=` per executor; asyncio's default executor is `min(32, cpus + 4)` | you can have several pools of different sizes |
| `fs.readFile(cb)` goes to the pool automatically | `open().read()` blocks the calling thread; in asyncio use `await asyncio.to_thread(...)` | nothing is offloaded unless you ask |
| `worker_threads`: separate heap, message passing | `threading.Thread`: **one shared heap**, no copying | threads can race on any object; Workers cannot |
| `worker_threads` (isolation + parallelism) | subinterpreters (5.2.6): own GIL, own modules, message passing | the closest match to a Worker |
| `child_process.fork`, `cluster` | `multiprocessing`, `ProcessPoolExecutor` (5.2) | Python pickles messages; Node uses JSON/structured clone over IPC |
| `SharedArrayBuffer` + `Atomics` | `multiprocessing.shared_memory` + locks (5.2.5) | |

**The pool, side by side.** In Node, four `crypto.pbkdf2()` calls run in
parallel on the libuv pool. In Python, `hashlib.pbkdf2_hmac` releases the GIL
while OpenSSL works, so a `ThreadPoolExecutor` of 4 gets the same parallelism.

In [64]:
def derive_key(password: bytes) -> str:
    """PBKDF2-HMAC-SHA256 with many iterations (the GIL is released inside)."""
    iterations = 300_000 * DEMO_SCALE
    return hashlib.pbkdf2_hmac("sha256", password, b"per-user-salt", iterations).hex()[:12]


passwords = [b"asha-pass", b"ravi-pass", b"meena-pass", b"john-pass"]
with Timer("4 keys, one after another") as one_by_one:
    sequential_keys = [derive_key(password) for password in passwords]
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as key_pool:
    with Timer("4 keys, pool of 4 threads") as pooled:
        pooled_keys = list(key_pool.map(derive_key, passwords))
print(
    f"same keys: {pooled_keys == sequential_keys}; speedup "
    f"{one_by_one.elapsed_seconds / pooled.elapsed_seconds:.2f}x"
)

4 keys, one after another took 125.36 ms
4 keys, pool of 4 threads took 38.27 ms
same keys: True; speedup 3.28x


**From asyncio** (Unit 5.3 explains the event loop), `asyncio.to_thread(f, *args)`
runs `f` in the loop's default thread pool and gives back an awaitable. That is
the closest thing to `util.promisify(crypto.pbkdf2)`: blocking or GIL-releasing
work leaves the event loop thread free. The kernel already runs a loop, so the
cell uses top-level `await`.

In [65]:
with Timer("asyncio.gather of 4 to_thread calls"):
    async_keys = await asyncio.gather(
        *(asyncio.to_thread(derive_key, password) for password in passwords)
    )
print(f"same keys again: {async_keys == sequential_keys}")

asyncio.gather of 4 to_thread calls took 38.81 ms
same keys again: True


#### Predict the output

**Predict the output.**

```python
import threading

def work():
    print("working in", threading.current_thread().name)

worker = threading.Thread(target=work(), name="worker")
worker.start()
worker.join()
```

<details><summary>Answer</summary>

`working in MainThread`. `target=work()` **calls** `work` immediately, in the
main thread, and passes its return value (`None`) as the target. The `worker`
thread then starts with no target and does nothing. Pass the function itself:
`target=work`. (In JS the same slip is `setTimeout(work(), 0)`.)

</details>

Run the next cell to check your prediction.

In [66]:
def work() -> None:
    print("working in", threading.current_thread().name)


worker = threading.Thread(target=work(), name="worker")
worker.start()
worker.join()

working in MainThread


**Predict the output.**

```python
from concurrent.futures import ThreadPoolExecutor
import time

def nap(seconds):
    time.sleep(seconds)
    return seconds

with ThreadPoolExecutor(max_workers=3) as executor:
    print(list(executor.map(nap, [0.3, 0.1, 0.2])))
```

<details><summary>Answer</summary>

`[0.3, 0.1, 0.2]`. The tasks *finish* in the order 0.1, 0.2, 0.3, but `map`
always yields results in **input order**. Use `as_completed` for finish order.
The whole call takes about 0.3 s (the longest task), not 0.6 s.

</details>

Run the next cell to check your prediction.

In [67]:
def nap(seconds: float) -> float:
    time.sleep(seconds)
    return seconds


with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    print(list(executor.map(nap, [0.3, 0.1, 0.2])))

[0.3, 0.1, 0.2]


**Predict the output.**

```python
import threading

def explode():
    raise ValueError("boom")

worker = threading.Thread(target=explode)
worker.start()
worker.join()
print("joined")
```

<details><summary>Answer</summary>

The thread's `ValueError` does **not** propagate to `join()`. The default
`threading.excepthook` prints `Exception in thread Thread-N (explode):` and the
traceback to stderr, then the main thread prints `joined`. The proof cell
installs a temporary hook that prints one line instead of the full traceback.

</details>

Run the next cell to check your prediction.

In [68]:
def explode() -> None:
    raise ValueError("boom")


def one_line_hook(hook_args: threading.ExceptHookArgs) -> None:
    print(f"excepthook: {hook_args.exc_type.__name__}: {hook_args.exc_value}")


saved_hook, threading.excepthook = threading.excepthook, one_line_hook
try:
    worker = threading.Thread(target=explode)
    worker.start()
    worker.join()
    print("joined")
finally:
    threading.excepthook = saved_hook

excepthook: ValueError: boom
joined


### Lab 5.1

Six exercises; replace each `raise NotImplementedError`, then run the check cell.
Until you finish, the checks print FAIL; that is expected. The setup cell below
defines the helpers the exercises use: `UnsafeCounter` (exercise 1),
`Account`, `transfer_naive`, `acquire_or_fail` and `DeadlockError`
(exercise 5), and `run_threads` for the checks.

In [69]:
import concurrent.futures
import functools
import inspect
import itertools
import json
import math
import os
import queue
import subprocess
import sys
import threading
import time
from collections.abc import Callable, Iterator
from contextlib import contextmanager
from dataclasses import dataclass, field

import httpx


def run_threads(work: Callable[[object], object], items: list[object]) -> None:
    """Run work(item) in one thread per item and join them all (a helper for the checks)."""
    threads = [threading.Thread(target=work, args=(item,)) for item in items]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()


class UnsafeCounter:
    """Exercise 1: reads, gives up the GIL, then writes, so concurrent updates get lost."""

    def __init__(self) -> None:
        self.value = 0

    def increment(self) -> None:
        current = self.value
        time.sleep(0)  # let another thread run between the read and the write
        self.value = current + 1


class DeadlockError(RuntimeError):
    """Exercise 5: raised when a lock cannot be acquired in time."""


def acquire_or_fail(lock: threading.Lock, timeout: float = 0.5) -> None:
    """Acquire lock or raise DeadlockError after timeout seconds (so a deadlock cannot hang)."""
    if not lock.acquire(timeout=timeout):
        raise DeadlockError(f"could not acquire a lock within {timeout} s")


@dataclass(eq=False)
class Account:
    """Exercise 5: a bank account with its own lock."""

    account_id: int
    balance: int
    lock: threading.Lock = field(default_factory=threading.Lock, repr=False)


def transfer_naive(
    source: Account,
    target: Account,
    amount: int,
    pause: Callable[[], None] = lambda: time.sleep(0.05),
) -> None:
    """Exercise 5: lock source, pause, lock target, move money. Deadlocks on opposite transfers."""
    acquire_or_fail(source.lock)
    try:
        pause()  # widen the window so the deadlock shows every time
        acquire_or_fail(target.lock)
        try:
            source.balance -= amount
            target.balance += amount
        finally:
            target.lock.release()
    finally:
        source.lock.release()


print(
    "Lab 5.1 helpers ready: UnsafeCounter, Account, transfer_naive, acquire_or_fail, DeadlockError"
)

Lab 5.1 helpers ready: UnsafeCounter, Account, transfer_naive, acquire_or_fail, DeadlockError


#### Exercise 1 — prove a race condition and fix it

1. `hammer(counter, thread_count, increments_per_thread)` starts `thread_count`
   threads, each calling `counter.increment()` `increments_per_thread` times,
   joins them all and returns `counter.value`. Run on an `UnsafeCounter`, it
   should return **less** than `thread_count × increments_per_thread`: that
   proves the race.
2. `LockedCounter` has a `value` attribute starting at 0 and an `increment()`
   that does the same read / `time.sleep(0)` / write as `UnsafeCounter`, but
   inside `with self._lock:` (a `threading.Lock` created in `__init__`).

In [70]:
def hammer(counter: object, thread_count: int, increments_per_thread: int) -> int:
    """Increment counter from several threads at once; return the final value."""
    raise NotImplementedError


class LockedCounter:
    """UnsafeCounter's read / sleep(0) / write, made safe with a Lock."""

    def __init__(self) -> None:
        raise NotImplementedError

    def increment(self) -> None:
        raise NotImplementedError

Check exercise 1:

In [71]:
def threads_left_behind() -> int:
    """How many more threads are alive after hammer() than before."""
    before = threading.active_count()
    hammer(LockedCounter(), 3, 10)
    return threading.active_count() - before


run_checks(
    [
        (
            "race proven: UnsafeCounter loses updates",
            lambda: hammer(UnsafeCounter(), 4, 200) < 800,
            True,
        ),
        ("LockedCounter: 4 threads x 200", lambda: hammer(LockedCounter(), 4, 200), 800),
        ("LockedCounter: 1 thread x 50", lambda: hammer(LockedCounter(), 1, 50), 50),
        (
            "LockedCounter keeps a threading.Lock",
            lambda: any(
                isinstance(value, type(threading.Lock()))
                for value in vars(LockedCounter()).values()
            ),
            True,
        ),
        ("hammer joins every thread it starts", threads_left_behind, 0),
    ]
)

FAIL  race proven: UnsafeCounter loses updates: raised NotImplementedError: 
FAIL  LockedCounter: 4 threads x 200: raised NotImplementedError: 
FAIL  LockedCounter: 1 thread x 50: raised NotImplementedError: 
FAIL  LockedCounter keeps a threading.Lock: raised NotImplementedError: 
FAIL  hammer joins every thread it starts: raised NotImplementedError: 
0/5 checks passed


#### Exercise 2 — producer–consumer with `Queue`

`process_with_workers(items, handle, worker_count)` returns
`[handle(item) for item in items]`, computed by `worker_count` consumer threads:

- put `(index, item)` pairs on a `queue.Queue`;
- start `worker_count` threads named `consumer-1`, `consumer-2`, …; each takes
  pairs, stores `handle(item)` at its index and calls `task_done()`;
- shut down with **one sentinel per consumer**, then join every consumer;
- return the results in input order.

In [72]:
def process_with_workers[T, R](
    items: list[T], handle: Callable[[T], R], worker_count: int
) -> list[R]:
    """Run handle over items with worker_count consumer threads fed by a Queue."""
    raise NotImplementedError

Check exercise 2:

In [73]:
def square_slowly(number: int) -> int:
    """A handler that takes a few milliseconds."""
    time.sleep(0.005)
    return number * number


def consumer_names_used(worker_count: int) -> tuple[int, bool]:
    """How many distinct threads ran the handler, and whether all were named consumer-N."""
    seen: set[str] = set()

    def record(number: int) -> int:
        seen.add(threading.current_thread().name)
        time.sleep(0.01)
        return number

    process_with_workers(list(range(20)), record, worker_count)
    return len(seen), all(name.startswith("consumer-") for name in seen)


def consumers_still_alive() -> list[str]:
    """Names of consumer threads alive after process_with_workers returns."""
    process_with_workers(list(range(10)), square_slowly, 3)
    return [thread.name for thread in threading.enumerate() if thread.name.startswith("consumer-")]


run_checks(
    [
        (
            "results in input order",
            lambda: process_with_workers(list(range(12)), square_slowly, 4),
            [n * n for n in range(12)],
        ),
        (
            "works with any handler",
            lambda: process_with_workers(["tea", "mug"], str.upper, 2),
            ["TEA", "MUG"],
        ),
        ("empty input", lambda: process_with_workers([], square_slowly, 3), []),
        ("all 4 consumers took work, named consumer-N", lambda: consumer_names_used(4), (4, True)),
        ("no consumer thread left alive", consumers_still_alive, []),
        ("uses queue.Queue", lambda: "Queue(" in inspect.getsource(process_with_workers), True),
    ]
)

FAIL  results in input order: raised NotImplementedError: 
FAIL  works with any handler: raised NotImplementedError: 
FAIL  empty input: raised NotImplementedError: 
FAIL  all 4 consumers took work, named consumer-N: raised NotImplementedError: 
FAIL  no consumer thread left alive: raised NotImplementedError: 
FAIL  uses queue.Queue: expected True, got False
0/6 checks passed


#### Exercise 3 — a thread-pool downloader

`download_all(client, urls, max_workers)` downloads every URL with
`client.get(url)` on a `ThreadPoolExecutor(max_workers)`, collects results with
`as_completed`, and returns a dict from each URL to:

- the number of bytes in the body, for a successful (2xx) response;
- the exception class name otherwise: call `response.raise_for_status()` and
  catch `httpx.HTTPError`, so a 404 gives `"HTTPStatusError"` and a refused
  connection gives `"ConnectError"`.

The checks use an `httpx.Client` with a `MockTransport` that waits 50 ms per
request and records how many requests were in flight at once.

In [74]:
def download_all(
    client: httpx.Client, urls: list[str], max_workers: int = 8
) -> dict[str, int | str]:
    """Download urls concurrently; map each to its byte count or error class name."""
    raise NotImplementedError

Check exercise 3:

In [75]:
def lab_client(latency: float = 0.05) -> tuple[httpx.Client, dict[str, int]]:
    """A client whose transport sleeps per request and tracks the peak in-flight count."""
    stats = {"now": 0, "peak": 0}
    stats_lock = threading.Lock()

    def handler(request: httpx.Request) -> httpx.Response:
        with stats_lock:
            stats["now"] += 1
            stats["peak"] = max(stats["peak"], stats["now"])
        try:
            time.sleep(latency)
            path = request.url.path
            if path == "/broken":
                raise httpx.ConnectError("connection refused", request=request)
            if path.startswith("/files/"):
                return httpx.Response(200, content=b"x" * int(path.rsplit("/", 1)[1]))
            return httpx.Response(404)
        finally:
            with stats_lock:
                stats["now"] -= 1

    return httpx.Client(transport=httpx.MockTransport(handler)), stats


def peak_in_flight(url_count: int, max_workers: int) -> int:
    """Download url_count files and return the peak number of concurrent requests."""
    client, stats = lab_client()
    download_all(client, [f"http://lab.test/files/{n}" for n in range(url_count)], max_workers)
    return stats["peak"]


FILE_URLS = ["http://lab.test/files/10", "http://lab.test/files/2048"]
BAD_URLS = ["http://lab.test/missing", "http://lab.test/broken"]
run_checks(
    [
        (
            "byte counts",
            lambda: download_all(lab_client()[0], FILE_URLS, 4),
            {FILE_URLS[0]: 10, FILE_URLS[1]: 2048},
        ),
        (
            "errors become class names",
            lambda: download_all(lab_client()[0], BAD_URLS, 4),
            {BAD_URLS[0]: "HTTPStatusError", BAD_URLS[1]: "ConnectError"},
        ),
        ("12 URLs, 4 workers: concurrent but capped", lambda: 1 < peak_in_flight(12, 4) <= 4, True),
        ("1 worker: one request at a time", lambda: peak_in_flight(4, 1), 1),
        ("no URLs", lambda: download_all(lab_client()[0], [], 4), {}),
        (
            "uses ThreadPoolExecutor and as_completed",
            lambda: all(
                word in inspect.getsource(download_all)
                for word in ("ThreadPoolExecutor", "as_completed")
            ),
            True,
        ),
    ]
)

FAIL  byte counts: raised NotImplementedError: 
FAIL  errors become class names: raised NotImplementedError: 
FAIL  12 URLs, 4 workers: concurrent but capped: raised NotImplementedError: 
FAIL  1 worker: one request at a time: raised NotImplementedError: 
FAIL  no URLs: raised NotImplementedError: 
FAIL  uses ThreadPoolExecutor and as_completed: expected True, got False
0/6 checks passed


#### Exercise 4 — a semaphore-capped connection pool

`ConnectionPool(size, connect)` hands out at most `size` connections at a time:

- `connection()` is a context manager (use `@contextmanager`) that blocks while
  `size` connections are in use (a `threading.BoundedSemaphore(size)`), then
  yields an idle connection, or a new one from `connect()` if none is idle;
  on exit (even after an exception) the connection goes back to the idle list
  and the semaphore is released;
- attributes `created` (how many times `connect()` ran), `in_use` (connections
  currently handed out) and `peak_in_use` (the highest `in_use` seen). Protect
  the counters and the idle list with a `Lock`.

In [76]:
class ConnectionPool:
    """At most `size` connections in use; idle connections are reused."""

    def __init__(self, size: int, connect: Callable[[], object]) -> None:
        raise NotImplementedError

    @contextmanager
    def connection(self) -> Iterator[object]:
        raise NotImplementedError
        yield  # (makes this a generator; remove when you implement it)

Check exercise 4:

In [77]:
def make_pool(size: int) -> ConnectionPool:
    """A pool whose connections are numbered strings."""
    numbers = itertools.count(1)
    return ConnectionPool(size, lambda: f"conn-{next(numbers)}")


def busy_pool(size: int, users: int) -> tuple[int, int, int]:
    """`users` threads each hold a connection for 20 ms; return (peak, created, in_use)."""
    pool = make_pool(size)

    def use(_: object) -> None:
        with pool.connection():
            time.sleep(0.02)

    run_threads(use, list(range(users)))
    return pool.peak_in_use, pool.created, pool.in_use


def sequential_reuse() -> list[object]:
    """Five uses one after another should reuse a single connection."""
    pool = make_pool(3)
    handed_out = []
    for _ in range(5):
        with pool.connection() as connection:
            handed_out.append(connection)
    return handed_out


def released_after_error() -> tuple[int, object]:
    """An exception inside the with block must still return the connection."""
    pool = make_pool(1)
    try:
        with pool.connection():
            raise ValueError("query failed")
    except ValueError:
        pass
    with pool.connection() as connection:  # would block forever if the slot leaked
        return pool.in_use, connection


run_checks(
    [
        (
            "12 users, size 3: peak 3, at most 3 created, all returned",
            lambda: busy_pool(3, 12),
            (3, 3, 0),
        ),
        ("sequential uses reuse one connection", sequential_reuse, ["conn-1"] * 5),
        ("exception releases the connection", released_after_error, (1, "conn-1")),
        (
            "uses a BoundedSemaphore",
            lambda: any(
                isinstance(value, threading.BoundedSemaphore)
                for value in vars(make_pool(2)).values()
            ),
            True,
        ),
    ]
)

FAIL  12 users, size 3: peak 3, at most 3 created, all returned: raised NotImplementedError: 
FAIL  sequential uses reuse one connection: raised NotImplementedError: 
FAIL  exception releases the connection: raised NotImplementedError: 
FAIL  uses a BoundedSemaphore: raised NotImplementedError: 
0/4 checks passed


#### Exercise 5 — reproduce and fix a deadlock

`transfer_naive` (setup cell) locks the source account, pauses 50 ms, then locks
the target. `acquire_or_fail` raises `DeadlockError` after 0.5 s instead of
hanging.

1. `run_opposite_transfers(transfer)` creates `Account(1, 100)` and
   `Account(2, 100)`, runs `transfer(account_1, account_2, 10)` and
   `transfer(account_2, account_1, 20)` in two threads at the same time, and
   returns `{"outcomes": sorted list of "ok" or the exception class name per
   thread, "balances": (account_1.balance, account_2.balance)}`. With
   `transfer_naive` the outcomes include `"DeadlockError"`: that reproduces it.
2. `transfer_ordered(source, target, amount, pause=...)` fixes it with **lock
   ordering**: acquire the two locks (with `acquire_or_fail`) in order of
   `account_id`, call `pause()` between the two acquisitions, move the money,
   release both.

In [78]:
def run_opposite_transfers(transfer: Callable[[Account, Account, int], None]) -> dict[str, object]:
    """Run two opposite transfers at once; report outcomes and final balances."""
    raise NotImplementedError


def transfer_ordered(
    source: Account,
    target: Account,
    amount: int,
    pause: Callable[[], None] = lambda: time.sleep(0.05),
) -> None:
    """Move amount from source to target, locking accounts in account_id order."""
    raise NotImplementedError

Check exercise 5:

In [79]:
class RecordingLock:
    """A lock that records the account id each time it is acquired."""

    def __init__(self, account_id: int, log: list[int]) -> None:
        self._lock, self._account_id, self._log = threading.Lock(), account_id, log

    def acquire(self, blocking: bool = True, timeout: float = -1) -> bool:
        acquired = self._lock.acquire(blocking, timeout)
        if acquired:
            self._log.append(self._account_id)
        return acquired

    def release(self) -> None:
        self._lock.release()


def lock_order_used() -> list[int]:
    """Account ids in the order transfer_ordered locked them, transferring 2 -> 1."""
    log: list[int] = []
    first = Account(1, 50, lock=RecordingLock(1, log))
    second = Account(2, 50, lock=RecordingLock(2, log))
    transfer_ordered(second, first, 5, pause=lambda: None)
    return log


run_checks(
    [
        (
            "naive transfers deadlock",
            lambda: "DeadlockError" in run_opposite_transfers(transfer_naive)["outcomes"],
            True,
        ),
        (
            "ordered transfers both succeed",
            lambda: run_opposite_transfers(transfer_ordered)["outcomes"],
            ["ok", "ok"],
        ),
        (
            "ordered transfers move the money",
            lambda: run_opposite_transfers(transfer_ordered)["balances"],
            (110, 90),
        ),
        ("lower account_id is locked first", lock_order_used, [1, 2]),
    ]
)

FAIL  naive transfers deadlock: raised NotImplementedError: 
FAIL  ordered transfers both succeed: raised NotImplementedError: 
FAIL  ordered transfers move the money: raised NotImplementedError: 
FAIL  lower account_id is locked first: raised NotImplementedError: 
0/4 checks passed


#### Exercise 6 — GIL vs free-threaded CPU benchmark

`measure_scaling(python_path, thread_count=4, loops=1_000_000)` runs a benchmark
**in a child interpreter** (`subprocess.run([python_path, "-c", source], ...)`).
The child script you write times a pure-Python loop of `loops` iterations in one
thread, then `thread_count` threads each running the same loop at once, and
prints one JSON line. Return a dict with exactly these keys:

- `"gil_enabled"`: the child's `sys._is_gil_enabled()`;
- `"one_thread_seconds"`, `"all_threads_seconds"`: the two wall times;
- `"speedup"`: `thread_count * one_thread_seconds / all_threads_seconds`.

The checks run it on this kernel's interpreter and, if installed, on
`python3.15t` (`FREE_THREADED_PYTHON` from the unit setup). Speedups are only
printed, except one loose sanity check: the default build cannot reach 1.5×.

In [80]:
def measure_scaling(
    python_path: str, thread_count: int = 4, loops: int = 1_000_000
) -> dict[str, object]:
    """Benchmark 1 vs thread_count CPU-bound threads in a child interpreter."""
    raise NotImplementedError

Check exercise 6 (the free-threaded check prints a skip line when 3.15t is missing):

In [81]:
@functools.cache
def default_build_scaling() -> dict[str, object]:
    """Benchmark this kernel's interpreter once (exceptions are not cached)."""
    return measure_scaling(sys.executable)


def free_threaded_reports_no_gil() -> bool:
    """True if 3.15t reports the GIL disabled, or if no 3.15t is installed."""
    if FREE_THREADED_PYTHON is None:
        print("SKIP  free-threaded interpreter not installed")
        return True
    return measure_scaling(FREE_THREADED_PYTHON)["gil_enabled"] is False


def speedup_formula_holds() -> bool:
    """speedup == 4 * one / all for the default-build result."""
    result = default_build_scaling()
    expected = 4 * result["one_thread_seconds"] / result["all_threads_seconds"]
    return math.isclose(result["speedup"], expected, rel_tol=1e-6)


run_checks(
    [
        (
            "returns the four keys",
            lambda: sorted(default_build_scaling()),
            ["all_threads_seconds", "gil_enabled", "one_thread_seconds", "speedup"],
        ),
        (
            "default build reports the GIL enabled",
            lambda: default_build_scaling()["gil_enabled"],
            True,
        ),
        ("speedup = threads x one / all", speedup_formula_holds, True),
        (
            "default build: no real speedup (< 1.5x)",
            lambda: default_build_scaling()["speedup"] < 1.5,
            True,
        ),
        ("3.15t reports the GIL disabled", free_threaded_reports_no_gil, True),
    ]
)

FAIL  returns the four keys: raised NotImplementedError: 
FAIL  default build reports the GIL enabled: raised NotImplementedError: 
FAIL  speedup = threads x one / all: raised NotImplementedError: 
FAIL  default build: no real speedup (< 1.5x): raised NotImplementedError: 
FAIL  3.15t reports the GIL disabled: raised NotImplementedError: 
0/5 checks passed


Once the checks pass, this cell prints your measurements side by side. Explain
them in one sentence each: why the default build stays near 1×, and what limits
the free-threaded build (core count, the single-thread overhead, other load on
the machine).

In [82]:
try:
    results = {"default build": default_build_scaling()}
    if FREE_THREADED_PYTHON is not None:
        results["free-threaded"] = measure_scaling(FREE_THREADED_PYTHON)
    print(f"CPU cores available: {os.process_cpu_count()}")
    for build_name, result in results.items():
        print(
            f"{build_name:<14} GIL={result['gil_enabled']!s:<5} "
            f"1 thread {result['one_thread_seconds']:.3f} s, "
            f"4 threads {result['all_threads_seconds']:.3f} s, speedup {result['speedup']:.2f}x"
        )
except NotImplementedError:
    print("Finish exercise 6 first.")

Finish exercise 6 first.


### Recap — Unit 5.1

| Concept | One-line takeaway | JS / Node equivalent |
| --- | --- | --- |
| Concurrency vs parallelism | Interleaving needs one core; running at the same instant needs several (and no GIL in the way) | one event loop vs `worker_threads` |
| CPU- vs I/O-bound | Measure `thread_time` / wall time; threads help I/O and GIL-releasing C code | libuv pool helps `fs`/`crypto` |
| `threading.Thread` | `target=` a function, `start`, `join`; no return value; daemon threads die at exit | `new Worker()` (but isolated) |
| The GIL | Protects CPython's internals, not your invariants; released for I/O, `sleep` and cooperative C code; handed over every 5 ms | — |
| Race conditions | `+=` and check-then-act are several bytecodes; built-in single operations are atomic | `Atomics` |
| Primitives | `Lock`, `RLock`, `(Bounded)Semaphore`, `Event`, `Condition`, `Barrier`, `Timer`; always `with` | `p-limit`, `Atomics.wait` |
| Deadlocks | Break circular wait with lock ordering, or hold-and-wait with try-lock + back-off | — |
| Queues | `Queue`/`LifoQueue`/`PriorityQueue`; `task_done` + `join`; sentinels or `shutdown()` (3.13) | message channels |
| `threading.local` | Per-thread attributes; leaks between tasks in a pool | `AsyncLocalStorage` |
| `ThreadPoolExecutor` | `submit` → `Future`; `map` keeps order; `as_completed` finish order; check every future | `Promise.all`, `Promise.race` |
| Errors and shutdown | `threading.excepthook`; no `kill()`; stop with an `Event` and `wait(timeout)` | `worker.terminate()` exists in Node |
| Free-threaded build | `python3.15t`, `sys._is_gil_enabled()`; real thread parallelism, hidden races exposed | `worker_threads` parallelism |

**Further reading**

- [`threading`](https://docs.python.org/3/library/threading.html),
  [`queue`](https://docs.python.org/3/library/queue.html),
  [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html)
- [`sys.setswitchinterval`](https://docs.python.org/3/library/sys.html#sys.setswitchinterval),
  [FAQ: what kinds of global value mutation are thread-safe?](https://docs.python.org/3/faq/library.html#what-kinds-of-global-value-mutation-are-thread-safe)
- [Python support for free threading](https://docs.python.org/3/howto/free-threading-python.html),
  [C API extension support for free threading](https://docs.python.org/3/howto/free-threading-extensions.html)
- [PEP 703](https://peps.python.org/pep-0703/) (making the GIL optional),
  [PEP 779](https://peps.python.org/pep-0779/) (supported status for free-threaded Python)

## Unit 5.2 — Multiprocessing and Subinterpreters

**Objectives**

- Run CPU-bound Python in parallel with processes, and know what each one costs
  to start.
- Pick a start method (`fork`, `spawn`, `forkserver`) and write code that works
  with all three, including from a notebook.
- Use `Process`, `Pool` and `ProcessPoolExecutor`, and tune `chunksize`,
  `initializer` and `max_tasks_per_child`.
- Know what can be pickled, what pickling costs, and when parallel code is slower
  than serial code.
- Move data between processes with queues, pipes, shared values, managers and
  shared memory (with NumPy).
- Use subinterpreters (`concurrent.interpreters`, `InterpreterPoolExecutor`):
  parallelism inside one process.
- Choose a concurrency model with a decision matrix, Amdahl's law and a
  benchmark.

**JS bridge**

| Node | Python | The difference that bites |
| --- | --- | --- |
| `child_process.fork(file)` | `multiprocessing.Process(target=func)` | Python starts the child at a *function*, but that function must be importable by the child |
| `cluster` (fork workers that share a port) | `ProcessPoolExecutor`, `multiprocessing.Pool` | pools hand out tasks; they do not share sockets |
| `process.send` / `'message'` (JSON or structured clone over IPC) | `Queue`, `Pipe` (pickle over a pipe) | pickle can rebuild arbitrary classes, and cannot send lambdas, locks or open files |
| `worker_threads` (isolate per thread, own heap) | subinterpreters: own GIL, own modules, one process | the closest match to a Worker |
| `SharedArrayBuffer` + typed arrays | `multiprocessing.shared_memory` + NumPy arrays | you must `close()` and `unlink()` the block yourself |
| `os.availableParallelism()` | `os.process_cpu_count()` (*new in 3.13*) | both respect CPU affinity |

**How this unit runs.** A process started with `spawn` or `forkserver` is a
fresh interpreter: it can only run a function it can **import**. Functions
typed into a notebook cell live in the kernel's `__main__`, which a child cannot
import (5.2.2 shows the failure). So each section first saves its worker
functions to a small module under `notebooks/_scratch/phase5/` with
`%%writefile`, then imports it with `fresh_import()`. Demos that need a clean
parent process (start-method costs, the main guard, Ctrl+C handling) run as
scripts in a child interpreter with `run_child()`.

Unit setup, part 1: imports, `DEMO_SCALE`, the scratch folder (also put on
`sys.path` so worker modules saved there are importable here and in every child
process), feature flags for NumPy and subinterpreters, and the error and
interpreter helpers.

In [83]:
import concurrent.futures
import functools
import importlib
import importlib.util
import itertools
import math
import multiprocessing
import operator
import os
import pickle
import pickletools
import platform
import random
import re
import signal
import subprocess
import sys
import sysconfig
import textwrap
import threading
import time
from collections import Counter
from collections.abc import Callable, Iterable
from multiprocessing import shared_memory
from pathlib import Path
from types import ModuleType

DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))  # the one knob for workload sizes
REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
PHASE5_DIR = scratch_dir / "phase5"
PHASE5_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT / "notebooks")  # %%writefile cells use paths relative to notebooks/
if str(PHASE5_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE5_DIR))  # child processes receive a copy of sys.path
CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0", "PYTHONUNBUFFERED": "1"}
HAS_NUMPY = importlib.util.find_spec("numpy") is not None
HAS_INTERPRETERS = importlib.util.find_spec("concurrent.interpreters") is not None


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" (memory addresses hidden), or "no error"."""
    try:
        thunk()
    except Exception as error:
        return re.sub(r" at 0x[0-9a-f]+", "", f"{type(error).__name__}: {error}")
    return "no error"


def interpreter_label() -> str:
    """Implementation, version and build flavour: print it next to every benchmark."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"

Unit setup, part 2: `Timer` (Unit 1.14), `run_child()` (Unit 5.1, now also
hiding process ids) and `fresh_import(module_name)`, which re-imports a module
from disk after a `%%writefile` cell has changed it.

In [84]:
class Timer:
    """Timing context manager (shared since Unit 1.14): reports on exit, keeps elapsed_seconds."""

    def __init__(self, label: str, report: Callable[[str], None] | None = print) -> None:
        self.label = label
        self.report = report
        self.elapsed_seconds = 0.0

    def __enter__(self) -> Timer:
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc_info: object) -> None:
        self.elapsed_seconds = time.perf_counter() - self._start
        if self.report is not None:
            self.report(f"{self.label} took {self.elapsed_seconds * 1000:.2f} ms")


def run_child(
    source: str, python: str = sys.executable, extra_env: dict[str, str] | None = None
) -> str:
    """Run dedented source as a script in a child interpreter; return its stdout + stderr."""
    script = PHASE5_DIR / "child_script.py"
    script.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    completed = subprocess.run(
        [python, str(script)], capture_output=True, text=True, timeout=120,
        env={**CHILD_ENV, **(extra_env or {})}, cwd=PHASE5_DIR,
    )  # fmt: skip
    output = (completed.stdout + completed.stderr).rstrip()
    for path, label in ((str(PHASE5_DIR), "<scratch>"), (str(Path(python).parent), "<python>")):
        output = output.replace(path, label)
    output = re.sub(r"pid=\d+", "pid=<pid>", output)
    return re.sub(r"/[^\s\"']*/lib/python3\.\d+t?/", "<stdlib>/", output)


def fresh_import(module_name: str) -> ModuleType:
    """Import a module saved by a %%writefile cell, always re-reading the file from disk."""
    for cached in (PHASE5_DIR / "__pycache__").glob(f"{module_name}.*.pyc"):
        cached.unlink()  # a stale .pyc could survive a quick edit
    importlib.invalidate_caches()
    sys.modules.pop(module_name, None)
    return importlib.import_module(module_name)


print(interpreter_label())
print(f"{DEMO_SCALE=}  {os.process_cpu_count()=}  {multiprocessing.get_start_method()=}")
print(
    f"numpy: {'available' if HAS_NUMPY else 'missing'}, "
    f"subinterpreters: {'available' if HAS_INTERPRETERS else 'missing'}"
)

CPython 3.15.1, default (GIL) build, x86_64
DEMO_SCALE=1  os.process_cpu_count()=16  multiprocessing.get_start_method()='forkserver'
numpy: available, subinterpreters: available


### 5.2.1 Processes

A process has its **own address space** and runs its **own interpreter with its
own GIL**. Nothing is shared by accident: no races on ordinary objects, and
CPU-bound Python code really runs in parallel. The price: starting a process
takes milliseconds, and every argument and result must be serialized (pickled)
to cross the boundary.

The worker module for 5.2.1–5.2.4. `%%writefile` saves the cell's text to a
file instead of running it; the module-level `ORDERS` list will show that each
process has its own copy of module state.

In [85]:
%%writefile _scratch/phase5/p5_basics.py
"""Worker functions for Units 5.2.1-5.2.4. Child processes import this module by name."""

import os
import sys
import time

ORDERS: list[str] = []  # module-level state: every process has its own copy
PRICE_TABLE: dict[str, float] = {}


def busy_sum(count: int) -> int:
    """Pure-Python CPU work (the same loop as Unit 5.1)."""
    return sum(number % 1000 for number in range(count))


def add_order(order_id: str) -> tuple[int, int]:
    """Append to this process's ORDERS; return (pid, number of orders it holds)."""
    ORDERS.append(order_id)
    return os.getpid(), len(ORDERS)


def square(number: int) -> int:
    """A tiny task."""
    return number * number


def power(base: int, exponent: int) -> int:
    """Two arguments, for starmap."""
    return base**exponent


def slow_square(number: int) -> int:
    """Square after a delay that makes later numbers finish first."""
    time.sleep(0.05 * (3 - number % 3))
    return number * number


def nap_then_exit(seconds: float, exit_code: int) -> None:
    """Sleep, then end the process with exit_code."""
    time.sleep(seconds)
    sys.exit(exit_code)


def load_price_table(gst_percent: float) -> None:
    """Pool initializer: runs once in each worker process."""
    PRICE_TABLE.update({"TEA-500": 240.0, "MUG-02": 199.0, "gst": gst_percent})
    PRICE_TABLE["loaded_by"] = os.getpid()


def price_with_gst(sku: str) -> tuple[str, float, bool]:
    """Use the table the initializer loaded in this worker."""
    total = round(PRICE_TABLE[sku] * (1 + PRICE_TABLE["gst"] / 100), 2)
    return sku, total, PRICE_TABLE["loaded_by"] == os.getpid()


def pid_after_nap(_: object) -> int:
    """Return the worker's pid after a short nap."""
    time.sleep(0.01)
    return os.getpid()


def silence_stderr() -> None:
    """Initializer that sends this worker's stderr to /dev/null (keeps one demo short)."""
    os.dup2(os.open(os.devnull, os.O_WRONLY), 2)

Overwriting _scratch/phase5/p5_basics.py


Import it. `fresh_import` re-reads the file, so editing and re-running the cell
above always takes effect.

In [86]:
p5_basics = fresh_import("p5_basics")
print(f"{p5_basics.__name__} loaded; {p5_basics.square(12)=}")

p5_basics loaded; p5_basics.square(12)=144


**Separate memory.** One worker process receives three orders; each call
appends to *its* `ORDERS`. The parent's `p5_basics.ORDERS` never changes, and
the worker's process id differs from the kernel's.

In [87]:
with concurrent.futures.ProcessPoolExecutor(max_workers=1) as one_worker:
    replies = list(one_worker.map(p5_basics.add_order, ["A-1", "A-2", "A-3"]))
worker_pids = {pid for pid, _ in replies}
print(f"worker saw order counts {[count for _, count in replies]}")
print(f"worker pid differs from the kernel's: {worker_pids != {os.getpid()}}")
print(f"the kernel's p5_basics.ORDERS: {p5_basics.ORDERS}")

worker saw order counts [1, 2, 3]
worker pid differs from the kernel's: True
the kernel's p5_basics.ORDERS: []


**Separate GILs.** The 5.1.3 benchmark again: the same total `busy_sum` work,
now split across 1 and 4 *processes*. Each process has its own GIL, so on a
machine with at least 4 cores this should approach 4× (minus start-up and
pickling).

In [88]:
total_numbers = 16_000_000 * DEMO_SCALE
with Timer("1 process (this kernel)") as serial_run:
    p5_basics.busy_sum(total_numbers)
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as pool:
    with Timer("4 worker processes, including pool start-up") as parallel_run:
        list(pool.map(p5_basics.busy_sum, [total_numbers // 4] * 4))
print(
    f"[{interpreter_label()}] speedup "
    f"{serial_run.elapsed_seconds / parallel_run.elapsed_seconds:.2f}x"
)

1 process (this kernel) took 342.78 ms


4 worker processes, including pool start-up took 129.63 ms
[CPython 3.15.1, default (GIL) build, x86_64] speedup 2.64x


**Creation cost.** How long does it take to start a process and wait for it to
finish, for each start method (explained in 5.2.2), compared with a thread? The
measurement runs as a script in a fresh child interpreter, because `fork` from
this multi-threaded kernel would be unsafe (5.2.2 shows why).

In [89]:
CREATION_COST = """
    import multiprocessing, threading, time

    def do_nothing():
        pass

    def average_ms(start_one, repeats=5):
        start = time.perf_counter()
        for _ in range(repeats):
            start_one()
        return (time.perf_counter() - start) / repeats * 1000

    def thread_once():
        worker = threading.Thread(target=do_nothing)
        worker.start()
        worker.join()

    if __name__ == "__main__":
        print(f"{'thread':<11} {average_ms(thread_once):8.3f} ms")
        for method in multiprocessing.get_all_start_methods():
            context = multiprocessing.get_context(method)

            def process_once():
                child = context.Process(target=do_nothing)
                child.start()
                child.join()

            process_once()  # warm-up (starts the fork server once)
            print(f"{method:<11} {average_ms(process_once):8.3f} ms")
"""
print(f"[{interpreter_label()}] start + join, average of 5 (varies by machine):")
print(run_child(CREATION_COST))

[CPython 3.15.1, default (GIL) build, x86_64] start + join, average of 5 (varies by machine):


thread         0.105 ms
forkserver     1.958 ms
fork           1.178 ms
spawn         24.361 ms


A thread costs tens of microseconds. `fork` copies the parent (cheap thanks to
copy-on-write pages) and costs about a millisecond; `forkserver` forks from a
small, already-running server process; `spawn` starts a whole new interpreter
and re-imports modules, the slowest. That is why you **reuse** processes in a
pool instead of starting one per task.

### 5.2.2 Start methods

| Method | How the child starts | Speed | Safe with threads in the parent? | Default on |
| --- | --- | --- | --- | --- |
| `fork` | `os.fork()`: a copy of the parent, mid-program, with all its memory | fastest | **no**: locks held by other threads are copied in their locked state | nothing since 3.14 (was Linux/POSIX until 3.13) |
| `spawn` | a fresh interpreter that imports the parent's `__main__` module, then unpickles the target | slowest | yes | Windows, macOS (since 3.8) |
| `forkserver` | a single-threaded server process, started once, forks a child for each request | fast after the first | yes | **POSIX except macOS, since 3.14** |

Query and choose with `get_start_method()`, `get_all_start_methods()` and
`get_context(method)`, which returns an object with the whole `multiprocessing`
API bound to that method. Prefer a context over `set_start_method()`, which may
be called only once per program.

In [90]:
print(f"{multiprocessing.get_start_method()=}")
print(f"{multiprocessing.get_all_start_methods()=}")
spawn_context = multiprocessing.get_context("spawn")
with concurrent.futures.ProcessPoolExecutor(max_workers=1, mp_context=spawn_context) as spawn_pool:
    print(f"spawn pool: {spawn_pool.submit(p5_basics.square, 9).result()=}")
print(f"{spawn_context.Process.__name__=}")

multiprocessing.get_start_method()='forkserver'
multiprocessing.get_all_start_methods()=['forkserver', 'fork', 'spawn']


spawn pool: spawn_pool.submit(p5_basics.square, 9).result()=81
spawn_context.Process.__name__='SpawnProcess'


**Why `fork` and threads do not mix.** `fork()` copies only the calling thread.
If another thread held a lock at that moment, the child gets the lock in its
locked state and nobody left to release it. Python warns
(`DeprecationWarning: This process ... is multi-threaded, use of fork() may lead
to deadlocks`). The script below makes it happen: a thread holds a lock, the
main thread forks, and the child tries the lock with a timeout.

In [91]:
FORK_WITH_THREADS = """
    import multiprocessing, threading, time, warnings

    warnings.simplefilter("always")  # show the DeprecationWarning
    inventory_lock = threading.Lock()

    def hold_lock():
        with inventory_lock:
            time.sleep(1)

    def child_wants_lock():
        got_it = inventory_lock.acquire(timeout=0.5)
        print(f"child: acquired the lock copied from the parent: {got_it}")

    if __name__ == "__main__":
        threading.Thread(target=hold_lock).start()
        time.sleep(0.05)  # the other thread now holds the lock
        child = multiprocessing.get_context("fork").Process(target=child_wants_lock)
        child.start()
        child.join()
"""
if "fork" in multiprocessing.get_all_start_methods():
    print(run_child(FORK_WITH_THREADS))
else:
    print("SKIP: fork is not available on this platform")

child: acquired the lock copied from the parent: False
<stdlib>/multiprocessing/popen_fork.py:76: DeprecationWarning: This process (pid=<pid>) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


The child waits half a second and gives up: the lock will never be released in
its copy of the world. Real programs deadlock this way inside `logging`,
`malloc` or a database driver. A notebook kernel always runs extra threads,
which is one reason `forkserver` became the default.

**The main guard.** With `spawn` and `forkserver`, the child imports the
parent's main module (as `__mp_main__`) to find the target function. Without an
`if __name__ == "__main__":` guard, that import runs the process-starting code
again inside the child, and `multiprocessing` stops it with a `RuntimeError`.
The output is trimmed to the last lines.

In [92]:
NO_GUARD = """
    import multiprocessing

    def greet():
        print("hello from the child")

    context = multiprocessing.get_context("spawn")
    child = context.Process(target=greet)
    child.start()
    child.join()
    print(f"parent: child exit code {child.exitcode}")
"""
WITH_GUARD = """
    import multiprocessing

    def greet():
        print("hello from the child")

    if __name__ == "__main__":
        context = multiprocessing.get_context("spawn")
        child = context.Process(target=greet)
        child.start()
        child.join()
        print(f"parent: child exit code {child.exitcode}")
"""
print("--- without the guard (key lines of the output)")
key_phrases = ("RuntimeError", "bootstrapping phase", "if __name__", "parent:")
for line in run_child(NO_GUARD).splitlines():
    if any(phrase in line for phrase in key_phrases):
        print(line.strip())
print("--- with the guard")
print(run_child(WITH_GUARD))

--- without the guard (key lines of the output)
parent: child exit code 1
raise RuntimeError('''
RuntimeError:
current process has finished its bootstrapping phase.
if __name__ == '__main__':
--- with the guard
hello from the child
parent: child exit code 0


#### Notebook caveat: worker functions must live in importable modules

Functions are pickled **by reference**: the pickle stores the module name and
the function name, never the code. A function typed into a notebook cell
belongs to the kernel's `__main__` module. A child interpreter's `__main__` is a
different module, so the lookup fails. First the mechanism, by hand: pickle a
notebook function and try to load it in a fresh interpreter.

In [93]:
def tax_for(amount: float) -> float:
    """Defined in a notebook cell, so it belongs to the kernel's __main__ module."""
    return round(amount * 0.18, 2)


payload = pickle.dumps(tax_for)
print(
    "the pickle names the function: "
    f"{b'__main__' in payload and b'tax_for' in payload}, {len(payload)} bytes"
)
loader = subprocess.run(
    [sys.executable, "-c", "import pickle, sys; pickle.loads(sys.stdin.buffer.read())"],
    input=payload, capture_output=True, env=CHILD_ENV,
)  # fmt: skip
print(f"loading it in a fresh interpreter: {loader.stderr.decode().strip().splitlines()[-1]}")

the pickle names the function: True, 35 bytes
loading it in a fresh interpreter: AttributeError: module '__main__' has no attribute 'tax_for'


A process pool hits exactly that error inside the worker, which then dies, and
the pool reports `BrokenProcessPool`. (The `silence_stderr` initializer hides
the worker's own traceback to keep the output short.) The same call with the
function from the importable `p5_basics` module works.

In [94]:
with concurrent.futures.ProcessPoolExecutor(
    max_workers=1, initializer=p5_basics.silence_stderr
) as pool:
    print(
        "notebook function: "
        f"{raised_message(lambda: pool.submit(tax_for, 100.0).result(timeout=60))}"
    )
with concurrent.futures.ProcessPoolExecutor(max_workers=1) as pool:
    print(f"module function:   {pool.submit(p5_basics.square, 100).result()=}")

notebook function: BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.


module function:   pool.submit(p5_basics.square, 100).result()=10000


| Option | Works because | Trade-off |
| --- | --- | --- |
| Save workers to a module (`%%writefile`, or a real `.py` file in your project) and import it | the child imports the module by name | the approach used in this phase |
| Run the code as a script (`python script.py`) | the child imports the script as `__mp_main__`; it needs the main guard | not interactive |
| `get_context("fork")` | the child is a copy of the kernel, function included | unsafe with the kernel's threads (above); not on Windows |
| Third-party pickling (`cloudpickle`, used by Dask, Ray, joblib) | pickles functions by value | an extra dependency |

### 5.2.3 Process APIs

#### `Process`

`multiprocessing.Process` mirrors `threading.Thread` (`target`, `args`, `name`,
`daemon`, `start`, `join`, `is_alive`) and adds process controls:

| Attribute / method | Meaning |
| --- | --- |
| `pid` | the OS process id (after `start()`) |
| `exitcode` | `None` while running, `0` on success, `n` after `sys.exit(n)`, `-signal` if killed by a signal |
| `terminate()` / `kill()` | send `SIGTERM` / `SIGKILL` (no `finally` blocks run in the child) |
| `interrupt()` | send `SIGINT`, so the child gets `KeyboardInterrupt` (*new in 3.14*) |
| `close()` | release the `Process` object's resources once it has finished |

Three children: one exits normally, one calls `sys.exit(3)`, one is terminated
while it sleeps.

In [95]:
children = {
    "finishes": multiprocessing.Process(
        target=p5_basics.nap_then_exit, args=(0.05, 0), name="finishes"
    ),
    "exits with 3": multiprocessing.Process(target=p5_basics.nap_then_exit, args=(0.05, 3)),
    "terminated": multiprocessing.Process(target=p5_basics.nap_then_exit, args=(10, 0)),
}
for child in children.values():
    child.start()
print(
    f"right after start: exitcode={children['finishes'].exitcode}, "
    f"has pid={children['finishes'].pid is not None}"
)
children["terminated"].terminate()
for label, child in children.items():
    child.join()
    print(f"{label:<13} exitcode={child.exitcode:>4}  alive={child.is_alive()}")
    child.close()
print(f"-signal.SIGTERM = {-signal.SIGTERM}")

right after start: exitcode=None, has pid=True
finishes      exitcode=   0  alive=False
exits with 3  exitcode=   3  alive=False
terminated    exitcode= -15  alive=False
-signal.SIGTERM = -15


#### `Pool`

`multiprocessing.Pool(processes)` is the original pool API:

| Method | Returns | Order |
| --- | --- | --- |
| `map(func, iterable, chunksize)` | a list | input order |
| `imap(func, iterable, chunksize)` | a lazy iterator | input order |
| `imap_unordered(func, iterable, chunksize)` | a lazy iterator | **completion** order |
| `starmap(func, iterable_of_tuples)` | a list of `func(*args)` | input order |
| `apply_async`, `map_async` | `AsyncResult` objects | — |

> **Warning:** `Pool.__exit__` calls `terminate()`, not `close()` + `join()`.
> Collect every result *inside* the `with` block.

In [96]:
with multiprocessing.Pool(processes=3) as pool:
    print(f"map:            {pool.map(p5_basics.slow_square, range(6))}")
    print(f"imap_unordered: {list(pool.imap_unordered(p5_basics.slow_square, range(6)))}")
    print(f"starmap:        {pool.starmap(p5_basics.power, [(2, 10), (3, 4), (10, 3)])}")

map:            [0, 1, 4, 9, 16, 25]


imap_unordered: [4, 1, 0, 25, 16, 9]
starmap:        [1024, 81, 1000]


`slow_square` sleeps longer for numbers divisible by 3, so `imap_unordered`
yields them last (the exact order can vary slightly between runs).

**`chunksize`.** Each task is pickled and sent through a pipe; `chunksize=n`
sends `n` items per message, so the per-message overhead is paid once per chunk.
For many tiny tasks that is the difference between slow and fast. (Lab 5.2
draws the full curve.)

In [97]:
tiny_items = range(20_000 * DEMO_SCALE)
with multiprocessing.Pool(processes=4) as pool:
    pool.map(p5_basics.square, range(8))  # start the workers before timing
    for chunk_size in (1, 2_000):
        with Timer(f"Pool.map of {len(tiny_items):,} tiny tasks, chunksize={chunk_size:>5}"):
            pool.map(p5_basics.square, tiny_items, chunksize=chunk_size)

Pool.map of 20,000 tiny tasks, chunksize=    1 took 1380.81 ms
Pool.map of 20,000 tiny tasks, chunksize= 2000 took 7.05 ms


#### `ProcessPoolExecutor`

The `concurrent.futures` version: the same `submit` / `map` / `Future` API as
`ThreadPoolExecutor` (5.1.8), plus process-specific options:

| Option | Effect |
| --- | --- |
| `max_workers` | defaults to `os.process_cpu_count()` |
| `mp_context` | a start-method context (`get_context("spawn")`) |
| `initializer`, `initargs` | a function run once in each new worker (load a model, open a connection, set up logging) |
| `max_tasks_per_child` | replace a worker after it has run `n` tasks (contains memory leaks in long-running pools; *new in 3.11*) |
| `map(..., chunksize=n)` | the same batching as `Pool.map` |
| `terminate_workers()`, `kill_workers()` | stop all workers now (*new in 3.14*) |

**`initializer`.** Each worker loads the price table once; every task then uses
the worker's own copy. The third field shows the table was loaded in the same
process that ran the task.

In [98]:
with concurrent.futures.ProcessPoolExecutor(
    max_workers=2, initializer=p5_basics.load_price_table, initargs=(18.0,)
) as pool:
    for sku, price_with_tax, loaded_here in pool.map(
        p5_basics.price_with_gst, ["TEA-500", "MUG-02", "TEA-500"]
    ):
        print(f"{sku:<8} ₹{price_with_tax:>7.2f}  table loaded in this worker: {loaded_here}")
print(f"the kernel's PRICE_TABLE stays empty: {p5_basics.PRICE_TABLE}")

TEA-500  ₹ 283.20  table loaded in this worker: True
MUG-02   ₹ 234.82  table loaded in this worker: True
TEA-500  ₹ 283.20  table loaded in this worker: True
the kernel's PRICE_TABLE stays empty: {}


**`max_tasks_per_child`.** With one worker and a limit of 2 tasks, six tasks run
in three different processes.

In [99]:
with concurrent.futures.ProcessPoolExecutor(max_workers=1, max_tasks_per_child=2) as pool:
    task_pids = list(pool.map(p5_basics.pid_after_nap, range(6)))
print(f"distinct worker processes for 6 tasks: {len(set(task_pids))}")
print(
    "pattern (same letter = same process): "
    f"{''.join(chr(65 + sorted(set(task_pids)).index(pid)) for pid in task_pids)}"
)

distinct worker processes for 6 tasks: 3
pattern (same letter = same process): AABBCC


### 5.2.4 Data transfer

Every argument, return value and exception that crosses a process boundary is
**pickled** in the sender and unpickled in the receiver (Unit 4.5 covered
`pickle`). Two consequences: some objects cannot cross at all, and big objects
are expensive to send.

**Pickling constraints.**

| Can be pickled | Cannot be pickled |
| --- | --- |
| `None`, numbers, strings, bytes, tuples, lists, dicts, sets of picklable items | lambdas and functions defined inside functions |
| module-level functions and classes (by reference: module + name) | open files, sockets, database connections |
| instances of module-level classes (dataclasses included) | locks, threads, generators, coroutines |
| `functools.partial` of a module-level function | functions in the notebook's `__main__` (they pickle, but a child cannot load them: 5.2.2) |

In [100]:
def make_discount(percent: int) -> Callable[[float], float]:
    """Return a nested function (not picklable)."""

    def apply(amount: float) -> float:
        return amount * (100 - percent) / 100

    return apply


candidates = {
    "tuple of numbers and text": ("TEA-500", 2, 240.0),
    "module-level function": p5_basics.square,
    "functools.partial(p5_basics.power, 2)": functools.partial(p5_basics.power, 2),
    "lambda": lambda amount: amount * 2,
    "nested function": make_discount(10),
    "threading.Lock()": threading.Lock(),
    "generator": (number for number in range(3)),
    "open file": open(os.devnull),
}


def pickle_outcome(candidate: object) -> str:
    """ "OK" if candidate pickles, else the error."""
    return raised_message(lambda: pickle.dumps(candidate)).replace("no error", "OK")


for label, candidate in candidates.items():
    print(f"{label:<40} {pickle_outcome(candidate)}")
candidates["open file"].close()

tuple of numbers and text                OK
module-level function                    OK
functools.partial(p5_basics.power, 2)    OK
lambda                                   PicklingError: Can't pickle <function <lambda>>: it's not found as __main__.<lambda>
nested function                          PicklingError: Can't pickle local object <function make_discount.<locals>.apply>
threading.Lock()                         TypeError: cannot pickle '_thread.lock' object
generator                                TypeError: cannot pickle 'generator' object
open file                                TypeError: cannot pickle 'TextIOWrapper' instances


#### Under the hood: what a pickled function contains

`pickletools.dis` disassembles a pickle. A function is just two strings, its
module and its name, plus `STACK_GLOBAL` ("import this and look it up"). The
code itself is never sent: the receiver must be able to import it.

In [101]:
os.environ["PYTHON_COLORS"] = "0"  # the kernel sets FORCE_COLOR=1; this wins over it
try:
    pickletools.dis(pickle.dumps(p5_basics.square))
finally:
    del os.environ["PYTHON_COLORS"]

    0: \x80 PROTO      5
    2: \x95 FRAME      24
   11: \x8c SHORT_BINUNICODE 'p5_basics'
   22: \x94 MEMOIZE    (as 0)
   23: \x8c SHORT_BINUNICODE 'square'
   31: \x94 MEMOIZE    (as 1)
   32: \x93 STACK_GLOBAL
   33: \x94 MEMOIZE    (as 2)
   34: .    STOP
highest protocol among opcodes = 4


**Pickling cost.** A list of a million integers: `sys.getsizeof` counts only
the list's pointer array, the pickle holds every value, and sending it to a
worker means pickle → pipe → unpickle (and the same back for the result).

In [102]:
order_ids = list(range(1_000_000 * DEMO_SCALE))
with Timer("pickle.dumps(1M ints)"):
    order_payload = pickle.dumps(order_ids)
with Timer("pickle.loads(1M ints)"):
    pickle.loads(order_payload)
print(
    f"sys.getsizeof(list): {sys.getsizeof(order_ids) / 1e6:.1f} "
    f"MB, pickle: {len(order_payload) / 1e6:.1f} MB"
)

with concurrent.futures.ProcessPoolExecutor(max_workers=1) as pool:
    pool.submit(len, []).result()  # start the worker before timing
    with Timer("len() of the list in a worker (send 1M ints)"):
        pool.submit(len, order_ids).result()
with Timer("len() of the list in the kernel"):
    len(order_ids)

pickle.dumps(1M ints) took 7.53 ms


pickle.loads(1M ints) took 20.56 ms
sys.getsizeof(list): 8.0 MB, pickle: 4.9 MB


len() of the list in a worker (send 1M ints) took 27.45 ms
len() of the list in the kernel took 0.00 ms


**When parallel is slower.** Squaring a number takes about 30 ns; sending it to
a worker and getting the answer back takes tens of microseconds. With tiny
tasks the pool spends its time on overhead, and even large chunks only pay off
when the work per item outweighs pickling it.

In [103]:
numbers = list(range(200_000 * DEMO_SCALE))
with Timer("sequential, in the kernel"):
    [p5_basics.square(number) for number in numbers]
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as pool:
    list(pool.map(p5_basics.square, range(8)))  # start the workers
    with Timer("4 processes, chunksize=1 (only the first 10% of the items!)"):
        list(pool.map(p5_basics.square, numbers[: len(numbers) // 10]))
    with Timer("4 processes, chunksize=10_000"):
        list(pool.map(p5_basics.square, numbers, chunksize=10_000))

sequential, in the kernel took 10.85 ms


4 processes, chunksize=1 (only the first 10% of the items!) took 802.10 ms
4 processes, chunksize=10_000 took 30.55 ms


Both parallel versions lose. Rules of thumb:

- Make each task **coarse**: seconds of work, not microseconds (batch items, or
  use `chunksize`).
- **Send little, compute a lot**: pass a file name or a row range, not the data;
  return a summary, not every intermediate value.
- For big arrays, share memory instead of pickling (5.2.5).
- Measure. A parallel version that is not faster is pure complexity.

### 5.2.5 IPC

| Tool | What it is | Use it for |
| --- | --- | --- |
| `multiprocessing.Queue` | many-to-many FIFO; items are pickled and sent by a background feeder thread | work and result streams |
| `Pipe(duplex=True)` | two `Connection` ends; `send` / `recv` pickled objects | a conversation between exactly two processes |
| `Value(typecode, initial)`, `Array(typecode, size)` | a C value / array in shared memory, with an optional lock | counters, small numeric buffers |
| `Manager()` | a server process holding real Python objects; others get **proxies** | shared dicts and lists (slow: every access is a round trip) |
| `shared_memory.SharedMemory` | a named block of raw shared memory | large arrays without copying (with NumPy) |

The worker module for this section:

In [104]:
%%writefile _scratch/phase5/p5_ipc.py
"""Worker functions for Unit 5.2.5 (inter-process communication)."""

import os
from multiprocessing import shared_memory


def produce_orders(order_queue, count: int) -> None:
    """Put count orders on a multiprocessing.Queue, then a None sentinel."""
    for number in range(1, count + 1):
        order_queue.put({"order_id": number, "producer_pid": os.getpid()})
    order_queue.put(None)


def price_service(connection) -> None:
    """Answer price requests over one end of a Pipe until told to stop."""
    prices = {"TEA-500": 240.0, "MUG-02": 199.0}
    while (request := connection.recv()) != "stop":
        connection.send({"sku": request, "price": prices.get(request)})
    connection.close()


def add_visits(counter, times: int, use_lock: bool) -> None:
    """Increment a shared multiprocessing.Value, with or without its lock."""
    for _ in range(times):
        if use_lock:
            with counter.get_lock():
                counter.value += 1
        else:
            counter.value += 1  # read and write are separate steps: updates get lost


def fill_squares(shared_array, start: int, stop: int) -> None:
    """Write index squared into a slice of a shared multiprocessing.Array."""
    for index in range(start, stop):
        shared_array[index] = index * index


def record_stock(stock, log, sku: str, quantity: int) -> None:
    """Write through Manager proxies (a dict and a list)."""
    stock[sku] = quantity
    log.append(f"set {sku}")


def square_in_place(block_name: str, length: int) -> None:
    """Attach to a shared float64 block by name and square every value in place."""
    import numpy as np

    block = shared_memory.SharedMemory(name=block_name, track=False)
    try:
        values = np.ndarray((length,), dtype=np.float64, buffer=block.buf)
        values **= 2
        del values  # drop the view before closing the block
    finally:
        block.close()

Overwriting _scratch/phase5/p5_ipc.py


Import it:

In [105]:
p5_ipc = fresh_import("p5_ipc")
print(f"{p5_ipc.__name__} loaded")

p5_ipc loaded


**`Queue`.** A producer process puts five orders and a `None` sentinel; the
parent reads until the sentinel (the same pattern as 5.1.6, across processes).

In [106]:
order_queue = multiprocessing.Queue()
producer = multiprocessing.Process(target=p5_ipc.produce_orders, args=(order_queue, 5))
producer.start()
received = []
while (order := order_queue.get(timeout=30)) is not None:
    received.append(order)
producer.join()
print(f"received order ids {[order['order_id'] for order in received]}")
print(f"all from the producer process: {len({order['producer_pid'] for order in received}) == 1}")

received order ids [1, 2, 3, 4, 5]
all from the producer process: True


**`Pipe`.** A two-way channel: the parent asks a price service in another
process and gets answers back. `poll(timeout)` checks whether a message is
waiting.

In [107]:
parent_end, child_end = multiprocessing.Pipe()
service = multiprocessing.Process(target=p5_ipc.price_service, args=(child_end,))
service.start()
for sku in ("TEA-500", "SPOON-1"):
    parent_end.send(sku)
    print(f"asked {sku}: {parent_end.recv()}")
print(f"anything else waiting? {parent_end.poll(timeout=0.05)}")
parent_end.send("stop")
service.join()
print(f"service exit code: {service.exitcode}")

asked TEA-500: {'sku': 'TEA-500', 'price': 240.0}
asked SPOON-1: {'sku': 'SPOON-1', 'price': None}
anything else waiting? False
service exit code: 0


**`Value` and `Array`.** Shared C storage. `Value` comes with a lock, but
`counter.value += 1` is still a read followed by a write: the 5.1.4 race, now
between processes. Wrap it in `with counter.get_lock():`. `Array` slices can be
filled by different processes without a lock when they do not overlap.

In [108]:
for use_lock in (False, True):
    visits = multiprocessing.Value("i", 0)
    workers = [
        multiprocessing.Process(target=p5_ipc.add_visits, args=(visits, 5_000, use_lock))
        for _ in range(4)
    ]
    for worker in workers:
        worker.start()
    for worker in workers:
        worker.join()
    print(f"use_lock={use_lock!s:<5}: expected 20,000 visits, got {visits.value:,}")

squares = multiprocessing.Array("q", 8)  # 8 signed 64-bit integers, zeroed
halves = [
    multiprocessing.Process(target=p5_ipc.fill_squares, args=(squares, start, start + 4))
    for start in (0, 4)
]
for half in halves:
    half.start()
for half in halves:
    half.join()
print(f"Array filled by two processes: {list(squares)}")

use_lock=False: expected 20,000 visits, got 12,085


use_lock=True : expected 20,000 visits, got 20,000


Array filled by two processes: [0, 1, 4, 9, 16, 25, 36, 49]


**`Manager`.** A manager runs a server process that owns ordinary Python
objects; other processes get proxies that forward every operation to it. That
makes it flexible (any picklable values, nested structures) and slow. The timing
compares 1 000 writes to a managed dict with 1 000 writes to a plain dict.

In [109]:
with multiprocessing.Manager() as manager:
    stock, change_log = manager.dict(), manager.list()
    writers = [
        multiprocessing.Process(target=p5_ipc.record_stock, args=(stock, change_log, sku, quantity))
        for sku, quantity in (("TEA-500", 40), ("MUG-02", 12))
    ]
    for writer in writers:
        writer.start()
    for writer in writers:
        writer.join()
    print(f"stock={dict(sorted(stock.items()))}, log={sorted(change_log)}")
    with Timer("1,000 writes to a Manager dict (a round trip each)"):
        for number in range(1_000):
            stock[f"sku-{number}"] = number
plain_dict: dict[str, int] = {}
with Timer("1,000 writes to a plain dict"):
    for number in range(1_000):
        plain_dict[f"sku-{number}"] = number

stock={'MUG-02': 12, 'TEA-500': 40}, log=['set MUG-02', 'set TEA-500']


1,000 writes to a Manager dict (a round trip each) took 14.06 ms
1,000 writes to a plain dict took 0.11 ms


#### `shared_memory` with NumPy

`SharedMemory(create=True, size=n)` allocates a named block that any process
can attach to by name. Wrapping its buffer in a NumPy array gives both processes
a view of the **same bytes**: nothing is pickled except the block's name. The
owner must `close()` its view and `unlink()` the block when done (otherwise the
memory stays allocated until reboot on some systems). The worker attaches with
`track=False` (*new in 3.13*) so only the owner's resource tracker manages the
block.

In [110]:
if not HAS_NUMPY:
    print("SKIP: numpy is not installed")
else:
    import numpy as np

    readings = np.arange(1.0, 9.0)  # 8 float64 values
    block = shared_memory.SharedMemory(create=True, size=readings.nbytes)
    try:
        shared_readings = np.ndarray(readings.shape, dtype=np.float64, buffer=block.buf)
        shared_readings[:] = readings
        squarer = multiprocessing.Process(
            target=p5_ipc.square_in_place, args=(block.name, len(readings))
        )
        squarer.start()
        squarer.join()
        print(f"after the child squared them: {shared_readings.tolist()}")
        print(
            f"sent to the child: the name {block.name!r:.12}... "
            f"({len(pickle.dumps(block.name))} bytes pickled)"
        )
        print(f"pickling the array instead would send {len(pickle.dumps(readings))} bytes")
        del shared_readings  # release the view before closing
    finally:
        block.close()
        block.unlink()

after the child squared them: [1.0, 4.0, 9.0, 16.0, 25.0, 36.0, 49.0, 64.0]
sent to the child: the name 'psm_fd893d8... (27 bytes pickled)
pickling the array instead would send 191 bytes


Eight floats is a toy; the same code with 100 million floats still sends only
the name. Lab 5.2 splits one shared array across several workers.

### 5.2.6 Subinterpreters

A **subinterpreter** is a second, isolated Python interpreter inside the *same
process*: its own modules, its own `sys`, its own `__main__`, and (since
[PEP 684](https://peps.python.org/pep-0684/), 3.12) **its own GIL**. So several
interpreters can run Python bytecode in parallel on different threads, with
process-like isolation and thread-like start-up cost.

| | |
| --- | --- |
| [PEP 734](https://peps.python.org/pep-0734/) | `concurrent.interpreters` in the standard library (*new in 3.14*) |
| `interpreters.create()` | a new interpreter; `close()` it when done |
| `interp.exec(source)` | run code in its `__main__`; errors come back as `ExecutionFailed` |
| `interp.call(func, *args)` | call a function there and return its result |
| `interp.call_in_thread(func, *args)` | the same in a new thread (returns the `Thread`) |
| `interp.prepare_main(name=value)` | bind shareable objects in its `__main__` |
| `interpreters.create_queue()` | a queue that works across interpreters |
| `interpreters.is_shareable(obj)` | `None`, `bool`, numbers, `str`, `bytes`, tuples of those, queues |
| `InterpreterPoolExecutor` | `concurrent.futures` pool of interpreters (*new in 3.14*) |

> **Note:** output printed *inside* a subinterpreter goes to the process's real
> stdout, which the notebook does not show. The demos send results back through
> return values and queues instead.

**Basics.** Create an interpreter, pass it a queue, let it report its id, call
a function in it, and show that changing `sys` there does not affect the main
interpreter.

In [111]:
if not HAS_INTERPRETERS:
    print("SKIP: concurrent.interpreters is not available (new in 3.14)")
else:
    from concurrent import interpreters

    worker_interpreter = interpreters.create()
    try:
        reports = interpreters.create_queue()
        worker_interpreter.prepare_main(reports=reports, greeting="namaste")
        worker_interpreter.exec(
            "from concurrent import interpreters as i; import sys; "
            "sys.course_marker = True; reports.put((greeting, i.get_current().id))"
        )
        print(f"from the subinterpreter: {reports.get(timeout=10)}")
        print(f"main interpreter id: {interpreters.get_main().id}")
        print(f"call(len, 'namaste') = {worker_interpreter.call(len, 'namaste')}")
        print(f"main interpreter sees sys.course_marker: {hasattr(sys, 'course_marker')}")
        print(f"interpreters alive: {len(interpreters.list_all())}")
        samples = (42, "text", (1, 2), [1, 2], {"a": 1})
        shareable = {repr(value): interpreters.is_shareable(value) for value in samples}
        print(f"is_shareable: {shareable}")
    finally:
        worker_interpreter.close()

from the subinterpreter: ('namaste', 1)
main interpreter id: 0
call(len, 'namaste') = 7
main interpreter sees sys.course_marker: False
interpreters alive: 2
is_shareable: {'42': True, "'text'": True, '(1, 2)': True, '[1, 2]': False, "{'a': 1}": False}


**A subinterpreter has its own `sys.path`.** It starts with the default search
path (the working directory and the standard library), not the kernel's
modified `sys.path`, so our scratch folder is missing. Callables and arguments
are passed as shareable objects or pickled, and a function pickled by reference
must be importable *inside the subinterpreter*. The fix is a tiny function that
uses only built-ins, run once in each new interpreter: here with `call`, in a
pool as the `initializer`.

In [112]:
def add_to_sys_path(folder: str) -> None:
    """Initializer for a fresh subinterpreter: make `folder` importable there."""
    import sys

    if folder not in sys.path:
        sys.path.insert(0, folder)


if not HAS_INTERPRETERS:
    print("SKIP: concurrent.interpreters is not available (new in 3.14)")
else:
    path_test = interpreters.create()
    try:
        try:
            path_test.call(p5_basics.pid_after_nap, None)
        except interpreters.NotShareableError as error:
            cause = str(error.__cause__).splitlines()[0]
            print(f"before: {type(error).__name__}: {error} ({cause})")
        path_test.call(add_to_sys_path, str(PHASE5_DIR))
        same_process = path_test.call(p5_basics.pid_after_nap, None) == os.getpid()
        print(f"after add_to_sys_path: it runs, in the kernel's own process: {same_process}")
    finally:
        path_test.close()

before: NotShareableError: func not shareable (ModuleNotFoundError: No module named 'p5_basics')
after add_to_sys_path: it runs, in the kernel's own process: True


**One GIL per interpreter.** The same four `busy_sum` jobs on a pool of four
threads (one shared GIL) and on a pool of four interpreters (four GILs, each
interpreter initialized with `add_to_sys_path`). Unlike processes, the
interpreter pool starts in milliseconds and stays in one process.

In [113]:
if not HAS_INTERPRETERS:
    print("SKIP: concurrent.interpreters is not available (new in 3.14)")
else:
    jobs = [4_000_000 * DEMO_SCALE] * 4
    print(f"[{interpreter_label()}] 4 x busy_sum({jobs[0]:,})")
    pools = {
        "ThreadPoolExecutor": lambda: concurrent.futures.ThreadPoolExecutor(max_workers=4),
        "InterpreterPoolExecutor": lambda: concurrent.futures.InterpreterPoolExecutor(
            max_workers=4, initializer=add_to_sys_path, initargs=(str(PHASE5_DIR),)
        ),
    }
    for pool_name, make_pool in pools.items():
        with make_pool() as pool:
            list(pool.map(p5_basics.busy_sum, [10] * 4))  # start the workers
            with Timer(f"{pool_name:<24}"):
                list(pool.map(p5_basics.busy_sum, jobs))
    with Timer("create + close one interpreter", report=None) as interpreter_cost:
        interpreters.create().close()
    print(f"create + close one interpreter: {interpreter_cost.elapsed_seconds * 1000:.1f} ms")

[CPython 3.15.1, default (GIL) build, x86_64] 4 x busy_sum(4,000,000)


ThreadPoolExecutor       took 359.68 ms
InterpreterPoolExecutor  took 108.90 ms
create + close one interpreter: 11.0 ms


**Compatibility.** Every C extension imported in a subinterpreter must support
multi-phase initialization and a per-interpreter GIL. Most of the standard
library does; many third-party extensions do not yet. NumPy, for example,
refuses to load:

In [114]:
if not HAS_INTERPRETERS or not HAS_NUMPY:
    print("SKIP: needs both concurrent.interpreters and numpy")
else:
    numpy_test = interpreters.create()
    try:
        numpy_test.exec("import numpy")
        print("numpy imported in a subinterpreter")
    except interpreters.ExecutionFailed as failure:
        first_sentence = next(line for line in failure.excinfo.msg.splitlines() if line.strip())
        print(f"ExecutionFailed wrapping {failure.excinfo.type.__name__}: {first_sentence}")
    finally:
        numpy_test.close()

ExecutionFailed wrapping ImportError: IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!


| | Threads | Subinterpreters | Processes |
| --- | --- | --- | --- |
| Parallel Python bytecode | no (default build) | yes | yes |
| Isolation | none | modules, globals, `sys` | everything (address space) |
| Start-up | ~10 µs | ~ms | ~ms to tens of ms |
| Passing data | shared objects | shareable values, queues, pickling | pickling, shared memory |
| C extensions | all | only those that support it | all |
| A crash (segfault) takes down | the process | the process | only that worker |

### 5.2.7 Choosing a model

**Decision matrix.**

| Workload | Threads | Processes | Subinterpreters | asyncio (5.3–5.4) | Free-threaded threads |
| --- | --- | --- | --- | --- | --- |
| Many network calls (HTTP, DB) | good up to hundreds | wasteful | wasteful | **best** for thousands | good |
| Blocking library with no async API | **best** | works | depends on C extensions | via `to_thread` | good |
| CPU-bound pure Python | useless (GIL) | **best default** | good if extensions allow | useless | **good** |
| CPU-bound in NumPy / hashlib / zlib | **good** (they release the GIL) | works, pays pickling | NumPy not supported yet | via `to_thread` | good |
| Large shared in-memory data, CPU-bound | GIL-bound | shared memory only | shareable values only | — | **best** |
| Isolation from crashes and leaks | none | **best** (`max_tasks_per_child`) | partial | none | none |

The worker module for the last two sections:

In [115]:
%%writefile _scratch/phase5/p5_patterns.py
"""Worker functions for Units 5.2.7-5.2.8."""

import time
from collections import Counter

from p5_basics import busy_sum  # noqa: F401  (re-exported for the benchmark)


def wait_for_io(seconds: float) -> float:
    """Stand-in for a network call: the worker just waits."""
    time.sleep(seconds)
    return seconds


def revenue_by_city(records: list[tuple[str, int]]) -> Counter[str]:
    """Map step: total revenue per city in one chunk of sales records."""
    totals: Counter[str] = Counter()
    for city, amount in records:
        totals[city] += amount
    return totals

Overwriting _scratch/phase5/p5_patterns.py


Import it:

In [116]:
p5_patterns = fresh_import("p5_patterns")
print(f"{p5_patterns.__name__} loaded")

p5_patterns loaded


**Justify it with a benchmark.** The same two jobs on every model: CPU-bound
(8 × `busy_sum`) and I/O-bound (8 × a 0.1 s wait), with 4 workers each. One
table, measured on this machine.

In [117]:
cpu_jobs, io_jobs = [3_000_000 * DEMO_SCALE] * 8, [0.1] * 8
executors: dict[str, Callable[[], concurrent.futures.Executor]] = {
    "threads": lambda: concurrent.futures.ThreadPoolExecutor(4),
    "processes": lambda: concurrent.futures.ProcessPoolExecutor(4),
}
if HAS_INTERPRETERS:
    executors["subinterpreters"] = lambda: concurrent.futures.InterpreterPoolExecutor(
        4, initializer=add_to_sys_path, initargs=(str(PHASE5_DIR),)
    )

measured: dict[str, dict[str, float]] = {"sequential": {}}
for job_name, worker, jobs in (
    ("cpu", p5_patterns.busy_sum, cpu_jobs),
    ("io", p5_patterns.wait_for_io, io_jobs),
):
    with Timer("", report=None) as serial_timer:
        [worker(job) for job in jobs]
    measured["sequential"][job_name] = serial_timer.elapsed_seconds
    for model, make_executor in executors.items():
        with make_executor() as pool:
            list(pool.map(worker, jobs[:4]))  # warm up the workers
            with Timer("", report=None) as model_timer:
                list(pool.map(worker, jobs))
        measured.setdefault(model, {})[job_name] = model_timer.elapsed_seconds
print(f"[{interpreter_label()}], 4 workers per pool")
print(f"{'model':<16}{'CPU-bound':>12}{'I/O-bound':>12}")
for model, times in measured.items():
    print(f"{model:<16}{times['cpu']:>11.3f}s{times['io']:>11.3f}s")

[CPython 3.15.1, default (GIL) build, x86_64], 4 workers per pool
model              CPU-bound   I/O-bound
sequential            0.595s      0.801s
threads               0.569s      0.202s
processes             0.165s      0.204s
subinterpreters       0.183s      0.203s


Threads win nothing on the CPU column and everything on the I/O column;
processes and subinterpreters win on CPU. For I/O, all pools get about 4×
(4 workers waiting at once), which is why the cheapest option, threads (or
asyncio), is the right one there.

#### Amdahl's law

If a fraction `p` of a program's run time can be parallelized and the rest
`1 - p` cannot, the best possible speedup on `N` workers is

```text
S(N) = 1 / ((1 - p) + p / N)        and as N grows without limit, S → 1 / (1 - p)
```

A program that is 90% parallel can never run more than 10× faster, however many
cores you add. The table and chart show how quickly the serial part dominates.

In [118]:
def amdahl(parallel_fraction: float, workers: float) -> float:
    """Amdahl's law: the speedup limit for a parallel fraction on `workers` workers."""
    return 1 / ((1 - parallel_fraction) + parallel_fraction / workers)


worker_counts = [1, 2, 4, 8, 16, 64, math.inf]
print("p \\ N  " + "".join(f"{('inf' if n == math.inf else n):>7}" for n in worker_counts))
for fraction in (0.5, 0.75, 0.9, 0.95, 0.99):
    print(f"{fraction:<7}" + "".join(f"{amdahl(fraction, n):>7.1f}" for n in worker_counts))
print()
for n in (1, 2, 4, 8, 16, 32):
    print(f"p=0.9, N={n:>2} {'#' * round(amdahl(0.9, n) * 4):<40} {amdahl(0.9, n):.1f}x")

p \ N        1      2      4      8     16     64    inf
0.5        1.0    1.3    1.6    1.8    1.9    2.0    2.0
0.75       1.0    1.6    2.3    2.9    3.4    3.8    4.0
0.9        1.0    1.8    3.1    4.7    6.4    8.8   10.0
0.95       1.0    1.9    3.5    5.9    9.1   15.4   20.0
0.99       1.0    2.0    3.9    7.5   13.9   39.3  100.0

p=0.9, N= 1 ####                                     1.0x
p=0.9, N= 2 #######                                  1.8x
p=0.9, N= 4 ############                             3.1x
p=0.9, N= 8 ###################                      4.7x
p=0.9, N=16 ##########################               6.4x
p=0.9, N=32 ###############################          7.8x


Run it backwards to estimate the parallel fraction of a real run: from a
measured speedup `S` on `N` workers, `p = (1 - 1/S) / (1 - 1/N)`. Process
start-up and pickling count as serial time, so measured `p` is lower than the
"obvious" one.

In [119]:
for model in ("processes", "subinterpreters"):
    if model in measured:
        speedup = measured["sequential"]["cpu"] / measured[model]["cpu"]
        implied_fraction = (1 - 1 / speedup) / (1 - 1 / 4)
        print(
            f"{model:<16} CPU speedup on 4 workers {speedup:.2f}x -> "
            f"implied parallel fraction p = {implied_fraction:.2f}"
        )

processes        CPU speedup on 4 workers 3.61x -> implied parallel fraction p = 0.96
subinterpreters  CPU speedup on 4 workers 3.24x -> implied parallel fraction p = 0.92


### 5.2.8 Patterns

#### Map-reduce

1. **Split** the input into chunks.
2. **Map**: each worker turns its chunk into a small partial result.
3. **Reduce**: combine the partial results in the parent.

Only the partial results travel back, so the reduce step stays cheap. Here:
revenue per city over 400 000 sales records, with `Counter` partials that add up
with `+`. (Lab 5.2 does word counts the same way.)

In [120]:
sales_random = random.Random(52)
cities = ["Pune", "Mumbai", "Bengaluru", "Chennai", "Kolkata"]
sales = [
    (sales_random.choice(cities), sales_random.randint(100, 5_000))
    for _ in range(400_000 * DEMO_SCALE)
]

with Timer("sequential map + reduce"):
    serial_totals = p5_patterns.revenue_by_city(sales)
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as pool:
    with Timer("parallel: 16 chunks on 4 workers"):
        chunks = [
            sales[start : start + len(sales) // 16]
            for start in range(0, len(sales), len(sales) // 16)
        ]
        partials = pool.map(p5_patterns.revenue_by_city, chunks)
        parallel_totals = functools.reduce(operator.add, partials, Counter())
print(
    f"same totals: {parallel_totals == serial_totals}; top city: {parallel_totals.most_common(1)}"
)

sequential map + reduce took 28.73 ms


parallel: 16 chunks on 4 workers took 47.06 ms
same totals: True; top city: [('Chennai', 204450646)]


On this workload the parallel version is often *not* faster: summing one number
per record is cheap compared with pickling the record to a worker. The pattern
pays off when the map step is expensive per item (parsing, hashing, matching),
and costs least when workers read their own input (a file name or a byte range)
instead of receiving it.

#### Chunking

How you split matters as much as how many workers you use:

| Too few chunks (1 per worker) | Too many chunks (1 per item) | About 4 chunks per worker |
| --- | --- | --- |
| one slow chunk leaves the others idle at the end | per-chunk overhead dominates | balances load and overhead |

`itertools.batched` (*new in 3.12*) cuts an iterable into chunks of a fixed
size; to get a fixed *number* of nearly equal chunks, compute the size first.

In [121]:
def split_evenly[T](items: list[T], chunk_count: int) -> list[list[T]]:
    """Split items into chunk_count chunks whose sizes differ by at most one."""
    size, extra = divmod(len(items), chunk_count)
    chunks, start = [], 0
    for index in range(chunk_count):
        stop = start + size + (1 if index < extra else 0)
        chunks.append(items[start:stop])
        start = stop
    return chunks


ten_orders = [f"order-{n}" for n in range(10)]
print(
    "batched(size 4):        "
    f"{[len(batch) for batch in itertools.batched(ten_orders, 4, strict=False)]}"
)
print(f"split_evenly(4 chunks): {[len(chunk) for chunk in split_evenly(ten_orders, 4)]}")
print(f"workers x 4 chunks for {os.process_cpu_count()} CPUs: {os.process_cpu_count() * 4} chunks")

batched(size 4):        [4, 4, 2]
split_evenly(4 chunks): [3, 3, 2, 2]
workers x 4 chunks for 16 CPUs: 64 chunks


#### `KeyboardInterrupt` handling

Pressing Ctrl+C in a terminal sends `SIGINT` to the whole process group: the
parent *and* every worker. Without care, each worker prints its own
`KeyboardInterrupt` traceback and the pool can hang. The usual recipe:

1. Workers ignore `SIGINT` (an `initializer` that sets `SIG_IGN`): stopping is
   the parent's job.
2. The parent catches `KeyboardInterrupt`, cancels pending work and stops the
   workers: `executor.terminate_workers()` (*new in 3.14*; it also cancels
   pending futures), or `pool.terminate()` for a `multiprocessing.Pool`.
3. Exit with a clear message (and exit code 130, the shell convention for
   Ctrl+C).

The script below follows that recipe. The notebook starts it, waits for two
results, then sends `SIGINT` to the parent process only (POSIX).

In [122]:
CTRL_C_DEMO = """
    import signal, sys, time
    from concurrent.futures import ProcessPoolExecutor

    def ignore_sigint():
        signal.signal(signal.SIGINT, signal.SIG_IGN)  # Ctrl+C is the parent's job

    def slow_task(number):
        time.sleep(0.2)
        return number

    if __name__ == "__main__":
        executor = ProcessPoolExecutor(max_workers=2, initializer=ignore_sigint)
        try:
            print("started", flush=True)
            for result in executor.map(slow_task, range(100)):
                print(f"got {result}", flush=True)
        except KeyboardInterrupt:
            print("KeyboardInterrupt: cancelling pending tasks and stopping workers")
            executor.terminate_workers()
            print("workers stopped; exiting cleanly")
            sys.exit(130)
"""
if not hasattr(signal, "SIGINT") or os.name != "posix":
    print("SKIP: this demo sends POSIX signals")
else:
    (PHASE5_DIR / "ctrl_c_demo.py").write_text(textwrap.dedent(CTRL_C_DEMO), encoding="utf-8")
    demo = subprocess.Popen(
        [sys.executable, "ctrl_c_demo.py"], cwd=PHASE5_DIR, env=CHILD_ENV,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )  # fmt: skip
    lines_before = [demo.stdout.readline().strip() for _ in range(3)]  # started, got 0, got 1
    demo.send_signal(signal.SIGINT)
    rest, _ = demo.communicate(timeout=30)
    print("\n".join(lines_before + rest.strip().splitlines()))
    print(f"exit code: {demo.returncode}")

started
got 0
got 1
KeyboardInterrupt: cancelling pending tasks and stopping workers
workers stopped; exiting cleanly
exit code: 130


#### `os.process_cpu_count()`

`os.cpu_count()` reports the CPUs in the machine. `os.process_cpu_count()`
(*new in 3.13*) reports how many **this process may use**: it respects CPU
affinity (`taskset`, container limits that pin CPUs). The `-X cpu_count=N` option
and the `PYTHON_CPU_COUNT` variable (*new in 3.13*) override **both** functions,
which is handy for testing pool sizing. Pools default to `process_cpu_count()`,
and so should your own sizing. The child script pins itself to two CPUs (Linux
only), then runs again with `PYTHON_CPU_COUNT=3`.

In [123]:
AFFINITY_DEMO = """
    import os
    print(f"os.cpu_count()={os.cpu_count()}  os.process_cpu_count()={os.process_cpu_count()}")
    if hasattr(os, "sched_setaffinity"):
        os.sched_setaffinity(0, set(sorted(os.sched_getaffinity(0))[:2]))
        print(f"pinned to 2 CPUs: os.process_cpu_count()={os.process_cpu_count()}")
    else:
        print("SKIP: sched_setaffinity is not available on this platform")
"""
print(run_child(AFFINITY_DEMO))
print("--- with PYTHON_CPU_COUNT=3")
print(run_child(AFFINITY_DEMO.split("if hasattr")[0], extra_env={"PYTHON_CPU_COUNT": "3"}))

os.cpu_count()=16  os.process_cpu_count()=16
pinned to 2 CPUs: os.process_cpu_count()=2
--- with PYTHON_CPU_COUNT=3
os.cpu_count()=3  os.process_cpu_count()=3


#### Predict the output

**Predict the output.**

```python
# p5_basics.add_order appends to the module-level list ORDERS
# and returns (pid, len(ORDERS)).
with ProcessPoolExecutor(max_workers=1) as pool:
    counts = [count for _, count in pool.map(p5_basics.add_order, ["B-1", "B-2"])]
print(counts, p5_basics.ORDERS)
```

<details><summary>Answer</summary>

`[1, 2] []`. The single worker process keeps its own copy of `ORDERS` across
both tasks (it is reused), so it counts 1 then 2. The kernel's module was never
touched: processes do not share memory. (A fresh pool starts from an empty
list again.)

</details>

Run the next cell to check your prediction.

In [124]:
with concurrent.futures.ProcessPoolExecutor(max_workers=1) as pool:
    counts = [count for _, count in pool.map(p5_basics.add_order, ["B-1", "B-2"])]
print(counts, p5_basics.ORDERS)

[1, 2] []


**Predict the output.**

```python
import pickle

def outer():
    def helper():
        return 1
    return helper

for candidate in (len, outer, outer()):
    try:
        pickle.dumps(candidate)
        print("ok", end=" ")
    except Exception as error:
        print(type(error).__name__, end=" ")
```

<details><summary>Answer</summary>

`ok ok PicklingError`. `len` and `outer` are found by name (`builtins.len`,
`__main__.outer`), so they pickle. The nested `helper` has the qualified name
`outer.<locals>.helper`, which cannot be looked up, so pickling fails, and it
could never be sent to a process. (Whether `__main__.outer` could be *loaded*
by a child is a separate question: 5.2.2.)

</details>

Run the next cell to check your prediction.

In [125]:
def outer() -> Callable[[], int]:
    def helper() -> int:
        return 1

    return helper


for candidate in (len, outer, outer()):
    try:
        pickle.dumps(candidate)
        print("ok", end=" ")
    except Exception as error:
        print(type(error).__name__, end=" ")
print()

ok ok PicklingError 


**Predict the output.**

```python
# 90% of a job parallelizes perfectly. What speedup on 16 cores, and the most on any number?
print(round(1 / ((1 - 0.9) + 0.9 / 16), 1), round(1 / (1 - 0.9), 1))
```

<details><summary>Answer</summary>

`6.4 10.0`. Amdahl's law: the 10% serial part costs 0.1 of the original time on
any number of cores, so the speedup can never exceed 1 / 0.1 = 10. Sixteen
cores give 1 / (0.1 + 0.05625) = 6.4.

</details>

Run the next cell to check your prediction.

In [126]:
print(round(1 / ((1 - 0.9) + 0.9 / 16), 1), round(1 / (1 - 0.9), 1))

6.4 10.0


### Lab 5.2

Four exercises. Worker functions go in `%%writefile` cells (child processes must
import them); edit the cell, run it, then run the cell after it, which
re-imports the module with `fresh_import`. Replace each
`raise NotImplementedError`, then run the check cell. Until you finish, the
checks print FAIL; that is expected.

The setup cell writes eight 8 MiB files for exercise 1 and builds a seeded
100 000-line corpus for exercise 4.

> **Warning:** after editing a worker module, re-run *both* the `%%writefile`
> cell and the cell after it. `fresh_import` puts a new module object in
> `sys.modules`; a function still taken from the old module object can no
> longer be pickled (`PicklingError: ... it's not the same object as ...`),
> because pickle checks that the name leads back to that exact object.

In [127]:
import concurrent.futures
import hashlib
import importlib.util
import inspect
import math
import multiprocessing
import random
import re
import sys
import time
from collections import Counter
from collections.abc import Iterable
from multiprocessing import shared_memory
from pathlib import Path

LAB52_DIR = PHASE5_DIR / "lab52"
LAB52_DIR.mkdir(parents=True, exist_ok=True)
lab_random = random.Random(52)

HASH_FILES = [LAB52_DIR / f"export-{number}.bin" for number in range(8)]
for path in HASH_FILES:  # 8 files of 8 MiB x DEMO_SCALE, same bytes every run
    path.write_bytes(lab_random.randbytes(8 * 1024 * 1024 * DEMO_SCALE))

WORDS = "chai masala the shop's order delivered late early pune mumbai".split()
WORDS += "refund invoice gst customer happy isn't tea mug kettle rain".split()
CORPUS = [
    " ".join(lab_random.choices(WORDS, k=12)).capitalize() for _ in range(100_000 * DEMO_SCALE)
]


def shm_segments() -> set[str]:
    """Names of POSIX shared-memory blocks made by multiprocessing (Linux; empty elsewhere)."""
    folder = Path("/dev/shm")
    return {entry.name for entry in folder.glob("psm_*")} if folder.exists() else set()


if HAS_NUMPY:
    import numpy as np
print(
    f"{len(HASH_FILES)} files of {HASH_FILES[0].stat().st_size // 2**20} "
    f"MiB, {len(CORPUS):,} corpus lines"
)
print(f"first corpus line: {CORPUS[0]!r}")

8 files of 8 MiB, 100,000 corpus lines
first corpus line: "Gst pune pune invoice chai isn't mumbai late mumbai refund refund mumbai"


#### Exercise 1 — parallel hashing with processes vs threads, with the result explained

1. In `lab52_hashing.py`, `sha256_file(path)` returns `(path.name, hex digest)`,
   reading the file in 1 MiB blocks and feeding them to one `hashlib.sha256()`.
2. `hash_files(paths, executor_class, max_workers=4)` hashes every file with
   `executor_class(max_workers=max_workers)` (it will be called with
   `ThreadPoolExecutor` and `ProcessPoolExecutor`) and returns
   `{file name: digest}`.
3. `benchmark_hashing(paths, max_workers=4)` returns the seconds taken for
   `"sequential"` (a plain loop), `"threads"` and `"processes"`.

In [128]:
%%writefile _scratch/phase5/lab52_hashing.py
"""Lab 5.2 exercise 1 worker."""

import hashlib  # noqa: F401 (for your solution)
from pathlib import Path


def sha256_file(path: Path) -> tuple[str, str]:
    """Return (file name, SHA-256 hex digest), reading 1 MiB blocks."""
    raise NotImplementedError

Overwriting _scratch/phase5/lab52_hashing.py


Re-import the worker module, then write the two functions:

In [129]:
hashing_workers = fresh_import("lab52_hashing")


def hash_files(
    paths: list[Path], executor_class: type[concurrent.futures.Executor], max_workers: int = 4
) -> dict[str, str]:
    """Hash files with executor_class(max_workers); return {file name: digest}."""
    raise NotImplementedError


def benchmark_hashing(paths: list[Path], max_workers: int = 4) -> dict[str, float]:
    """Seconds for "sequential", "threads" and "processes"."""
    raise NotImplementedError

Check exercise 1:

In [130]:
EXPECTED_DIGESTS = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in HASH_FILES}
run_checks(
    [
        (
            "sha256_file returns (name, digest)",
            lambda: hashing_workers.sha256_file(HASH_FILES[0]),
            (HASH_FILES[0].name, EXPECTED_DIGESTS[HASH_FILES[0].name]),
        ),
        (
            "threads: right digests",
            lambda: hash_files(HASH_FILES, concurrent.futures.ThreadPoolExecutor),
            EXPECTED_DIGESTS,
        ),
        (
            "processes: right digests",
            lambda: hash_files(HASH_FILES, concurrent.futures.ProcessPoolExecutor),
            EXPECTED_DIGESTS,
        ),
        (
            "benchmark: three positive timings",
            lambda: sorted(
                name for name, seconds in benchmark_hashing(HASH_FILES).items() if seconds > 0
            ),
            ["processes", "sequential", "threads"],
        ),
    ]
)

FAIL  sha256_file returns (name, digest): raised NotImplementedError: 
FAIL  threads: right digests: raised NotImplementedError: 
FAIL  processes: right digests: raised NotImplementedError: 
FAIL  benchmark: three positive timings: raised NotImplementedError: 
0/4 checks passed


**Explain the result.** Run the benchmark, then write down which model won and
why before opening the answer.

<details><summary>Answer</summary>

Threads usually match or beat processes here. `hashlib` releases the GIL while
it hashes each 1 MiB block, and file reads release it too, so threads already
hash on several cores at once, with no start-up cost and nothing to pickle
(each task sends only a path and returns a short string). Processes get the
same parallelism but pay for starting workers and importing the module. With
much bigger files the two converge; with many tiny files, threads win clearly;
if the hashing were pure Python (no GIL release), only processes would scale.

</details>

In [131]:
try:
    for model, seconds in benchmark_hashing(HASH_FILES).items():
        print(f"{model:<11} {seconds:.3f} s")
except NotImplementedError:
    print("Finish exercise 1 first.")

Finish exercise 1 first.


#### Exercise 2 — a `chunksize` tuning chart

The worker `tiny_task` is given (it is deliberately cheap).

1. `time_chunksizes(items, chunksizes, processes=4)` creates **one**
   `multiprocessing.Pool(processes)`, warms it up, then times
   `pool.map(tiny_task, items, chunksize=c)` for each `c` and returns
   `{chunksize: seconds}`.
2. `render_chart(timings, width=40)` returns one line per chunksize, in
   ascending chunksize order: `f"{chunksize:>6} | {bar} {seconds:.3f} s"` where
   `bar` is `"#"` repeated `max(1, round(width * seconds / slowest))` times.

In [132]:
%%writefile _scratch/phase5/lab52_chunks.py
"""Lab 5.2 exercise 2 worker (given)."""


def tiny_task(number: int) -> int:
    """A deliberately tiny task."""
    return (number * 7) % 11

Overwriting _scratch/phase5/lab52_chunks.py


Re-import the worker module, then write the two functions:

In [133]:
chunk_workers = fresh_import("lab52_chunks")


def time_chunksizes(
    items: list[int], chunksizes: list[int], processes: int = 4
) -> dict[int, float]:
    """Seconds for Pool.map(tiny_task, items, chunksize=c) for each c, using one pool."""
    raise NotImplementedError


def render_chart(timings: dict[int, float], width: int = 40) -> list[str]:
    """A text bar chart, one line per chunksize in ascending order."""
    raise NotImplementedError

Check exercise 2:

In [134]:
run_checks(
    [
        (
            "chart lines",
            lambda: render_chart({10: 0.25, 1: 0.5, 100: 0.05}, width=20),
            [
                "     1 | #################### 0.500 s",
                "    10 | ########## 0.250 s",
                "   100 | ## 0.050 s",
            ],
        ),
        (
            "tiny bars still show one #",
            lambda: render_chart({1: 1.0, 1000: 0.001}, width=10)[1],
            "  1000 | # 0.001 s",
        ),
        (
            "one key per chunksize",
            lambda: sorted(time_chunksizes(list(range(2_000)), [1, 50, 500])),
            [1, 50, 500],
        ),
        (
            "positive timings",
            lambda: all(
                seconds > 0 for seconds in time_chunksizes(list(range(500)), [1, 100]).values()
            ),
            True,
        ),
    ]
)

FAIL  chart lines: raised NotImplementedError: 
FAIL  tiny bars still show one #: raised NotImplementedError: 
FAIL  one key per chunksize: raised NotImplementedError: 
FAIL  positive timings: raised NotImplementedError: 
0/4 checks passed


Once the checks pass, draw the real chart for 40 000 tasks (this takes a few
seconds, most of it at `chunksize=1`). Where does the curve flatten on your
machine?

In [135]:
try:
    chart_timings = time_chunksizes(list(range(40_000 * DEMO_SCALE)), [1, 10, 100, 1_000, 10_000])
    print("\n".join(render_chart(chart_timings)))
except NotImplementedError:
    print("Finish exercise 2 first.")

Finish exercise 2 first.


#### Exercise 3 — shared-memory array workers

1. In `lab52_shared.py`, `scale_slice(block_name, length, start, stop, factor)`
   attaches to the shared block with `SharedMemory(name=block_name,
   track=False)`, views it as a `float64` NumPy array of `length` values,
   multiplies `values[start:stop]` by `factor` in place, and closes the block
   (delete the array view first).
2. `parallel_scale(values, factor, workers=4)` creates a `SharedMemory` block,
   copies `values` into it (as `float64`), starts one `multiprocessing.Process`
   per slice (split as evenly as possible), joins them, returns a **copy** of
   the result, and always closes and unlinks the block. The input array must
   not change.

In [136]:
%%writefile _scratch/phase5/lab52_shared.py
"""Lab 5.2 exercise 3 worker."""

from multiprocessing import shared_memory  # noqa: F401 (for your solution)


def scale_slice(block_name: str, length: int, start: int, stop: int, factor: float) -> None:
    """Multiply values[start:stop] of the shared float64 array by factor, in place."""
    raise NotImplementedError

Overwriting _scratch/phase5/lab52_shared.py


Re-import the worker module, then write `parallel_scale`:

In [137]:
shared_workers = fresh_import("lab52_shared")


def parallel_scale(values: np.ndarray, factor: float, workers: int = 4) -> np.ndarray:
    """Scale a copy of values in shared memory with one process per slice."""
    raise NotImplementedError

Check exercise 3 (prints a skip message without NumPy):

In [138]:
def segments_left_behind() -> list[str]:
    """Shared-memory blocks that exist after parallel_scale but did not before."""
    before = shm_segments()
    parallel_scale(np.arange(1_000.0), 2.0)
    return sorted(shm_segments() - before)


def input_unchanged() -> list[float]:
    """parallel_scale must not modify its argument."""
    original = np.arange(5.0)
    parallel_scale(original, 3.0)
    return original.tolist()


if not HAS_NUMPY:
    print("SKIP: numpy is not installed")
else:
    run_checks(
        [
            (
                "scales every value",
                lambda: parallel_scale(np.arange(6.0), 2.5).tolist(),
                [0.0, 2.5, 5.0, 7.5, 10.0, 12.5],
            ),
            (
                "uneven split: 10 values, 4 workers",
                lambda: parallel_scale(np.arange(10.0), 10.0, 4).tolist(),
                [10.0 * n for n in range(10)],
            ),
            (
                "one worker",
                lambda: parallel_scale(np.arange(3.0), -1.0, 1).tolist(),
                [0.0, -1.0, -2.0],
            ),
            ("returns float64", lambda: str(parallel_scale(np.arange(4), 2.0).dtype), "float64"),
            ("input array unchanged", input_unchanged, [0.0, 1.0, 2.0, 3.0, 4.0]),
            ("no shared memory left behind", segments_left_behind, []),
            (
                "uses Process and SharedMemory",
                lambda: all(
                    word in inspect.getsource(parallel_scale)
                    for word in ("Process(", "SharedMemory(")
                ),
                True,
            ),
        ]
    )

FAIL  scales every value: raised NotImplementedError: 
FAIL  uneven split: 10 values, 4 workers: raised NotImplementedError: 
FAIL  one worker: raised NotImplementedError: 
FAIL  returns float64: raised NotImplementedError: 
FAIL  input array unchanged: raised NotImplementedError: 
FAIL  no shared memory left behind: raised NotImplementedError: 
FAIL  uses Process and SharedMemory: expected True, got False
0/7 checks passed


#### Exercise 4 — map-reduce word counts, measured speedup vs Amdahl's prediction

1. In `lab52_wordcount.py`, `count_words(lines)` (the map step) returns a
   `Counter` of the words in a list of lines: lowercase each line and take
   `re.findall(r"[a-z']+", line)`.
2. `merge_counts(partials)` (the reduce step) adds an iterable of `Counter`s.
3. `parallel_word_count(lines, workers=4, chunks_per_worker=4)` splits `lines`
   into `workers * chunks_per_worker` nearly equal chunks, maps them with a
   `ProcessPoolExecutor(workers)` and reduces with `merge_counts`.
4. `amdahl_speedup(parallel_fraction, workers)` returns
   `1 / ((1 - p) + p / workers)`; raise `ValueError` unless `0 <= p <= 1`.
5. `speedup_report(lines, workers=4)` returns a dict with
   - `"sequential_seconds"`: time to run `count_words` on every chunk in the
     kernel (the map time) plus time to `merge_counts` them (the reduce time);
   - `"parallel_seconds"`: time for `parallel_word_count(lines, workers)`;
   - `"measured_speedup"`: sequential ÷ parallel;
   - `"parallel_fraction"`: map time ÷ sequential time (the part that can run
     in parallel);
   - `"predicted_speedup"`: `amdahl_speedup(parallel_fraction, workers)`.

In [139]:
%%writefile _scratch/phase5/lab52_wordcount.py
"""Lab 5.2 exercise 4 worker."""

import re  # noqa: F401 (for your solution)
from collections import Counter


def count_words(lines: list[str]) -> Counter[str]:
    """Map step: count the lowercase words in a chunk of lines."""
    raise NotImplementedError

Overwriting _scratch/phase5/lab52_wordcount.py


Re-import the worker module, then write the four functions:

In [140]:
wordcount_workers = fresh_import("lab52_wordcount")


def merge_counts(partials: Iterable[Counter[str]]) -> Counter[str]:
    """Reduce step: add the partial counts."""
    raise NotImplementedError


def parallel_word_count(
    lines: list[str], workers: int = 4, chunks_per_worker: int = 4
) -> Counter[str]:
    """Map-reduce word count with a process pool."""
    raise NotImplementedError


def amdahl_speedup(parallel_fraction: float, workers: int) -> float:
    """Amdahl's law."""
    raise NotImplementedError


def speedup_report(lines: list[str], workers: int = 4) -> dict[str, float]:
    """Measured speedup next to Amdahl's prediction."""
    raise NotImplementedError

Check exercise 4:

In [141]:
SAMPLE_LINES = ["The shop's tea", "the TEA was late", ""]


def report_consistent() -> bool:
    """The report's numbers agree with each other."""
    report = speedup_report(CORPUS[:20_000], 4)
    return (
        math.isclose(
            report["measured_speedup"], report["sequential_seconds"] / report["parallel_seconds"]
        )
        and math.isclose(
            report["predicted_speedup"], amdahl_speedup(report["parallel_fraction"], 4)
        )
        and 0 < report["parallel_fraction"] <= 1
    )


run_checks(
    [
        (
            "count_words (map step)",
            lambda: dict(wordcount_workers.count_words(SAMPLE_LINES)),
            {"the": 2, "shop's": 1, "tea": 2, "was": 1, "late": 1},
        ),
        (
            "merge_counts (reduce step)",
            lambda: dict(merge_counts([Counter(a=1, b=2), Counter(b=3), Counter()])),
            {"a": 1, "b": 5},
        ),
        (
            "parallel result equals sequential",
            lambda: (
                parallel_word_count(CORPUS[:20_000])
                == wordcount_workers.count_words(CORPUS[:20_000])
            ),
            True,
        ),
        ("amdahl_speedup(0.9, 16)", lambda: round(amdahl_speedup(0.9, 16), 2), 6.4),
        (
            "amdahl_speedup at p=1 and p=0",
            lambda: (amdahl_speedup(1.0, 8), amdahl_speedup(0.0, 8)),
            (8.0, 1.0),
        ),
        (
            "p outside 0..1 is rejected",
            lambda: raised_error_name(lambda: amdahl_speedup(1.5, 4)),
            "ValueError",
        ),
        (
            "speedup_report keys",
            lambda: sorted(speedup_report(CORPUS[:20_000], 4)),
            [
                "measured_speedup",
                "parallel_fraction",
                "parallel_seconds",
                "predicted_speedup",
                "sequential_seconds",
            ],
        ),
        ("speedup_report is consistent", report_consistent, True),
    ]
)

FAIL  count_words (map step): raised NotImplementedError: 
FAIL  merge_counts (reduce step): raised NotImplementedError: 
FAIL  parallel result equals sequential: raised NotImplementedError: 
FAIL  amdahl_speedup(0.9, 16): raised NotImplementedError: 
FAIL  amdahl_speedup at p=1 and p=0: raised NotImplementedError: 
FAIL  p outside 0..1 is rejected: expected 'ValueError', got 'NotImplementedError'
FAIL  speedup_report keys: raised NotImplementedError: 
FAIL  speedup_report is consistent: raised NotImplementedError: 
0/8 checks passed


Once the checks pass, compare the measurement with the prediction on the full
corpus. Expect the measured speedup to fall short of Amdahl's number: the law
assumes the parallel part splits perfectly, but here every chunk of lines must
be pickled to a worker and every `Counter` pickled back, and the pool must
start. That overhead behaves like extra serial time. The last number turns the
measurement back into a parallel fraction (`p = (1 - 1/S) / (1 - 1/N)`, from
5.2.7): the gap between it and the profiled `p` is the cost of going parallel.

In [142]:
try:
    for workers in (2, 4):
        report = speedup_report(CORPUS, workers)
        effective_p = (1 - 1 / report["measured_speedup"]) / (1 - 1 / workers)
        print(
            f"{workers} workers: sequential {report['sequential_seconds']:.3f} s, "
            f"parallel {report['parallel_seconds']:.3f} s"
        )
        print(
            f"  measured {report['measured_speedup']:.2f}x, "
            f"Amdahl predicts {report['predicted_speedup']:.2f}x"
        )
        print(
            f"  p from the profile {report['parallel_fraction']:.3f}, "
            f"p implied by the measurement {effective_p:.2f}"
        )
except NotImplementedError:
    print("Finish exercise 4 first.")

Finish exercise 4 first.


### Recap — Unit 5.2

| Concept | One-line takeaway | JS / Node equivalent |
| --- | --- | --- |
| Processes | Own memory and own GIL: real parallelism for Python code; milliseconds to start | `child_process.fork` |
| Start methods | `forkserver` is the POSIX default since 3.14; `fork` is unsafe with threads; `spawn` needs the main guard | — |
| Importable workers | Functions are pickled by name; notebook functions live in `__main__` and fail in children | — |
| `Process`, `Pool`, `ProcessPoolExecutor` | Reuse workers; `chunksize` for small tasks; `initializer` per worker; `max_tasks_per_child` against leaks | `cluster`, worker pools |
| Data transfer | Everything crosses as a pickle: coarse tasks, small messages, or parallel gets slower | structured clone |
| IPC | `Queue`, `Pipe`, `Value`/`Array` (+ lock), `Manager` (slow proxies), `shared_memory` (zero-copy with NumPy) | `process.send`, `SharedArrayBuffer` |
| Subinterpreters | `concurrent.interpreters` (3.14): own GIL inside one process; C extensions must opt in | `worker_threads` |
| Choosing a model | Decision matrix + a benchmark on your workload | — |
| Amdahl's law | `S = 1 / ((1 - p) + p / N)`; the serial part caps the speedup | — |
| Patterns | Map-reduce with `Counter`s, ~4 chunks per worker, workers ignore SIGINT, size by `os.process_cpu_count()` | `os.availableParallelism()` |

**Further reading**

- [`multiprocessing`](https://docs.python.org/3/library/multiprocessing.html)
  (read *Programming guidelines*),
  [`multiprocessing.shared_memory`](https://docs.python.org/3/library/multiprocessing.shared_memory.html)
- [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html)
  (`ProcessPoolExecutor`, `InterpreterPoolExecutor`),
  [`concurrent.interpreters`](https://docs.python.org/3/library/concurrent.interpreters.html)
- [`pickle`: what can be pickled](https://docs.python.org/3/library/pickle.html#what-can-be-pickled-and-unpickled),
  [`os.process_cpu_count`](https://docs.python.org/3/library/os.html#os.process_cpu_count)
- [PEP 371](https://peps.python.org/pep-0371/) (the `multiprocessing` package),
  [PEP 684](https://peps.python.org/pep-0684/) (a per-interpreter GIL),
  [PEP 734](https://peps.python.org/pep-0734/) (multiple interpreters in the standard library)